# J2 — 최신 LLM 응답 환각 탐지율 올리기: 응답 끝 2차 검증기 (11/27 재설계)

## 왜 하나

배포 판정기 v1.0(1차, 스트리밍 중 8토큰마다)은 **최신 Claude 응답의 환각을 18.8%만** 잡는다(K2, devM, 오경보 6.6%).
놓치는 것의 대부분은 근거 단어를 그대로 쓰며 내용만 바꾼 직접 모순이고(M1a), 경량 판정기 재학습(K1·K2)은 +2%p에 그쳤다.
그래서 응답이 끝난 뒤 **2차 검증기**가 한 번 더 본다. 서비스에서 사용자가 받는 경고는 "1차 또는 2차"이므로 **그 결합 탐지율**을 잰다.
사용자 결정(11/27): 오경보 20%까지 허용, 결합 탐지율 **40% 이상**이면 성공, 후보에서 Sonnet 제외.

## 무엇을 하나 (학습·새 생성 없음)

- **데이터**: K2에서 만든 최신 Claude 응답(`claude-haiku-5-5`·`claude-sonnet-5-5`, RAGTruth train 분할 근거, 정답 `claude-opus-5-5` 문장 S/H/A, 환각 응답 = H 문장 있음)
  - **calM**(1,072응답, 환각 178): 2차 임계값·후보 선택
  - **devM**(1,072응답, 환각 191): 판정
  - 1차 점수·정답은 저장소 `k2_{calM,devM}.csv`(K2 보고 수치의 원본)를 그대로, 응답 원문·근거는 드라이브 `k2/`
- **후보(연속 점수)**
  - `haiku_H` / `haiku_HA`: `claude-haiku-5-5` 문장 S/H/A 판정 5회 투표, 점수 = H(또는 H·A) 문장이 있다고 본 비율
  - `minicheck`: `Bespoke-MiniCheck-7B` 지지 확률, 점수 = 1 − 확률 (안 되면 `flan-t5-large`, 둘 다 안 되면 제외)
  - `ens`: `haiku_H`와 `minicheck` 평균
  - Opus는 정답을 매긴 모델이라, Sonnet은 비용 때문에 뺀다
- **근거** = Opus가 정답을 매길 때 본 것과 같은 글(QA는 질문 + 게이트웨이 검색 조각)

## 사전 정의 기준 (SSOT §6 J2, 11/27 고정)

- **J2-0(유효성)**: 1차 재현 — devM 1차 탐지 36/191, 오경보 58/881(K2 보고와 같음). Haiku 판정 파싱 성공 calM·devM 각각 ≥ 95%(미달이면 Haiku 후보 제외)
- **임계값·선택(calM)**: 후보마다 calM 멀쩡한 응답의 **결합 경고율 ≤ 20%**가 되는 가장 낮은 2차 임계값. 그 임계값에서 calM 결합 탐지율이 가장 높은 후보(동점이면 haiku_H → haiku_HA → minicheck → ens)
- **J2-1(주, devM)**: 결합 탐지율 ≥ **0.40**, 그리고 응답 단위 부트스트랩 95% 하한 ≥ **0.30** (2,000회, 시드 2041)
- **J2-2(devM)**: 결합 오경보(멀쩡한 응답 경고율) ≤ **0.20**
- **J2 채택 = J2-1 그리고 J2-2** → 게이트웨이 응답 끝 2단 검증(`post_check`)의 모델·투표 수·임계값을 이것으로 바꾸고 SPEC부터 갱신
- **J2-3(지연, 방식 결정)**: 선택 후보 응답 1개 검증 p95(devM 30개, 실시간, 투표 병렬) ≤ 10초면 hold 가능, 넘으면 응답 뒤 경고
- 보고만: 모든 후보의 calM·devM 결합·2차 단독 탐지/오경보, 생성 모델·과제별. **API 상한 $10**(제출 전 추정 > $8이면 멈춤)
- 한계(실행 전 명시): 정답이 Opus 판정이고 Haiku는 같은 Claude 계열 → 사람 정답보다 유리하게 나올 수 있음. 응답의 절반은 Haiku가 생성

> 런타임 **GPU**(MiniCheck용, L4·A100). Colab 비밀 값 `ANTHROPIC_API_KEY`. 드라이브 `k2/`에 K2a 파일(`cases_`·`gen_`·`label_` calM·devM)이 있어야 한다.
> 끊기면 다시 모두 실행(배치·점수는 드라이브 `j2/`에서 이어서). 끝나면 `j2_results.zip`(원문 없음)이 내려받아진다.


## 1. 환경 준비

In [ ]:
import os, sys, json, time, random, subprocess, shutil, traceback
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'anthropic', 'pysbd'], check=True)
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
K2_DIR, J2_DIR = os.path.join(OUT, 'k2'), os.path.join(OUT, 'j2'); os.makedirs(J2_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
for sp in ['calM', 'devM']:
    for n in ['cases', 'gen']:
        assert os.path.exists(os.path.join(K2_DIR, f'{n}_{sp}.json')), f'k2/{n}_{sp}.json 없음 (K2a 산출물)'
print('device', GPU)

## 2. 최신 Claude 응답 (calM·devM) — K2 정답·1차 점수 + 응답 원문

In [ ]:
K2T = {"calM": {"calM-haiku-11320-0": [false, 1.0494962171994495, "Summary", "haiku"], "calM-haiku-11320-1": [false, 1.7416631054896436, "Summary", "haiku"], "calM-sonnet-11320-0": [true, 1.3975406026975652, "Summary", "sonnet"], "calM-sonnet-11320-1": [true, 2.4434344727575708, "Summary", "sonnet"], "calM-haiku-11348-0": [true, 0.33170414635773793, "Summary", "haiku"], "calM-haiku-11348-1": [true, 0.26114947522300225, "Summary", "haiku"], "calM-sonnet-11348-0": [true, 0.27191059237360626, "Summary", "sonnet"], "calM-sonnet-11348-1": [true, 0.902803721755059, "Summary", "sonnet"], "calM-haiku-11413-0": [true, 0.22658752253701497, "Summary", "haiku"], "calM-haiku-11413-1": [true, 0.5081675658878856, "Summary", "haiku"], "calM-sonnet-11413-0": [true, 3.6826007347941743, "Summary", "sonnet"], "calM-sonnet-11413-1": [true, 3.3430830361077275, "Summary", "sonnet"], "calM-haiku-11414-0": [false, 0.4910257067366249, "Summary", "haiku"], "calM-haiku-11414-1": [false, 1.09666079171167, "Summary", "haiku"], "calM-sonnet-11414-0": [true, 3.5766117538677253, "Summary", "sonnet"], "calM-sonnet-11414-1": [true, 2.7014808815946036, "Summary", "sonnet"], "calM-haiku-11770-0": [true, 1.5560899964556443, "Summary", "haiku"], "calM-haiku-11770-1": [true, 1.2337262262096336, "Summary", "haiku"], "calM-sonnet-11770-0": [false, 1.2854352117966428, "Summary", "sonnet"], "calM-sonnet-11770-1": [true, 0.2485344180377267, "Summary", "sonnet"], "calM-haiku-11778-0": [true, 1.3186695527169967, "Summary", "haiku"], "calM-haiku-11778-1": [true, 0.5181896391515659, "Summary", "haiku"], "calM-sonnet-11778-0": [true, 1.2701556175456612, "Summary", "sonnet"], "calM-sonnet-11778-1": [true, 0.8934004306988759, "Summary", "sonnet"], "calM-haiku-11781-0": [true, 2.0424385726952217, "Summary", "haiku"], "calM-haiku-11781-1": [false, 1.933053546921703, "Summary", "haiku"], "calM-sonnet-11781-0": [true, 0.7613382335549769, "Summary", "sonnet"], "calM-sonnet-11781-1": [false, 1.1469842894705349, "Summary", "sonnet"], "calM-haiku-11782-0": [true, 1.167578087512767, "Summary", "haiku"], "calM-haiku-11782-1": [true, 3.0254842962430186, "Summary", "haiku"], "calM-sonnet-11782-0": [true, 0.44490850841520957, "Summary", "sonnet"], "calM-sonnet-11782-1": [true, 0.6212020213087805, "Summary", "sonnet"], "calM-haiku-11790-0": [true, 0.47835921040371077, "Summary", "haiku"], "calM-haiku-11790-1": [true, 0.5473042993465831, "Summary", "haiku"], "calM-sonnet-11790-0": [true, 0.38113780986648116, "Summary", "sonnet"], "calM-sonnet-11790-1": [true, 0.3198867272187954, "Summary", "sonnet"], "calM-haiku-11800-0": [true, 1.5417763442250643, "Summary", "haiku"], "calM-haiku-11800-1": [true, 1.584670460934985, "Summary", "haiku"], "calM-sonnet-11800-0": [true, 1.7609329126062065, "Summary", "sonnet"], "calM-sonnet-11800-1": [true, 2.513877513360296, "Summary", "sonnet"], "calM-haiku-11824-0": [true, 0.38991169248739205, "Summary", "haiku"], "calM-haiku-11824-1": [true, 0.1072302023835465, "Summary", "haiku"], "calM-sonnet-11824-0": [true, 0.04821489718929132, "Summary", "sonnet"], "calM-sonnet-11824-1": [true, 0.0693663584625896, "Summary", "sonnet"], "calM-haiku-11859-0": [false, 1.0380612596828895, "Summary", "haiku"], "calM-haiku-11859-1": [false, 1.3324578323562688, "Summary", "haiku"], "calM-sonnet-11859-0": [true, 2.226196390737342, "Summary", "sonnet"], "calM-sonnet-11859-1": [true, 0.6102498495600441, "Summary", "sonnet"], "calM-haiku-11912-0": [true, 0.8867415616688634, "Summary", "haiku"], "calM-haiku-11912-1": [true, 1.8901847061849817, "Summary", "haiku"], "calM-sonnet-11912-0": [true, 1.515310926327724, "Summary", "sonnet"], "calM-sonnet-11912-1": [true, 1.3380901076192795, "Summary", "sonnet"], "calM-haiku-11914-0": [true, 1.5086549612463493, "Summary", "haiku"], "calM-haiku-11914-1": [false, 2.032560567816126, "Summary", "haiku"], "calM-sonnet-11914-0": [true, 1.534836521972297, "Summary", "sonnet"], "calM-sonnet-11914-1": [true, 2.0158645743772556, "Summary", "sonnet"], "calM-haiku-11927-0": [true, 0.5239754243736641, "Summary", "haiku"], "calM-haiku-11927-1": [true, -0.14591489096036897, "Summary", "haiku"], "calM-sonnet-11927-0": [true, 0.24783453375750505, "Summary", "sonnet"], "calM-sonnet-11927-1": [true, 0.8128410056645772, "Summary", "sonnet"], "calM-haiku-11935-0": [true, 0.9096309277502423, "Summary", "haiku"], "calM-haiku-11935-1": [false, 1.8637477193010317, "Summary", "haiku"], "calM-sonnet-11935-0": [true, 1.0656548499194654, "Summary", "sonnet"], "calM-sonnet-11935-1": [true, 0.736916613465048, "Summary", "sonnet"], "calM-haiku-11939-0": [true, 0.22977942576118762, "Summary", "haiku"], "calM-haiku-11939-1": [true, -0.3381731912678128, "Summary", "haiku"], "calM-sonnet-11939-0": [true, 0.059879096870433156, "Summary", "sonnet"], "calM-sonnet-11939-1": [true, -0.07465246364231437, "Summary", "sonnet"], "calM-haiku-11949-0": [false, 1.962211714797689, "Summary", "haiku"], "calM-haiku-11949-1": [false, 1.5854381623165539, "Summary", "haiku"], "calM-sonnet-11949-0": [true, 1.0879363042949572, "Summary", "sonnet"], "calM-sonnet-11949-1": [true, 1.2049041970897814, "Summary", "sonnet"], "calM-haiku-11952-0": [false, 2.768373598335991, "Summary", "haiku"], "calM-haiku-11952-1": [true, 0.9102370375499798, "Summary", "haiku"], "calM-sonnet-11952-0": [true, 1.450976400167163, "Summary", "sonnet"], "calM-sonnet-11952-1": [true, 1.339433195337946, "Summary", "sonnet"], "calM-haiku-11953-0": [true, 0.8988759523322789, "Summary", "haiku"], "calM-haiku-11953-1": [true, 0.31356860592159896, "Summary", "haiku"], "calM-sonnet-11953-0": [true, 2.7363811024926368, "Summary", "sonnet"], "calM-sonnet-11953-1": [true, 0.6275760159639252, "Summary", "sonnet"], "calM-haiku-11959-0": [false, 2.465767108637488, "Summary", "haiku"], "calM-haiku-11959-1": [true, 2.16585133790291, "Summary", "haiku"], "calM-sonnet-11959-0": [false, 0.6735465156917969, "Summary", "sonnet"], "calM-sonnet-11959-1": [true, 1.3876198813918348, "Summary", "sonnet"], "calM-haiku-11982-0": [true, 0.9680229790791293, "Summary", "haiku"], "calM-haiku-11982-1": [false, 1.4667909607022025, "Summary", "haiku"], "calM-sonnet-11982-0": [true, 1.4564631427661217, "Summary", "sonnet"], "calM-sonnet-11982-1": [true, 1.1924830792785237, "Summary", "sonnet"], "calM-haiku-11985-0": [true, 2.0017001555202283, "Summary", "haiku"], "calM-haiku-11985-1": [true, 2.822376549831754, "Summary", "haiku"], "calM-sonnet-11985-0": [true, 3.1041200715096187, "Summary", "sonnet"], "calM-sonnet-11985-1": [true, 3.1153286580851898, "Summary", "sonnet"], "calM-haiku-12009-0": [false, 3.177528147250882, "Summary", "haiku"], "calM-haiku-12009-1": [true, 3.515766886535668, "Summary", "haiku"], "calM-sonnet-12009-0": [true, 3.891959911921046, "Summary", "sonnet"], "calM-sonnet-12009-1": [true, 1.2823824070404224, "Summary", "sonnet"], "calM-haiku-12029-0": [true, 0.6876095240773706, "Summary", "haiku"], "calM-haiku-12029-1": [true, 1.1715284327849242, "Summary", "haiku"], "calM-sonnet-12029-0": [true, 0.8304571997731917, "Summary", "sonnet"], "calM-sonnet-12029-1": [true, 0.5095102272165126, "Summary", "sonnet"], "calM-haiku-12036-0": [true, 0.9349651188045495, "Summary", "haiku"], "calM-haiku-12036-1": [true, 1.5275236775764494, "Summary", "haiku"], "calM-sonnet-12036-0": [true, 0.6559032743336772, "Summary", "sonnet"], "calM-sonnet-12036-1": [true, 0.8725607839618974, "Summary", "sonnet"], "calM-haiku-12041-0": [false, 2.718566265965447, "Summary", "haiku"], "calM-haiku-12041-1": [false, 2.056347859439889, "Summary", "haiku"], "calM-sonnet-12041-0": [false, 1.9324629751328581, "Summary", "sonnet"], "calM-sonnet-12041-1": [true, 2.1954071625724305, "Summary", "sonnet"], "calM-haiku-12045-0": [true, 0.4338106572385768, "Summary", "haiku"], "calM-haiku-12045-1": [true, 1.4770179643488281, "Summary", "haiku"], "calM-sonnet-12045-0": [true, 2.7356805673032927, "Summary", "sonnet"], "calM-sonnet-12045-1": [true, 0.40428347949623966, "Summary", "sonnet"], "calM-haiku-12049-0": [false, 0.25249197343708213, "Summary", "haiku"], "calM-haiku-12049-1": [false, 1.3332647347164457, "Summary", "haiku"], "calM-sonnet-12049-0": [false, 1.5676613339999537, "Summary", "sonnet"], "calM-sonnet-12049-1": [true, 3.2589853491015486, "Summary", "sonnet"], "calM-haiku-12075-0": [false, 1.7312249410971026, "Summary", "haiku"], "calM-haiku-12075-1": [false, 2.3359517631487305, "Summary", "haiku"], "calM-sonnet-12075-0": [true, 1.2568598841516165, "Summary", "sonnet"], "calM-sonnet-12075-1": [true, 1.0070927521409039, "Summary", "sonnet"], "calM-haiku-12176-0": [true, 2.8061623155312962, "QA", "haiku"], "calM-haiku-12176-1": [false, 2.672439099010068, "QA", "haiku"], "calM-sonnet-12176-0": [true, 2.650305469880906, "QA", "sonnet"], "calM-sonnet-12176-1": [true, 2.6235789373214895, "QA", "sonnet"], "calM-haiku-12185-0": [false, 2.0688863521925156, "QA", "haiku"], "calM-haiku-12185-1": [true, 1.1832706463870615, "QA", "haiku"], "calM-sonnet-12185-0": [true, 1.4781416984203803, "QA", "sonnet"], "calM-sonnet-12185-1": [true, 1.9928389533008866, "QA", "sonnet"], "calM-haiku-12192-0": [true, 2.2625687615097103, "QA", "haiku"], "calM-haiku-12192-1": [true, 2.122070817992538, "QA", "haiku"], "calM-sonnet-12192-0": [true, 2.566583721803708, "QA", "sonnet"], "calM-sonnet-12192-1": [true, 3.0006002840245625, "QA", "sonnet"], "calM-haiku-12198-0": [true, 1.5662646692969326, "QA", "haiku"], "calM-haiku-12198-1": [true, 1.114591872694554, "QA", "haiku"], "calM-sonnet-12198-0": [true, 0.9466932277211609, "QA", "sonnet"], "calM-sonnet-12198-1": [true, 1.4376873102398933, "QA", "sonnet"], "calM-haiku-12200-0": [true, 0.7250368270754566, "QA", "haiku"], "calM-haiku-12200-1": [false, 1.8333480446398709, "QA", "haiku"], "calM-sonnet-12200-0": [true, 1.3218723391470146, "QA", "sonnet"], "calM-sonnet-12200-1": [true, 1.3611079696531898, "QA", "sonnet"], "calM-haiku-12202-0": [true, 0.8887594763271176, "QA", "haiku"], "calM-haiku-12202-1": [true, 0.9185642129335007, "QA", "haiku"], "calM-sonnet-12202-0": [true, 0.8887594763271176, "QA", "sonnet"], "calM-sonnet-12202-1": [true, 0.8887594763271176, "QA", "sonnet"], "calM-haiku-12224-0": [true, 1.5891910901393997, "QA", "haiku"], "calM-haiku-12224-1": [true, 1.5891910901393997, "QA", "haiku"], "calM-sonnet-12224-0": [true, 2.4782917802634428, "QA", "sonnet"], "calM-sonnet-12224-1": [true, 2.521795376970484, "QA", "sonnet"], "calM-haiku-12226-0": [true, 3.677681929094911, "QA", "haiku"], "calM-haiku-12226-1": [true, 3.6049418630411747, "QA", "haiku"], "calM-sonnet-12226-0": [true, 3.1171900702053894, "QA", "sonnet"], "calM-sonnet-12226-1": [true, 3.957876532632661, "QA", "sonnet"], "calM-haiku-12227-0": [false, 3.3537392897767, "QA", "haiku"], "calM-haiku-12227-1": [true, 1.8863727898227025, "QA", "haiku"], "calM-sonnet-12227-0": [true, 2.0528888642443954, "QA", "sonnet"], "calM-sonnet-12227-1": [true, 2.2166556388289886, "QA", "sonnet"], "calM-haiku-12232-0": [true, 2.0276396449661003, "QA", "haiku"], "calM-haiku-12232-1": [true, 2.5177262373164555, "QA", "haiku"], "calM-sonnet-12232-0": [true, 2.0441688488430683, "QA", "sonnet"], "calM-sonnet-12232-1": [true, 1.3512250135812423, "QA", "sonnet"], "calM-haiku-12233-0": [true, 2.7654532045558744, "QA", "haiku"], "calM-haiku-12233-1": [true, 1.8724029473614832, "QA", "haiku"], "calM-sonnet-12233-0": [true, 2.3445217434652625, "QA", "sonnet"], "calM-sonnet-12233-1": [true, 3.351205707498343, "QA", "sonnet"], "calM-haiku-12266-0": [true, 2.591446984298462, "QA", "haiku"], "calM-haiku-12266-1": [false, 2.683968566489191, "QA", "haiku"], "calM-sonnet-12266-0": [true, 2.0441249748677466, "QA", "sonnet"], "calM-sonnet-12266-1": [true, 1.720681958519181, "QA", "sonnet"], "calM-haiku-12290-0": [true, 1.141561103098928, "QA", "haiku"], "calM-haiku-12290-1": [true, 1.2555798250687262, "QA", "haiku"], "calM-sonnet-12290-0": [true, 1.3873315013798029, "QA", "sonnet"], "calM-sonnet-12290-1": [true, 1.4064872712933925, "QA", "sonnet"], "calM-haiku-12310-0": [true, 2.045965300552135, "QA", "haiku"], "calM-haiku-12310-1": [true, 1.2568455741280744, "QA", "haiku"], "calM-sonnet-12310-0": [true, 2.3020156566405383, "QA", "sonnet"], "calM-sonnet-12310-1": [true, 1.786674737959359, "QA", "sonnet"], "calM-haiku-12323-0": [true, 3.641375035716217, "QA", "haiku"], "calM-haiku-12323-1": [true, 2.067551420459601, "QA", "haiku"], "calM-sonnet-12323-0": [true, 1.5383170898232712, "QA", "sonnet"], "calM-sonnet-12323-1": [false, 1.6137594335661989, "QA", "sonnet"], "calM-haiku-12326-0": [true, 2.8990203904673626, "QA", "haiku"], "calM-haiku-12326-1": [true, 2.077424422430963, "QA", "haiku"], "calM-sonnet-12326-0": [true, 2.969776844793542, "QA", "sonnet"], "calM-sonnet-12326-1": [true, 2.969776844793542, "QA", "sonnet"], "calM-haiku-12329-0": [false, 2.160277003928014, "QA", "haiku"], "calM-haiku-12329-1": [false, 1.6640743628590786, "QA", "haiku"], "calM-sonnet-12329-0": [true, 1.5172610879046624, "QA", "sonnet"], "calM-sonnet-12329-1": [true, 1.0980103296314447, "QA", "sonnet"], "calM-haiku-12348-0": [true, 1.3678498540735595, "QA", "haiku"], "calM-haiku-12348-1": [true, 1.7780269967838713, "QA", "haiku"], "calM-sonnet-12348-0": [true, 2.1454567878921242, "QA", "sonnet"], "calM-sonnet-12348-1": [true, 1.9657366755686987, "QA", "sonnet"], "calM-haiku-12354-0": [true, 1.3141031608299252, "QA", "haiku"], "calM-haiku-12354-1": [true, 2.391452323639206, "QA", "haiku"], "calM-sonnet-12354-0": [true, 1.6798686844338695, "QA", "sonnet"], "calM-sonnet-12354-1": [true, 1.5275925926629115, "QA", "sonnet"], "calM-haiku-12361-0": [true, 3.2283287406688386, "QA", "haiku"], "calM-haiku-12361-1": [true, 3.016292318583112, "QA", "haiku"], "calM-sonnet-12361-0": [true, 2.0139107292647616, "QA", "sonnet"], "calM-sonnet-12361-1": [true, 1.279383243288588, "QA", "sonnet"], "calM-haiku-12365-0": [true, 2.355638268969198, "QA", "haiku"], "calM-haiku-12365-1": [false, 2.876910018601829, "QA", "haiku"], "calM-sonnet-12365-0": [true, 2.0737334294491805, "QA", "sonnet"], "calM-sonnet-12365-1": [true, 1.682909545446691, "QA", "sonnet"], "calM-haiku-12378-0": [true, 1.231724438413535, "QA", "haiku"], "calM-haiku-12378-1": [true, 1.4112857828542418, "QA", "haiku"], "calM-sonnet-12378-0": [true, 1.1334729704821558, "QA", "sonnet"], "calM-sonnet-12378-1": [true, 1.4932300401266172, "QA", "sonnet"], "calM-haiku-12399-0": [true, 0.9579914829647265, "QA", "haiku"], "calM-haiku-12399-1": [true, 0.6323339981554515, "QA", "haiku"], "calM-sonnet-12399-0": [true, 1.9571990479497845, "QA", "sonnet"], "calM-sonnet-12399-1": [true, 1.2565279989529026, "QA", "sonnet"], "calM-haiku-12401-0": [true, 0.5253232387660995, "QA", "haiku"], "calM-haiku-12401-1": [true, 0.7934612723290193, "QA", "haiku"], "calM-sonnet-12401-0": [true, 0.7095641822979255, "QA", "sonnet"], "calM-sonnet-12401-1": [true, 1.533213408173963, "QA", "sonnet"], "calM-haiku-12405-0": [true, 0.8537323084620059, "QA", "haiku"], "calM-haiku-12405-1": [true, 1.8109643442874717, "QA", "haiku"], "calM-sonnet-12405-0": [true, 1.8449126244956915, "QA", "sonnet"], "calM-sonnet-12405-1": [false, 3.267820229609573, "QA", "sonnet"], "calM-haiku-12410-0": [true, 1.5048625877719741, "QA", "haiku"], "calM-haiku-12410-1": [true, 0.7575324825545501, "QA", "haiku"], "calM-sonnet-12410-0": [true, 1.183754758450399, "QA", "sonnet"], "calM-sonnet-12410-1": [true, 0.5827607871405907, "QA", "sonnet"], "calM-haiku-12412-0": [true, 2.1815930868096727, "QA", "haiku"], "calM-haiku-12412-1": [true, 2.1071864012388053, "QA", "haiku"], "calM-sonnet-12412-0": [true, 2.7462750983451354, "QA", "sonnet"], "calM-sonnet-12412-1": [true, 1.4511486588165612, "QA", "sonnet"], "calM-haiku-12434-0": [true, 1.7234948894288702, "QA", "haiku"], "calM-haiku-12434-1": [true, 2.1277121041165814, "QA", "haiku"], "calM-sonnet-12434-0": [true, 2.2421559837155, "QA", "sonnet"], "calM-sonnet-12434-1": [true, 1.4745944030723699, "QA", "sonnet"], "calM-haiku-12454-0": [false, 2.549140371005353, "QA", "haiku"], "calM-haiku-12454-1": [false, 2.4263825491463793, "QA", "haiku"], "calM-sonnet-12454-0": [true, 1.4046852834645562, "QA", "sonnet"], "calM-sonnet-12454-1": [true, 1.9321937785819812, "QA", "sonnet"], "calM-haiku-12457-0": [true, 0.2511331419122207, "QA", "haiku"], "calM-haiku-12457-1": [true, 1.178754395753729, "QA", "haiku"], "calM-sonnet-12457-0": [true, 1.7303209073452939, "QA", "sonnet"], "calM-sonnet-12457-1": [true, 1.2368464630490674, "QA", "sonnet"], "calM-haiku-13508-0": [true, 0.5654736173500736, "Summary", "haiku"], "calM-haiku-13508-1": [false, 0.5654736173500736, "Summary", "haiku"], "calM-sonnet-13508-0": [true, 0.43777185416384945, "Summary", "sonnet"], "calM-sonnet-13508-1": [true, 1.9733594684416564, "Summary", "sonnet"], "calM-haiku-13515-0": [true, 1.5076621327415503, "Summary", "haiku"], "calM-haiku-13515-1": [true, 0.8764739626707108, "Summary", "haiku"], "calM-sonnet-13515-0": [false, 0.8411804775930618, "Summary", "sonnet"], "calM-sonnet-13515-1": [true, 1.1686557477144226, "Summary", "sonnet"], "calM-haiku-13523-0": [true, 0.24685529053623206, "Summary", "haiku"], "calM-haiku-13523-1": [true, 0.53630255647159, "Summary", "haiku"], "calM-sonnet-13523-0": [true, 0.13545095376936772, "Summary", "sonnet"], "calM-sonnet-13523-1": [true, 0.454810200872319, "Summary", "sonnet"], "calM-haiku-13548-0": [false, 0.07237498839922499, "Summary", "haiku"], "calM-haiku-13548-1": [false, 1.7997334398965081, "Summary", "haiku"], "calM-sonnet-13548-0": [false, 0.8379131344938251, "Summary", "sonnet"], "calM-sonnet-13548-1": [false, 1.4532088906159948, "Summary", "sonnet"], "calM-haiku-13549-0": [true, 2.424888449713176, "Summary", "haiku"], "calM-haiku-13549-1": [true, 2.0912929139153187, "Summary", "haiku"], "calM-sonnet-13549-0": [true, 0.8251973329256432, "Summary", "sonnet"], "calM-sonnet-13549-1": [true, 1.5380446264216336, "Summary", "sonnet"], "calM-haiku-13567-0": [false, 0.9534765341496384, "Summary", "haiku"], "calM-haiku-13567-1": [true, 0.0567274137905161, "Summary", "haiku"], "calM-sonnet-13567-0": [true, 0.07826970894325608, "Summary", "sonnet"], "calM-sonnet-13567-1": [true, 0.786218838061974, "Summary", "sonnet"], "calM-haiku-13575-0": [true, 0.07267495174682026, "Summary", "haiku"], "calM-haiku-13575-1": [true, -0.036324100747606225, "Summary", "haiku"], "calM-sonnet-13575-0": [true, 1.4489591571039842, "Summary", "sonnet"], "calM-sonnet-13575-1": [true, 0.16421963042394824, "Summary", "sonnet"], "calM-haiku-13582-0": [true, 1.2659374622385418, "Summary", "haiku"], "calM-haiku-13582-1": [true, 0.7774459687201934, "Summary", "haiku"], "calM-sonnet-13582-0": [true, 0.05343582475881836, "Summary", "sonnet"], "calM-sonnet-13582-1": [true, 1.1065469149044713, "Summary", "sonnet"], "calM-haiku-13586-0": [true, 0.38570573005305214, "Summary", "haiku"], "calM-haiku-13586-1": [false, 1.83973868902623, "Summary", "haiku"], "calM-sonnet-13586-0": [true, -0.1389772272302704, "Summary", "sonnet"], "calM-sonnet-13586-1": [true, -0.11188588192452381, "Summary", "sonnet"], "calM-haiku-13642-0": [true, 2.5840122760927953, "Data2txt", "haiku"], "calM-haiku-13642-1": [true, 3.0689218892119534, "Data2txt", "haiku"], "calM-sonnet-13642-0": [false, 3.5919299454466165, "Data2txt", "sonnet"], "calM-sonnet-13642-1": [false, 1.2280617878148132, "Data2txt", "sonnet"], "calM-haiku-13650-0": [true, 1.9590261241481846, "Data2txt", "haiku"], "calM-haiku-13650-1": [false, 2.4612768134211196, "Data2txt", "haiku"], "calM-sonnet-13650-0": [false, 1.92821135095134, "Data2txt", "sonnet"], "calM-sonnet-13650-1": [true, 1.7911262792731115, "Data2txt", "sonnet"], "calM-haiku-13658-0": [true, 0.9458886537445939, "Data2txt", "haiku"], "calM-haiku-13658-1": [true, 0.6465596262014227, "Data2txt", "haiku"], "calM-sonnet-13658-0": [true, 1.311937159091726, "Data2txt", "sonnet"], "calM-sonnet-13658-1": [true, 1.7431084412804183, "Data2txt", "sonnet"], "calM-haiku-13661-0": [true, 1.8108396464355634, "Data2txt", "haiku"], "calM-haiku-13661-1": [true, 1.6395221823618247, "Data2txt", "haiku"], "calM-sonnet-13661-0": [true, 2.1626619555714144, "Data2txt", "sonnet"], "calM-sonnet-13661-1": [true, 2.2537611874574908, "Data2txt", "sonnet"], "calM-haiku-13663-0": [true, 2.257041485617084, "Data2txt", "haiku"], "calM-haiku-13663-1": [false, 1.8672810566525313, "Data2txt", "haiku"], "calM-sonnet-13663-0": [true, 2.518550515727578, "Data2txt", "sonnet"], "calM-sonnet-13663-1": [true, 2.5003537375703204, "Data2txt", "sonnet"], "calM-haiku-13671-0": [true, 0.4610810001584295, "Data2txt", "haiku"], "calM-haiku-13671-1": [true, 0.7057277471115644, "Data2txt", "haiku"], "calM-sonnet-13671-0": [true, 0.5812066745647615, "Data2txt", "sonnet"], "calM-sonnet-13671-1": [true, 0.398469553762038, "Data2txt", "sonnet"], "calM-haiku-13674-0": [false, 2.121916791050909, "Data2txt", "haiku"], "calM-haiku-13674-1": [true, 1.9603653212115537, "Data2txt", "haiku"], "calM-sonnet-13674-0": [true, 2.2926822127193356, "Data2txt", "sonnet"], "calM-sonnet-13674-1": [true, 2.087811530731827, "Data2txt", "sonnet"], "calM-haiku-13677-0": [true, 1.9676977301623448, "Data2txt", "haiku"], "calM-haiku-13677-1": [false, 1.8862033991875813, "Data2txt", "haiku"], "calM-sonnet-13677-0": [true, 1.5896673632907419, "Data2txt", "sonnet"], "calM-sonnet-13677-1": [false, 1.92557839601487, "Data2txt", "sonnet"], "calM-haiku-13681-0": [true, 2.51316612501295, "Data2txt", "haiku"], "calM-haiku-13681-1": [true, 2.8061551870919956, "Data2txt", "haiku"], "calM-sonnet-13681-0": [true, 0.7772814544659437, "Data2txt", "sonnet"], "calM-sonnet-13681-1": [true, 0.4570281665893883, "Data2txt", "sonnet"], "calM-haiku-13695-0": [true, 3.596567769388482, "Data2txt", "haiku"], "calM-haiku-13695-1": [true, 0.6013787051461796, "Data2txt", "haiku"], "calM-sonnet-13695-0": [true, 0.6952878279382687, "Data2txt", "sonnet"], "calM-sonnet-13695-1": [true, 1.2192993319547598, "Data2txt", "sonnet"], "calM-haiku-13697-0": [true, 1.3079198170309991, "Data2txt", "haiku"], "calM-haiku-13697-1": [false, 2.314682423391572, "Data2txt", "haiku"], "calM-sonnet-13697-0": [true, 0.9880353662165948, "Data2txt", "sonnet"], "calM-sonnet-13697-1": [true, 1.1566919800467954, "Data2txt", "sonnet"], "calM-haiku-13711-0": [true, 2.0026306803953244, "Data2txt", "haiku"], "calM-haiku-13711-1": [false, 1.8856205673374258, "Data2txt", "haiku"], "calM-sonnet-13711-0": [true, 2.229375044475526, "Data2txt", "sonnet"], "calM-sonnet-13711-1": [true, 2.0663328902476303, "Data2txt", "sonnet"], "calM-haiku-13717-0": [false, 2.8731344760436723, "Data2txt", "haiku"], "calM-haiku-13717-1": [false, 3.3717301621019007, "Data2txt", "haiku"], "calM-sonnet-13717-0": [false, 3.203016101318002, "Data2txt", "sonnet"], "calM-sonnet-13717-1": [true, 1.7199890933619773, "Data2txt", "sonnet"], "calM-haiku-13719-0": [true, 0.9612592751021283, "Data2txt", "haiku"], "calM-haiku-13719-1": [true, 2.4049472346260328, "Data2txt", "haiku"], "calM-sonnet-13719-0": [true, 1.807355697063144, "Data2txt", "sonnet"], "calM-sonnet-13719-1": [true, 1.742049350546242, "Data2txt", "sonnet"], "calM-haiku-13749-0": [true, 1.6292759253474953, "Data2txt", "haiku"], "calM-haiku-13749-1": [false, 0.9130525035864099, "Data2txt", "haiku"], "calM-sonnet-13749-0": [true, 1.517476437655852, "Data2txt", "sonnet"], "calM-sonnet-13749-1": [true, 1.5316118706893969, "Data2txt", "sonnet"], "calM-haiku-13750-0": [false, 3.4898912290189994, "Data2txt", "haiku"], "calM-haiku-13750-1": [false, 4.131699057378865, "Data2txt", "haiku"], "calM-sonnet-13750-0": [true, 2.7121281813803724, "Data2txt", "sonnet"], "calM-sonnet-13750-1": [false, 2.440031919351392, "Data2txt", "sonnet"], "calM-haiku-13766-0": [true, 1.3682494741619227, "Data2txt", "haiku"], "calM-haiku-13766-1": [true, 1.717049559696808, "Data2txt", "haiku"], "calM-sonnet-13766-0": [true, 1.7408029826286036, "Data2txt", "sonnet"], "calM-sonnet-13766-1": [true, 1.8996922551099251, "Data2txt", "sonnet"], "calM-haiku-13773-0": [false, 1.5533314992679903, "Data2txt", "haiku"], "calM-haiku-13773-1": [false, 0.6623307780287995, "Data2txt", "haiku"], "calM-sonnet-13773-0": [true, 0.5253761628791589, "Data2txt", "sonnet"], "calM-sonnet-13773-1": [true, 1.055900307056237, "Data2txt", "sonnet"], "calM-haiku-13813-0": [true, 1.108727132149366, "Summary", "haiku"], "calM-haiku-13813-1": [true, 0.16251423043173202, "Summary", "haiku"], "calM-sonnet-13813-0": [true, 0.5358129762328598, "Summary", "sonnet"], "calM-sonnet-13813-1": [true, 0.40971772633435083, "Summary", "sonnet"], "calM-haiku-13851-0": [true, 0.03883901994302328, "Summary", "haiku"], "calM-haiku-13851-1": [true, 0.5616787392125155, "Summary", "haiku"], "calM-sonnet-13851-0": [true, 0.07120347542973215, "Summary", "sonnet"], "calM-sonnet-13851-1": [true, -0.1742517966999259, "Summary", "sonnet"], "calM-haiku-13852-0": [true, 0.10503715811835848, "Summary", "haiku"], "calM-haiku-13852-1": [true, -0.12342250049107265, "Summary", "haiku"], "calM-sonnet-13852-0": [true, 0.26832057221815764, "Summary", "sonnet"], "calM-sonnet-13852-1": [true, -0.057370653243022096, "Summary", "sonnet"], "calM-haiku-13869-0": [true, 1.090034307613777, "Summary", "haiku"], "calM-haiku-13869-1": [true, 1.195570209613567, "Summary", "haiku"], "calM-sonnet-13869-0": [true, 0.1340172672963605, "Summary", "sonnet"], "calM-sonnet-13869-1": [true, 0.209716614717386, "Summary", "sonnet"], "calM-haiku-13880-0": [true, 1.358816454730564, "Summary", "haiku"], "calM-haiku-13880-1": [true, 1.9485286674286884, "Summary", "haiku"], "calM-sonnet-13880-0": [true, 3.1625831343897772, "Summary", "sonnet"], "calM-sonnet-13880-1": [true, 3.1625831343897772, "Summary", "sonnet"], "calM-haiku-13910-0": [true, 1.5833036708388692, "Summary", "haiku"], "calM-haiku-13910-1": [true, -0.1244542886937888, "Summary", "haiku"], "calM-sonnet-13910-0": [true, 0.9686842453454886, "Summary", "sonnet"], "calM-sonnet-13910-1": [true, 1.0210108900956105, "Summary", "sonnet"], "calM-haiku-13920-0": [true, -0.15016305087662907, "Summary", "haiku"], "calM-haiku-13920-1": [true, 0.20876993836983668, "Summary", "haiku"], "calM-sonnet-13920-0": [false, 0.34374802493497014, "Summary", "sonnet"], "calM-sonnet-13920-1": [false, 0.45886078532618957, "Summary", "sonnet"], "calM-haiku-13948-0": [true, 1.0739296595443746, "Summary", "haiku"], "calM-haiku-13948-1": [true, 0.6026681749076721, "Summary", "haiku"], "calM-sonnet-13948-0": [true, 1.2266434827779928, "Summary", "sonnet"], "calM-sonnet-13948-1": [true, 0.4098644702241455, "Summary", "sonnet"], "calM-haiku-13967-0": [false, 1.0600559283280533, "Summary", "haiku"], "calM-haiku-13967-1": [true, 1.949364608377299, "Summary", "haiku"], "calM-sonnet-13967-0": [true, 1.7046531727905374, "Summary", "sonnet"], "calM-sonnet-13967-1": [true, 1.3979759792041453, "Summary", "sonnet"], "calM-haiku-13980-0": [true, 0.7435654997268036, "Summary", "haiku"], "calM-haiku-13980-1": [true, 0.543323486683599, "Summary", "haiku"], "calM-sonnet-13980-0": [true, 0.3456420945213952, "Summary", "sonnet"], "calM-sonnet-13980-1": [true, 1.0447443974265171, "Summary", "sonnet"], "calM-haiku-14016-0": [true, 1.3744756372851343, "Data2txt", "haiku"], "calM-haiku-14016-1": [true, 1.6421825796428529, "Data2txt", "haiku"], "calM-sonnet-14016-0": [false, 3.3433479116889226, "Data2txt", "sonnet"], "calM-sonnet-14016-1": [true, 1.4385799718889862, "Data2txt", "sonnet"], "calM-haiku-14018-0": [true, 2.5134814963692023, "Data2txt", "haiku"], "calM-haiku-14018-1": [true, 2.1500284525022635, "Data2txt", "haiku"], "calM-sonnet-14018-0": [true, 2.193823836866281, "Data2txt", "sonnet"], "calM-sonnet-14018-1": [true, 2.2427457847114773, "Data2txt", "sonnet"], "calM-haiku-14024-0": [true, 1.0793585450290886, "Data2txt", "haiku"], "calM-haiku-14024-1": [true, 1.7107222607539252, "Data2txt", "haiku"], "calM-sonnet-14024-0": [true, 2.528858207520009, "Data2txt", "sonnet"], "calM-sonnet-14024-1": [true, 2.6106675853775494, "Data2txt", "sonnet"], "calM-haiku-14041-0": [true, 3.008342078646095, "Data2txt", "haiku"], "calM-haiku-14041-1": [true, 2.3735220286975967, "Data2txt", "haiku"], "calM-sonnet-14041-0": [true, 1.715286309008979, "Data2txt", "sonnet"], "calM-sonnet-14041-1": [true, 1.8946070937075006, "Data2txt", "sonnet"], "calM-haiku-14058-0": [false, 1.4079935882096728, "Data2txt", "haiku"], "calM-haiku-14058-1": [true, 0.9824091627796759, "Data2txt", "haiku"], "calM-sonnet-14058-0": [true, 1.5225652858804446, "Data2txt", "sonnet"], "calM-sonnet-14058-1": [false, 1.0370152991745283, "Data2txt", "sonnet"], "calM-haiku-14066-0": [true, 2.2981130778821868, "Data2txt", "haiku"], "calM-haiku-14066-1": [true, 1.885222986112472, "Data2txt", "haiku"], "calM-sonnet-14066-0": [true, 2.162234682886732, "Data2txt", "sonnet"], "calM-sonnet-14066-1": [false, 2.2868917027856415, "Data2txt", "sonnet"], "calM-haiku-14073-0": [true, 1.5097048989451367, "Data2txt", "haiku"], "calM-haiku-14073-1": [true, 1.646287282673917, "Data2txt", "haiku"], "calM-sonnet-14073-0": [true, 1.7534567730305413, "Data2txt", "sonnet"], "calM-sonnet-14073-1": [true, 1.8151546535081189, "Data2txt", "sonnet"], "calM-haiku-14109-0": [false, 1.6100983398016193, "Data2txt", "haiku"], "calM-haiku-14109-1": [true, 1.8431834975157282, "Data2txt", "haiku"], "calM-sonnet-14109-0": [true, 1.9669042338628622, "Data2txt", "sonnet"], "calM-sonnet-14109-1": [true, 1.9465662903308445, "Data2txt", "sonnet"], "calM-haiku-14110-0": [true, 1.7732814339463794, "Data2txt", "haiku"], "calM-haiku-14110-1": [true, 1.7719021791999778, "Data2txt", "haiku"], "calM-sonnet-14110-0": [true, 1.846014748730259, "Data2txt", "sonnet"], "calM-sonnet-14110-1": [true, 1.1385123061629778, "Data2txt", "sonnet"], "calM-haiku-14117-0": [true, 1.4726096318339534, "Data2txt", "haiku"], "calM-haiku-14117-1": [true, 1.3811765780268044, "Data2txt", "haiku"], "calM-sonnet-14117-0": [true, 1.2912968425978915, "Data2txt", "sonnet"], "calM-sonnet-14117-1": [true, 2.5831266468529552, "Data2txt", "sonnet"], "calM-haiku-14119-0": [true, 3.1409910869257764, "Data2txt", "haiku"], "calM-haiku-14119-1": [true, 2.3237873678761747, "Data2txt", "haiku"], "calM-sonnet-14119-0": [true, 1.882324476151042, "Data2txt", "sonnet"], "calM-sonnet-14119-1": [true, 2.283881463318895, "Data2txt", "sonnet"], "calM-haiku-14126-0": [true, 2.9845729119081046, "Data2txt", "haiku"], "calM-haiku-14126-1": [true, 3.5550351784324135, "Data2txt", "haiku"], "calM-sonnet-14126-0": [true, 2.3439174337425306, "Data2txt", "sonnet"], "calM-sonnet-14126-1": [true, 1.9109553640046042, "Data2txt", "sonnet"], "calM-haiku-14127-0": [false, 1.8585316937037601, "Data2txt", "haiku"], "calM-haiku-14127-1": [true, 0.9543595558286443, "Data2txt", "haiku"], "calM-sonnet-14127-0": [true, 2.015800591379875, "Data2txt", "sonnet"], "calM-sonnet-14127-1": [true, 1.468669676938112, "Data2txt", "sonnet"], "calM-haiku-14146-0": [true, 1.3675072934264747, "Data2txt", "haiku"], "calM-haiku-14146-1": [true, 2.191829088879153, "Data2txt", "haiku"], "calM-sonnet-14146-0": [true, 1.339414749454071, "Data2txt", "sonnet"], "calM-sonnet-14146-1": [true, 1.2960449509383578, "Data2txt", "sonnet"], "calM-haiku-14148-0": [true, 2.1022807544330284, "Data2txt", "haiku"], "calM-haiku-14148-1": [true, 1.786243459416579, "Data2txt", "haiku"], "calM-sonnet-14148-0": [true, 2.5584599381606568, "Data2txt", "sonnet"], "calM-sonnet-14148-1": [true, 1.4534594640223375, "Data2txt", "sonnet"], "calM-haiku-14160-0": [false, 2.902761924520453, "Data2txt", "haiku"], "calM-haiku-14160-1": [false, 1.5443952195435924, "Data2txt", "haiku"], "calM-sonnet-14160-0": [true, 0.7911658029971215, "Data2txt", "sonnet"], "calM-sonnet-14160-1": [false, 1.5776008163011437, "Data2txt", "sonnet"], "calM-haiku-14190-0": [true, 0.9498767740697103, "Data2txt", "haiku"], "calM-haiku-14190-1": [false, 0.8833973409875654, "Data2txt", "haiku"], "calM-sonnet-14190-0": [true, 1.031928300513594, "Data2txt", "sonnet"], "calM-sonnet-14190-1": [true, 2.916889621539654, "Data2txt", "sonnet"], "calM-haiku-14207-0": [true, 1.0865081026259227, "Data2txt", "haiku"], "calM-haiku-14207-1": [true, 0.862895064386302, "Data2txt", "haiku"], "calM-sonnet-14207-0": [false, 1.4955419575848299, "Data2txt", "sonnet"], "calM-sonnet-14207-1": [true, 3.3018742506898016, "Data2txt", "sonnet"], "calM-haiku-14209-0": [true, 2.4499457554606856, "Data2txt", "haiku"], "calM-haiku-14209-1": [true, 2.601385496654601, "Data2txt", "haiku"], "calM-sonnet-14209-0": [true, 2.139064363582598, "Data2txt", "sonnet"], "calM-sonnet-14209-1": [true, 1.8524425941094362, "Data2txt", "sonnet"], "calM-haiku-14211-0": [true, 1.546941171955388, "Data2txt", "haiku"], "calM-haiku-14211-1": [false, 2.178984962732582, "Data2txt", "haiku"], "calM-sonnet-14211-0": [true, 1.8772673006481873, "Data2txt", "sonnet"], "calM-sonnet-14211-1": [true, 1.3520427616997033, "Data2txt", "sonnet"], "calM-haiku-14222-0": [true, 1.7725313443923871, "Data2txt", "haiku"], "calM-haiku-14222-1": [true, 1.8620756030738654, "Data2txt", "haiku"], "calM-sonnet-14222-0": [true, 1.8737691396934175, "Data2txt", "sonnet"], "calM-sonnet-14222-1": [true, 2.660854075246754, "Data2txt", "sonnet"], "calM-haiku-14226-0": [true, 2.4473925494421125, "Data2txt", "haiku"], "calM-haiku-14226-1": [true, 1.7205715854902506, "Data2txt", "haiku"], "calM-sonnet-14226-0": [true, 1.571741340098219, "Data2txt", "sonnet"], "calM-sonnet-14226-1": [true, 1.501711901209579, "Data2txt", "sonnet"], "calM-haiku-14227-0": [true, 2.3234321040937935, "Data2txt", "haiku"], "calM-haiku-14227-1": [false, 2.079364110430488, "Data2txt", "haiku"], "calM-sonnet-14227-0": [true, 1.9560674956521902, "Data2txt", "sonnet"], "calM-sonnet-14227-1": [true, 1.6011703810184836, "Data2txt", "sonnet"], "calM-haiku-14261-0": [false, 3.346393071023301, "Data2txt", "haiku"], "calM-haiku-14261-1": [false, 2.5394823047002895, "Data2txt", "haiku"], "calM-sonnet-14261-0": [true, 3.829777089248177, "Data2txt", "sonnet"], "calM-sonnet-14261-1": [true, 2.7187095207136287, "Data2txt", "sonnet"], "calM-haiku-14263-0": [true, 2.7777486680984307, "Data2txt", "haiku"], "calM-haiku-14263-1": [true, 2.8997294523932706, "Data2txt", "haiku"], "calM-sonnet-14263-0": [true, 1.139584940808774, "Data2txt", "sonnet"], "calM-sonnet-14263-1": [true, 1.2753538009097563, "Data2txt", "sonnet"], "calM-haiku-14277-0": [true, 1.8162532888671306, "Data2txt", "haiku"], "calM-haiku-14277-1": [true, 1.723524772383322, "Data2txt", "haiku"], "calM-sonnet-14277-0": [true, 1.643767417643494, "Data2txt", "sonnet"], "calM-sonnet-14277-1": [true, 1.5189398155885876, "Data2txt", "sonnet"], "calM-haiku-14283-0": [true, 1.279905592271999, "Data2txt", "haiku"], "calM-haiku-14283-1": [true, 1.6101248309249063, "Data2txt", "haiku"], "calM-sonnet-14283-0": [true, 1.761164043611152, "Data2txt", "sonnet"], "calM-sonnet-14283-1": [true, 1.5032789196251695, "Data2txt", "sonnet"], "calM-haiku-14302-0": [true, 2.1223262707230104, "QA", "haiku"], "calM-haiku-14302-1": [true, 2.080754001215526, "QA", "haiku"], "calM-sonnet-14302-0": [true, 2.3501644775062513, "QA", "sonnet"], "calM-sonnet-14302-1": [true, 2.5187533461038263, "QA", "sonnet"], "calM-haiku-14336-0": [true, 1.505497089175368, "QA", "haiku"], "calM-haiku-14336-1": [true, 1.5327827844646502, "QA", "haiku"], "calM-sonnet-14336-0": [true, 1.295127966050537, "QA", "sonnet"], "calM-sonnet-14336-1": [true, 1.5084710803837684, "QA", "sonnet"], "calM-haiku-14337-0": [true, 1.0265326923752982, "QA", "haiku"], "calM-haiku-14337-1": [true, 1.0848185370324033, "QA", "haiku"], "calM-sonnet-14337-0": [true, 1.0909000571870888, "QA", "sonnet"], "calM-sonnet-14337-1": [true, 1.3826286320630767, "QA", "sonnet"], "calM-haiku-14354-0": [true, 1.8427975089882283, "QA", "haiku"], "calM-haiku-14354-1": [true, 1.5832049943861295, "QA", "haiku"], "calM-sonnet-14354-0": [true, 1.603235885540985, "QA", "sonnet"], "calM-sonnet-14354-1": [true, 1.4317984483269344, "QA", "sonnet"], "calM-haiku-14359-0": [true, 0.3516026735528895, "QA", "haiku"], "calM-haiku-14359-1": [true, 1.4538208782195567, "QA", "haiku"], "calM-sonnet-14359-0": [true, 0.9546727644718257, "QA", "sonnet"], "calM-sonnet-14359-1": [true, 0.9546727644718257, "QA", "sonnet"], "calM-haiku-14379-0": [true, 1.4193354313837325, "QA", "haiku"], "calM-haiku-14379-1": [true, 2.6407216450624276, "QA", "haiku"], "calM-sonnet-14379-0": [false, 2.666912045674352, "QA", "sonnet"], "calM-sonnet-14379-1": [true, 2.5023021482908496, "QA", "sonnet"], "calM-haiku-14393-0": [false, 2.0951098386228773, "QA", "haiku"], "calM-haiku-14393-1": [true, 1.9710048219856033, "QA", "haiku"], "calM-sonnet-14393-0": [true, 2.030601084856058, "QA", "sonnet"], "calM-sonnet-14393-1": [true, 1.724752717065211, "QA", "sonnet"], "calM-haiku-14399-0": [true, 0.22638535825178216, "QA", "haiku"], "calM-haiku-14399-1": [true, 0.79462502467387, "QA", "haiku"], "calM-sonnet-14399-0": [true, 1.6712930094606897, "QA", "sonnet"], "calM-sonnet-14399-1": [true, 2.304829741025952, "QA", "sonnet"], "calM-haiku-14434-0": [true, 3.006926385094453, "QA", "haiku"], "calM-haiku-14434-1": [true, 1.900320159855996, "QA", "haiku"], "calM-sonnet-14434-0": [true, 1.6024055900284262, "QA", "sonnet"], "calM-sonnet-14434-1": [true, 1.780770632727196, "QA", "sonnet"], "calM-haiku-14436-0": [false, 1.9126982574530973, "QA", "haiku"], "calM-haiku-14436-1": [true, 1.6378889989940482, "QA", "haiku"], "calM-sonnet-14436-0": [true, 1.8375104664021396, "QA", "sonnet"], "calM-sonnet-14436-1": [true, 2.432967127469211, "QA", "sonnet"], "calM-haiku-14438-0": [true, 1.8767876188708148, "QA", "haiku"], "calM-haiku-14438-1": [true, 1.698130853137887, "QA", "haiku"], "calM-sonnet-14438-0": [true, 1.716294722783094, "QA", "sonnet"], "calM-sonnet-14438-1": [true, 1.533536858966301, "QA", "sonnet"], "calM-haiku-14447-0": [false, 1.4664027136976414, "QA", "haiku"], "calM-haiku-14447-1": [true, 1.5216323307051742, "QA", "haiku"], "calM-sonnet-14447-0": [true, 2.4736057416717063, "QA", "sonnet"], "calM-sonnet-14447-1": [true, 1.8152435880742068, "QA", "sonnet"], "calM-haiku-14455-0": [true, 1.6942496018299804, "QA", "haiku"], "calM-haiku-14455-1": [false, 1.6404371067191967, "QA", "haiku"], "calM-sonnet-14455-0": [true, 1.782645229598323, "QA", "sonnet"], "calM-sonnet-14455-1": [true, 2.265402061930627, "QA", "sonnet"], "calM-haiku-14467-0": [true, 0.4864963198579315, "QA", "haiku"], "calM-haiku-14467-1": [true, 1.0433563234800924, "QA", "haiku"], "calM-sonnet-14467-0": [true, 1.6647720897307465, "QA", "sonnet"], "calM-sonnet-14467-1": [true, 1.0792993536376119, "QA", "sonnet"], "calM-haiku-14484-0": [true, 0.7117616074954539, "QA", "haiku"], "calM-haiku-14484-1": [true, 0.71433221350597, "QA", "haiku"], "calM-sonnet-14484-0": [true, 1.7793563633302591, "QA", "sonnet"], "calM-sonnet-14484-1": [true, 1.7733151253776882, "QA", "sonnet"], "calM-haiku-14486-0": [false, 1.291826274724352, "QA", "haiku"], "calM-haiku-14486-1": [false, 1.464178043969139, "QA", "haiku"], "calM-sonnet-14486-0": [true, 1.2699733782232072, "QA", "sonnet"], "calM-sonnet-14486-1": [true, 1.5249862974635262, "QA", "sonnet"], "calM-haiku-14502-0": [true, 2.297902532713819, "Data2txt", "haiku"], "calM-haiku-14502-1": [true, 2.8453919648038815, "Data2txt", "haiku"], "calM-sonnet-14502-0": [true, 1.9689189068127684, "Data2txt", "sonnet"], "calM-sonnet-14502-1": [true, 1.0300471354075436, "Data2txt", "sonnet"], "calM-haiku-14510-0": [true, 0.48765955718241283, "Data2txt", "haiku"], "calM-haiku-14510-1": [false, 1.5574942302553265, "Data2txt", "haiku"], "calM-sonnet-14510-0": [true, 2.781377194989287, "Data2txt", "sonnet"], "calM-sonnet-14510-1": [true, 1.800919172297041, "Data2txt", "sonnet"], "calM-haiku-14521-0": [true, 2.87504487436321, "Data2txt", "haiku"], "calM-haiku-14521-1": [true, 2.1137088719163737, "Data2txt", "haiku"], "calM-sonnet-14521-0": [true, 2.481292525010746, "Data2txt", "sonnet"], "calM-sonnet-14521-1": [true, 2.8968595585545796, "Data2txt", "sonnet"], "calM-haiku-14565-0": [true, 1.8386444573807128, "Data2txt", "haiku"], "calM-haiku-14565-1": [true, 1.1250074799531846, "Data2txt", "haiku"], "calM-sonnet-14565-0": [true, 0.7910898441773434, "Data2txt", "sonnet"], "calM-sonnet-14565-1": [true, 2.433538170188501, "Data2txt", "sonnet"], "calM-haiku-14568-0": [true, 3.379322274846298, "Data2txt", "haiku"], "calM-haiku-14568-1": [false, 1.756174617265268, "Data2txt", "haiku"], "calM-sonnet-14568-0": [true, 1.3862467602633273, "Data2txt", "sonnet"], "calM-sonnet-14568-1": [true, 1.9539774736857791, "Data2txt", "sonnet"], "calM-haiku-14576-0": [true, 1.927305679583749, "Data2txt", "haiku"], "calM-haiku-14576-1": [true, 3.2382564810975873, "Data2txt", "haiku"], "calM-sonnet-14576-0": [true, 1.7187099695369217, "Data2txt", "sonnet"], "calM-sonnet-14576-1": [true, 1.9506757594454114, "Data2txt", "sonnet"], "calM-haiku-14579-0": [true, 2.9170053229308794, "Data2txt", "haiku"], "calM-haiku-14579-1": [true, 1.4110725994744104, "Data2txt", "haiku"], "calM-sonnet-14579-0": [false, 2.997812187017283, "Data2txt", "sonnet"], "calM-sonnet-14579-1": [true, 2.7410532539754975, "Data2txt", "sonnet"], "calM-haiku-14629-0": [true, 1.7711554180224907, "Data2txt", "haiku"], "calM-haiku-14629-1": [true, 1.1569487572436383, "Data2txt", "haiku"], "calM-sonnet-14629-0": [true, 1.3767936511797365, "Data2txt", "sonnet"], "calM-sonnet-14629-1": [true, 3.284031382700734, "Data2txt", "sonnet"], "calM-haiku-14631-0": [true, 2.7996398222056027, "Data2txt", "haiku"], "calM-haiku-14631-1": [true, 2.332233573027426, "Data2txt", "haiku"], "calM-sonnet-14631-0": [true, 1.4242484660346528, "Data2txt", "sonnet"], "calM-sonnet-14631-1": [true, 2.1534352692767236, "Data2txt", "sonnet"], "calM-haiku-14633-0": [false, 3.284299767843438, "Data2txt", "haiku"], "calM-haiku-14633-1": [true, 1.482556956144212, "Data2txt", "haiku"], "calM-sonnet-14633-0": [true, 2.3280884538676663, "Data2txt", "sonnet"], "calM-sonnet-14633-1": [true, 3.496801540997042, "Data2txt", "sonnet"], "calM-haiku-14649-0": [true, 1.964862463727826, "Data2txt", "haiku"], "calM-haiku-14649-1": [false, 2.300927423439191, "Data2txt", "haiku"], "calM-sonnet-14649-0": [true, 3.3805001727912947, "Data2txt", "sonnet"], "calM-sonnet-14649-1": [false, 1.4752496305037397, "Data2txt", "sonnet"], "calM-haiku-14655-0": [true, 2.1134061611948467, "Data2txt", "haiku"], "calM-haiku-14655-1": [true, 2.393300260394948, "Data2txt", "haiku"], "calM-sonnet-14655-0": [true, 3.5036199937744055, "Data2txt", "sonnet"], "calM-sonnet-14655-1": [false, 2.5142358123760298, "Data2txt", "sonnet"], "calM-haiku-14657-0": [true, 2.1945363364849766, "Data2txt", "haiku"], "calM-haiku-14657-1": [true, 1.9981052747644306, "Data2txt", "haiku"], "calM-sonnet-14657-0": [true, 1.9262292202648594, "Data2txt", "sonnet"], "calM-sonnet-14657-1": [true, 2.328535225125088, "Data2txt", "sonnet"], "calM-haiku-14659-0": [true, 2.211940486304724, "Data2txt", "haiku"], "calM-haiku-14659-1": [true, 1.0469904641212417, "Data2txt", "haiku"], "calM-sonnet-14659-0": [true, 1.4997485541666835, "Data2txt", "sonnet"], "calM-sonnet-14659-1": [true, 1.6532648041782632, "Data2txt", "sonnet"], "calM-haiku-14673-0": [true, 1.7778031941270112, "Data2txt", "haiku"], "calM-haiku-14673-1": [true, 0.45699973557599805, "Data2txt", "haiku"], "calM-sonnet-14673-0": [false, 1.1155218867396828, "Data2txt", "sonnet"], "calM-sonnet-14673-1": [true, 3.4379409874844313, "Data2txt", "sonnet"], "calM-haiku-14681-0": [true, 1.8560209087051038, "Data2txt", "haiku"], "calM-haiku-14681-1": [true, 1.4048894989183476, "Data2txt", "haiku"], "calM-sonnet-14681-0": [true, 0.6119512042844288, "Data2txt", "sonnet"], "calM-sonnet-14681-1": [true, 0.65536982404397, "Data2txt", "sonnet"], "calM-haiku-14682-0": [false, 2.4211951396718465, "Data2txt", "haiku"], "calM-haiku-14682-1": [true, 2.1346267453120813, "Data2txt", "haiku"], "calM-sonnet-14682-0": [true, 1.6657708620215739, "Data2txt", "sonnet"], "calM-sonnet-14682-1": [true, 1.5876731588586266, "Data2txt", "sonnet"], "calM-haiku-14695-0": [true, 2.062303898869136, "Data2txt", "haiku"], "calM-haiku-14695-1": [true, 1.5094758112600601, "Data2txt", "haiku"], "calM-sonnet-14695-0": [true, 2.102062982230354, "Data2txt", "sonnet"], "calM-sonnet-14695-1": [true, 2.146806458000793, "Data2txt", "sonnet"], "calM-haiku-14715-0": [true, 3.1485920149458524, "Data2txt", "haiku"], "calM-haiku-14715-1": [true, 2.614446045927179, "Data2txt", "haiku"], "calM-sonnet-14715-0": [true, 2.6755795365141317, "Data2txt", "sonnet"], "calM-sonnet-14715-1": [true, 2.8199789355914566, "Data2txt", "sonnet"], "calM-haiku-14739-0": [true, 1.6693760902768948, "Data2txt", "haiku"], "calM-haiku-14739-1": [true, 1.0459176651767645, "Data2txt", "haiku"], "calM-sonnet-14739-0": [true, 1.5676646889489287, "Data2txt", "sonnet"], "calM-sonnet-14739-1": [true, 1.5299844501887019, "Data2txt", "sonnet"], "calM-haiku-14763-0": [true, 1.1604318686354145, "Data2txt", "haiku"], "calM-haiku-14763-1": [true, 1.193654136566734, "Data2txt", "haiku"], "calM-sonnet-14763-0": [true, 0.9596136287801813, "Data2txt", "sonnet"], "calM-sonnet-14763-1": [true, 0.9644947352278654, "Data2txt", "sonnet"], "calM-haiku-14764-0": [true, 3.0120699368231576, "Data2txt", "haiku"], "calM-haiku-14764-1": [true, 2.5855832609567493, "Data2txt", "haiku"], "calM-sonnet-14764-0": [false, 1.5426165588849339, "Data2txt", "sonnet"], "calM-sonnet-14764-1": [true, 1.756992542644388, "Data2txt", "sonnet"], "calM-haiku-14819-0": [true, 1.0998906366736845, "Data2txt", "haiku"], "calM-haiku-14819-1": [false, 1.8882395332484558, "Data2txt", "haiku"], "calM-sonnet-14819-0": [true, 2.3525195764342715, "Data2txt", "sonnet"], "calM-sonnet-14819-1": [true, 2.5486471758131755, "Data2txt", "sonnet"], "calM-haiku-14848-0": [true, 3.0992173719865583, "Data2txt", "haiku"], "calM-haiku-14848-1": [true, 2.301540686288527, "Data2txt", "haiku"], "calM-sonnet-14848-0": [true, 0.8947179874284452, "Data2txt", "sonnet"], "calM-sonnet-14848-1": [true, 2.3560322047438897, "Data2txt", "sonnet"], "calM-haiku-14849-0": [true, 2.2648453647202675, "Data2txt", "haiku"], "calM-haiku-14849-1": [true, 2.019016242277748, "Data2txt", "haiku"], "calM-sonnet-14849-0": [true, 2.3444884897480565, "Data2txt", "sonnet"], "calM-sonnet-14849-1": [true, 2.557341007088054, "Data2txt", "sonnet"], "calM-haiku-14856-0": [false, 1.2156344076305696, "Data2txt", "haiku"], "calM-haiku-14856-1": [true, 2.0683029846360377, "Data2txt", "haiku"], "calM-sonnet-14856-0": [true, 1.10214283345273, "Data2txt", "sonnet"], "calM-sonnet-14856-1": [true, 1.2259782516263562, "Data2txt", "sonnet"], "calM-haiku-14858-0": [false, 1.7309328949488556, "Data2txt", "haiku"], "calM-haiku-14858-1": [false, 2.4754189938001767, "Data2txt", "haiku"], "calM-sonnet-14858-0": [true, 0.6819646521996662, "Data2txt", "sonnet"], "calM-sonnet-14858-1": [true, 1.237739292596929, "Data2txt", "sonnet"], "calM-haiku-14861-0": [false, 3.2248945268460716, "Data2txt", "haiku"], "calM-haiku-14861-1": [false, 3.128492674504308, "Data2txt", "haiku"], "calM-sonnet-14861-0": [true, 2.9072762204642464, "Data2txt", "sonnet"], "calM-sonnet-14861-1": [true, 2.950799419622764, "Data2txt", "sonnet"], "calM-haiku-14868-0": [false, 3.096171223983348, "Data2txt", "haiku"], "calM-haiku-14868-1": [false, 3.149456636037234, "Data2txt", "haiku"], "calM-sonnet-14868-0": [true, 3.728971336639676, "Data2txt", "sonnet"], "calM-sonnet-14868-1": [true, 3.393531483566654, "Data2txt", "sonnet"], "calM-haiku-14898-0": [true, 2.0121441691819513, "Data2txt", "haiku"], "calM-haiku-14898-1": [false, 2.6438684041247495, "Data2txt", "haiku"], "calM-sonnet-14898-0": [true, 2.5696256472784285, "Data2txt", "sonnet"], "calM-sonnet-14898-1": [true, 2.9171352437531453, "Data2txt", "sonnet"], "calM-haiku-14901-0": [true, 2.582175744052571, "Data2txt", "haiku"], "calM-haiku-14901-1": [true, 3.2134496768876994, "Data2txt", "haiku"], "calM-sonnet-14901-0": [true, 2.164832572704927, "Data2txt", "sonnet"], "calM-sonnet-14901-1": [true, 3.1066801074637014, "Data2txt", "sonnet"], "calM-haiku-14916-0": [true, 2.6329595596992497, "Data2txt", "haiku"], "calM-haiku-14916-1": [true, 2.481093068163832, "Data2txt", "haiku"], "calM-sonnet-14916-0": [true, 1.960517733185526, "Data2txt", "sonnet"], "calM-sonnet-14916-1": [true, 2.1147566304065397, "Data2txt", "sonnet"], "calM-haiku-14925-0": [true, 2.268644279744853, "Data2txt", "haiku"], "calM-haiku-14925-1": [true, 1.9950693836682412, "Data2txt", "haiku"], "calM-sonnet-14925-0": [true, 2.4238949230408315, "Data2txt", "sonnet"], "calM-sonnet-14925-1": [true, 2.7078264345150815, "Data2txt", "sonnet"], "calM-haiku-14954-0": [true, 1.6892173448357484, "Data2txt", "haiku"], "calM-haiku-14954-1": [false, 1.0832698000772925, "Data2txt", "haiku"], "calM-sonnet-14954-0": [true, 0.48908942094449886, "Data2txt", "sonnet"], "calM-sonnet-14954-1": [true, 0.913475681782665, "Data2txt", "sonnet"], "calM-haiku-14956-0": [false, 3.0478022779210523, "Data2txt", "haiku"], "calM-haiku-14956-1": [true, 3.188744271894185, "Data2txt", "haiku"], "calM-sonnet-14956-0": [true, 1.4674846825638304, "Data2txt", "sonnet"], "calM-sonnet-14956-1": [true, 1.404365440450433, "Data2txt", "sonnet"], "calM-haiku-14968-0": [true, 2.69599034829474, "Data2txt", "haiku"], "calM-haiku-14968-1": [true, 2.8841400002576805, "Data2txt", "haiku"], "calM-sonnet-14968-0": [true, 2.288967468139508, "Data2txt", "sonnet"], "calM-sonnet-14968-1": [true, 2.266790067932676, "Data2txt", "sonnet"], "calM-haiku-14975-0": [false, 2.5129000795779186, "Data2txt", "haiku"], "calM-haiku-14975-1": [true, 1.8418209978924796, "Data2txt", "haiku"], "calM-sonnet-14975-0": [true, 2.422441202442577, "Data2txt", "sonnet"], "calM-sonnet-14975-1": [true, 2.36531160195662, "Data2txt", "sonnet"], "calM-haiku-14976-0": [false, 3.5516062043416117, "Data2txt", "haiku"], "calM-haiku-14976-1": [true, 2.5221157635396194, "Data2txt", "haiku"], "calM-sonnet-14976-0": [true, 1.338738357299537, "Data2txt", "sonnet"], "calM-sonnet-14976-1": [true, 1.2865815315367484, "Data2txt", "sonnet"], "calM-haiku-14978-0": [true, 0.8577315611223482, "Data2txt", "haiku"], "calM-haiku-14978-1": [true, 1.713133658224361, "Data2txt", "haiku"], "calM-sonnet-14978-0": [true, 1.1762345841343038, "Data2txt", "sonnet"], "calM-sonnet-14978-1": [true, 1.6643301751493293, "Data2txt", "sonnet"], "calM-haiku-15001-0": [true, 1.8830890943611416, "Data2txt", "haiku"], "calM-haiku-15001-1": [true, 1.6744562551902171, "Data2txt", "haiku"], "calM-sonnet-15001-0": [true, 1.0838241063972112, "Data2txt", "sonnet"], "calM-sonnet-15001-1": [true, 0.7464304010007372, "Data2txt", "sonnet"], "calM-haiku-15004-0": [true, 2.61384841364341, "Data2txt", "haiku"], "calM-haiku-15004-1": [false, 1.4728840228205453, "Data2txt", "haiku"], "calM-sonnet-15004-0": [true, 1.5694121052380614, "Data2txt", "sonnet"], "calM-sonnet-15004-1": [true, 0.9814335610843201, "Data2txt", "sonnet"], "calM-haiku-15013-0": [false, 2.061843252107379, "Data2txt", "haiku"], "calM-haiku-15013-1": [false, 2.304500779063989, "Data2txt", "haiku"], "calM-sonnet-15013-0": [false, 2.520479641003873, "Data2txt", "sonnet"], "calM-sonnet-15013-1": [false, 2.520479641003873, "Data2txt", "sonnet"], "calM-haiku-15023-0": [true, 1.9913097621769387, "Data2txt", "haiku"], "calM-haiku-15023-1": [true, 3.035899565741939, "Data2txt", "haiku"], "calM-sonnet-15023-0": [true, 2.5773936149018892, "Data2txt", "sonnet"], "calM-sonnet-15023-1": [true, 1.467130351509661, "Data2txt", "sonnet"], "calM-haiku-15024-0": [true, 1.4350652563766775, "Data2txt", "haiku"], "calM-haiku-15024-1": [false, 1.5450935308848925, "Data2txt", "haiku"], "calM-sonnet-15024-0": [true, 2.8487322854679915, "Data2txt", "sonnet"], "calM-sonnet-15024-1": [true, 1.0525718754773845, "Data2txt", "sonnet"], "calM-haiku-15026-0": [true, 2.2908970971786244, "Data2txt", "haiku"], "calM-haiku-15026-1": [true, 2.1746722377662993, "Data2txt", "haiku"], "calM-sonnet-15026-0": [true, 1.3450471153095835, "Data2txt", "sonnet"], "calM-sonnet-15026-1": [true, 1.3851730997671947, "Data2txt", "sonnet"], "calM-haiku-15027-0": [false, 1.6675015419113346, "Data2txt", "haiku"], "calM-haiku-15027-1": [true, 1.4861277168916254, "Data2txt", "haiku"], "calM-sonnet-15027-0": [true, 2.1599931467566216, "Data2txt", "sonnet"], "calM-sonnet-15027-1": [true, 1.5410802826557637, "Data2txt", "sonnet"], "calM-haiku-15034-0": [true, 2.738189318722271, "Data2txt", "haiku"], "calM-haiku-15034-1": [false, 2.1882810170762443, "Data2txt", "haiku"], "calM-sonnet-15034-0": [true, 2.484408312840434, "Data2txt", "sonnet"], "calM-sonnet-15034-1": [true, 1.9738795062520378, "Data2txt", "sonnet"], "calM-haiku-15038-0": [true, 2.472040142771448, "Data2txt", "haiku"], "calM-haiku-15038-1": [true, 1.199981659272099, "Data2txt", "haiku"], "calM-sonnet-15038-0": [true, 1.4781987209462115, "Data2txt", "sonnet"], "calM-sonnet-15038-1": [true, 2.143028930256282, "Data2txt", "sonnet"], "calM-haiku-15049-0": [true, 1.2614142565022228, "Data2txt", "haiku"], "calM-haiku-15049-1": [false, 2.0525008715407034, "Data2txt", "haiku"], "calM-sonnet-15049-0": [true, 2.384734034242175, "Data2txt", "sonnet"], "calM-sonnet-15049-1": [false, 1.9341974363616337, "Data2txt", "sonnet"], "calM-haiku-15053-0": [false, 1.7310127857958375, "Data2txt", "haiku"], "calM-haiku-15053-1": [true, 3.431198740694054, "Data2txt", "haiku"], "calM-sonnet-15053-0": [true, 2.6575055015857583, "Data2txt", "sonnet"], "calM-sonnet-15053-1": [true, 1.988554673748344, "Data2txt", "sonnet"], "calM-haiku-15055-0": [true, 1.2128855701178696, "Data2txt", "haiku"], "calM-haiku-15055-1": [true, 1.2255310809504512, "Data2txt", "haiku"], "calM-sonnet-15055-0": [true, 1.7546806005758775, "Data2txt", "sonnet"], "calM-sonnet-15055-1": [true, 1.9314647670829959, "Data2txt", "sonnet"], "calM-haiku-15075-0": [false, 2.3909164838891157, "Data2txt", "haiku"], "calM-haiku-15075-1": [false, 2.3988549386196123, "Data2txt", "haiku"], "calM-sonnet-15075-0": [true, 2.2538363141831423, "Data2txt", "sonnet"], "calM-sonnet-15075-1": [true, 2.7527237293158606, "Data2txt", "sonnet"], "calM-haiku-15076-0": [true, 1.3656860768402328, "Data2txt", "haiku"], "calM-haiku-15076-1": [true, 1.3307048139651627, "Data2txt", "haiku"], "calM-sonnet-15076-0": [true, 2.432332648713942, "Data2txt", "sonnet"], "calM-sonnet-15076-1": [true, 2.4836150824319403, "Data2txt", "sonnet"], "calM-haiku-15091-0": [true, 3.0309786015200073, "Data2txt", "haiku"], "calM-haiku-15091-1": [true, 3.488322225417055, "Data2txt", "haiku"], "calM-sonnet-15091-0": [false, 2.9201549380806298, "Data2txt", "sonnet"], "calM-sonnet-15091-1": [false, 2.4950025760291155, "Data2txt", "sonnet"], "calM-haiku-15115-0": [false, 1.9271007492008039, "QA", "haiku"], "calM-haiku-15115-1": [true, 1.7582293039010635, "QA", "haiku"], "calM-sonnet-15115-0": [true, 1.147273048614183, "QA", "sonnet"], "calM-sonnet-15115-1": [true, 2.6950628933048786, "QA", "sonnet"], "calM-haiku-15118-0": [false, 1.036048813333057, "QA", "haiku"], "calM-haiku-15118-1": [true, 0.12054069091495336, "QA", "haiku"], "calM-sonnet-15118-0": [true, 0.8415630319031473, "QA", "sonnet"], "calM-sonnet-15118-1": [true, 1.2030216862286218, "QA", "sonnet"], "calM-haiku-15131-0": [true, 0.2949072689983994, "QA", "haiku"], "calM-haiku-15131-1": [true, 1.0090910057651954, "QA", "haiku"], "calM-sonnet-15131-0": [true, 1.2383262989605515, "QA", "sonnet"], "calM-sonnet-15131-1": [true, 1.3919796595349563, "QA", "sonnet"], "calM-haiku-15139-0": [true, 1.4969914897805965, "QA", "haiku"], "calM-haiku-15139-1": [true, 3.3980660215469833, "QA", "haiku"], "calM-sonnet-15139-0": [false, 2.9121343827977144, "QA", "sonnet"], "calM-sonnet-15139-1": [true, 2.5989300524046435, "QA", "sonnet"], "calM-haiku-15141-0": [true, 1.2818737506946478, "QA", "haiku"], "calM-haiku-15141-1": [true, 1.1785540975277056, "QA", "haiku"], "calM-sonnet-15141-0": [true, 0.9974221266687082, "QA", "sonnet"], "calM-sonnet-15141-1": [true, 1.175080453680117, "QA", "sonnet"], "calM-haiku-15145-0": [true, 1.0185728897744608, "QA", "haiku"], "calM-haiku-15145-1": [true, 0.9663288699590173, "QA", "haiku"], "calM-sonnet-15145-0": [true, 0.9546100573662656, "QA", "sonnet"], "calM-sonnet-15145-1": [true, 0.9541016087779651, "QA", "sonnet"], "calM-haiku-15148-0": [true, 0.626554303483226, "QA", "haiku"], "calM-haiku-15148-1": [true, 1.220305170178057, "QA", "haiku"], "calM-sonnet-15148-0": [true, 1.3439496522314014, "QA", "sonnet"], "calM-sonnet-15148-1": [true, 1.3059788705473085, "QA", "sonnet"], "calM-haiku-15170-0": [true, 2.953084636238043, "QA", "haiku"], "calM-haiku-15170-1": [false, 3.0061287017518685, "QA", "haiku"], "calM-sonnet-15170-0": [true, 2.8278297614984544, "QA", "sonnet"], "calM-sonnet-15170-1": [true, 2.271654705834149, "QA", "sonnet"], "calM-haiku-15184-0": [false, 3.225528824416336, "QA", "haiku"], "calM-haiku-15184-1": [true, 2.996684716902967, "QA", "haiku"], "calM-sonnet-15184-0": [true, 2.1517136557541945, "QA", "sonnet"], "calM-sonnet-15184-1": [true, 2.1382643858285926, "QA", "sonnet"], "calM-haiku-15198-0": [true, 0.9806973285913599, "QA", "haiku"], "calM-haiku-15198-1": [true, 1.4419374565896512, "QA", "haiku"], "calM-sonnet-15198-0": [true, 3.1805610477302757, "QA", "sonnet"], "calM-sonnet-15198-1": [true, 3.292711783304619, "QA", "sonnet"], "calM-haiku-15201-0": [true, 0.14338513675077819, "QA", "haiku"], "calM-haiku-15201-1": [true, 0.5855964952135567, "QA", "haiku"], "calM-sonnet-15201-0": [true, 0.7735832877486096, "QA", "sonnet"], "calM-sonnet-15201-1": [true, -0.25691462737926163, "QA", "sonnet"], "calM-haiku-15228-0": [true, 1.25667640703095, "QA", "haiku"], "calM-haiku-15228-1": [true, 1.2164830998064078, "QA", "haiku"], "calM-sonnet-15228-0": [true, 0.7556255258521447, "QA", "sonnet"], "calM-sonnet-15228-1": [true, 0.6112147906064647, "QA", "sonnet"], "calM-haiku-15237-0": [true, 3.309967411805692, "QA", "haiku"], "calM-haiku-15237-1": [true, 1.3052963549848289, "QA", "haiku"], "calM-sonnet-15237-0": [true, 1.5882496884770905, "QA", "sonnet"], "calM-sonnet-15237-1": [true, 1.522488785467827, "QA", "sonnet"], "calM-haiku-15241-0": [false, 3.327105806691695, "QA", "haiku"], "calM-haiku-15241-1": [false, 3.672331805443134, "QA", "haiku"], "calM-sonnet-15241-0": [true, 1.566201220423691, "QA", "sonnet"], "calM-sonnet-15241-1": [false, 2.1106749355097234, "QA", "sonnet"], "calM-haiku-15270-0": [true, 0.7490719486559758, "QA", "haiku"], "calM-haiku-15270-1": [true, 1.0416395420232698, "QA", "haiku"], "calM-sonnet-15270-0": [true, 1.3625813366397854, "QA", "sonnet"], "calM-sonnet-15270-1": [true, 0.8598580034362764, "QA", "sonnet"], "calM-haiku-15272-0": [true, 1.5259291863194862, "QA", "haiku"], "calM-haiku-15272-1": [true, 1.9570636591766117, "QA", "haiku"], "calM-sonnet-15272-0": [true, 2.5165399692096004, "QA", "sonnet"], "calM-sonnet-15272-1": [true, 2.5564096481566594, "QA", "sonnet"], "calM-haiku-15289-0": [true, 1.4934547201738546, "QA", "haiku"], "calM-haiku-15289-1": [true, 2.204000254145347, "QA", "haiku"], "calM-sonnet-15289-0": [true, 1.6693349720563089, "QA", "sonnet"], "calM-sonnet-15289-1": [true, 1.6693349720563089, "QA", "sonnet"], "calM-haiku-15292-0": [true, 1.1729917506302874, "QA", "haiku"], "calM-haiku-15292-1": [true, 1.2805867415264451, "QA", "haiku"], "calM-sonnet-15292-0": [true, 1.1501292665768463, "QA", "sonnet"], "calM-sonnet-15292-1": [true, 1.81381767898745, "QA", "sonnet"], "calM-haiku-15294-0": [true, 2.064057296391858, "QA", "haiku"], "calM-haiku-15294-1": [true, 1.6628411096527902, "QA", "haiku"], "calM-sonnet-15294-0": [true, 1.5055655406902866, "QA", "sonnet"], "calM-sonnet-15294-1": [true, 1.0392830666895112, "QA", "sonnet"], "calM-haiku-15311-0": [false, 2.044860227072679, "QA", "haiku"], "calM-haiku-15311-1": [true, 0.7802384867180621, "QA", "haiku"], "calM-sonnet-15311-0": [true, 1.6929236665577485, "QA", "sonnet"], "calM-sonnet-15311-1": [true, 1.7071039376251496, "QA", "sonnet"], "calM-haiku-15312-0": [true, 0.7972210280129599, "QA", "haiku"], "calM-haiku-15312-1": [true, 0.997407235642593, "QA", "haiku"], "calM-sonnet-15312-0": [true, 0.6597689419977112, "QA", "sonnet"], "calM-sonnet-15312-1": [true, 2.0043401709516226, "QA", "sonnet"], "calM-haiku-15331-0": [true, -0.1766694964020534, "QA", "haiku"], "calM-haiku-15331-1": [true, -0.25778716566999155, "QA", "haiku"], "calM-sonnet-15331-0": [true, 0.1266739239569559, "QA", "sonnet"], "calM-sonnet-15331-1": [true, 0.36592902529581395, "QA", "sonnet"], "calM-haiku-15335-0": [true, 1.0394603548191137, "QA", "haiku"], "calM-haiku-15335-1": [true, 0.7313329761656604, "QA", "haiku"], "calM-sonnet-15335-0": [true, 1.5392668680660704, "QA", "sonnet"], "calM-sonnet-15335-1": [true, 1.1902841990303894, "QA", "sonnet"], "calM-haiku-15336-0": [true, 1.0800666473529217, "QA", "haiku"], "calM-haiku-15336-1": [true, 0.8797194256686576, "QA", "haiku"], "calM-sonnet-15336-0": [true, 1.7781914061782977, "QA", "sonnet"], "calM-sonnet-15336-1": [true, -0.05215791465034933, "QA", "sonnet"], "calM-haiku-15341-0": [true, 1.235328966539138, "QA", "haiku"], "calM-haiku-15341-1": [true, 1.3285341495732261, "QA", "haiku"], "calM-sonnet-15341-0": [true, 0.90076916521428, "QA", "sonnet"], "calM-sonnet-15341-1": [true, 0.9348346583516167, "QA", "sonnet"], "calM-haiku-15352-0": [true, 2.3281412893503965, "QA", "haiku"], "calM-haiku-15352-1": [true, 2.201374586185056, "QA", "haiku"], "calM-sonnet-15352-0": [true, 1.4473603939649058, "QA", "sonnet"], "calM-sonnet-15352-1": [true, 1.6451468948137768, "QA", "sonnet"], "calM-haiku-15353-0": [true, 1.2607218979845842, "QA", "haiku"], "calM-haiku-15353-1": [true, 0.9642077146354782, "QA", "haiku"], "calM-sonnet-15353-0": [true, 1.836174437083196, "QA", "sonnet"], "calM-sonnet-15353-1": [false, 2.632608305797004, "QA", "sonnet"], "calM-haiku-15356-0": [false, 2.1621459462672563, "QA", "haiku"], "calM-haiku-15356-1": [true, 2.951796927542908, "QA", "haiku"], "calM-sonnet-15356-0": [true, 1.0936361661206868, "QA", "sonnet"], "calM-sonnet-15356-1": [true, 2.8928823686937317, "QA", "sonnet"], "calM-haiku-15358-0": [true, 1.126132383878573, "QA", "haiku"], "calM-haiku-15358-1": [true, 1.1924040751838096, "QA", "haiku"], "calM-sonnet-15358-0": [true, 1.074335250523351, "QA", "sonnet"], "calM-sonnet-15358-1": [true, 1.4154343022592275, "QA", "sonnet"], "calM-haiku-15370-0": [true, 2.693633878398527, "QA", "haiku"], "calM-haiku-15370-1": [true, -0.17010899067536855, "QA", "haiku"], "calM-sonnet-15370-0": [true, 0.9542395561158619, "QA", "sonnet"], "calM-sonnet-15370-1": [true, 0.9278447214175809, "QA", "sonnet"], "calM-haiku-15375-0": [true, 1.1928025997813445, "QA", "haiku"], "calM-haiku-15375-1": [true, 0.8094971887251934, "QA", "haiku"], "calM-sonnet-15375-0": [true, 0.6940981408624065, "QA", "sonnet"], "calM-sonnet-15375-1": [true, 1.8449747262544376, "QA", "sonnet"], "calM-haiku-15393-0": [true, 1.5516101890071043, "QA", "haiku"], "calM-haiku-15393-1": [true, 1.8515804899863866, "QA", "haiku"], "calM-sonnet-15393-0": [true, 2.3567510292424028, "QA", "sonnet"], "calM-sonnet-15393-1": [true, 2.13626495842612, "QA", "sonnet"], "calM-haiku-15399-0": [true, 2.567436235177267, "QA", "haiku"], "calM-haiku-15399-1": [true, 2.637670597049134, "QA", "haiku"], "calM-sonnet-15399-0": [true, 3.764718342282431, "QA", "sonnet"], "calM-sonnet-15399-1": [true, 2.2327241671290796, "QA", "sonnet"], "calM-haiku-15404-0": [true, 1.0899530346990252, "QA", "haiku"], "calM-haiku-15404-1": [false, 1.6377145756694067, "QA", "haiku"], "calM-sonnet-15404-0": [true, 1.3408908145940024, "QA", "sonnet"], "calM-sonnet-15404-1": [true, 1.3408908145940024, "QA", "sonnet"], "calM-haiku-15410-0": [false, 1.2677565823599741, "QA", "haiku"], "calM-haiku-15410-1": [false, 3.342190074595458, "QA", "haiku"], "calM-sonnet-15410-0": [true, 1.5389077935765239, "QA", "sonnet"], "calM-sonnet-15410-1": [true, 1.7817428393488135, "QA", "sonnet"], "calM-haiku-15411-0": [true, 1.7235464767371536, "QA", "haiku"], "calM-haiku-15411-1": [false, 2.1614189574021867, "QA", "haiku"], "calM-sonnet-15411-0": [true, 2.059275875751699, "QA", "sonnet"], "calM-sonnet-15411-1": [true, 1.7835438798807235, "QA", "sonnet"], "calM-haiku-15432-0": [true, 3.3836283760506727, "QA", "haiku"], "calM-haiku-15432-1": [false, 1.3362659644362744, "QA", "haiku"], "calM-sonnet-15432-0": [true, 2.738502867353392, "QA", "sonnet"], "calM-sonnet-15432-1": [true, 1.9721815931447852, "QA", "sonnet"], "calM-haiku-15436-0": [true, 2.146902947396221, "QA", "haiku"], "calM-haiku-15436-1": [true, 1.6284981708920352, "QA", "haiku"], "calM-sonnet-15436-0": [true, 1.9806749336845733, "QA", "sonnet"], "calM-sonnet-15436-1": [true, 1.217151908330901, "QA", "sonnet"], "calM-haiku-15479-0": [false, 3.212795454354656, "QA", "haiku"], "calM-haiku-15479-1": [false, 2.9785875020172314, "QA", "haiku"], "calM-sonnet-15479-0": [false, 2.8083371856901147, "QA", "sonnet"], "calM-sonnet-15479-1": [false, 2.998775463875451, "QA", "sonnet"], "calM-haiku-15492-0": [false, 1.849143178470892, "QA", "haiku"], "calM-haiku-15492-1": [true, 2.0694519572332277, "QA", "haiku"], "calM-sonnet-15492-0": [true, 2.0618701487873037, "QA", "sonnet"], "calM-sonnet-15492-1": [true, 2.5897079795430944, "QA", "sonnet"], "calM-haiku-15502-0": [false, 2.038089664471711, "QA", "haiku"], "calM-haiku-15502-1": [true, 1.6301339763540241, "QA", "haiku"], "calM-sonnet-15502-0": [true, 1.2362514257237476, "QA", "sonnet"], "calM-sonnet-15502-1": [false, 1.9415544969489726, "QA", "sonnet"], "calM-haiku-15504-0": [false, 1.4257164131518794, "QA", "haiku"], "calM-haiku-15504-1": [true, 1.0843481894754232, "QA", "haiku"], "calM-sonnet-15504-0": [false, 0.8058622276382376, "QA", "sonnet"], "calM-sonnet-15504-1": [false, 1.0689918742170863, "QA", "sonnet"], "calM-haiku-15507-0": [true, 0.547014770166668, "QA", "haiku"], "calM-haiku-15507-1": [false, 1.3242185989456587, "QA", "haiku"], "calM-sonnet-15507-0": [true, 1.2280139882309196, "QA", "sonnet"], "calM-sonnet-15507-1": [true, 1.3379900594417464, "QA", "sonnet"], "calM-haiku-15511-0": [true, 1.9750823709451668, "QA", "haiku"], "calM-haiku-15511-1": [true, 0.7870596829450128, "QA", "haiku"], "calM-sonnet-15511-0": [true, 1.8860351075186614, "QA", "sonnet"], "calM-sonnet-15511-1": [true, 2.106983765267282, "QA", "sonnet"], "calM-haiku-15516-0": [false, 0.8039161527010364, "QA", "haiku"], "calM-haiku-15516-1": [false, 0.014786712924967027, "QA", "haiku"], "calM-sonnet-15516-0": [false, 2.6247000842521038, "QA", "sonnet"], "calM-sonnet-15516-1": [true, 1.392653218628487, "QA", "sonnet"], "calM-haiku-15524-0": [true, 0.9598924357854166, "QA", "haiku"], "calM-haiku-15524-1": [true, 1.949983879032695, "QA", "haiku"], "calM-sonnet-15524-0": [true, 1.2378824117224148, "QA", "sonnet"], "calM-sonnet-15524-1": [true, 1.431000946169354, "QA", "sonnet"], "calM-haiku-15530-0": [true, 0.883732931732228, "QA", "haiku"], "calM-haiku-15530-1": [true, 1.0707180351905525, "QA", "haiku"], "calM-sonnet-15530-0": [true, 2.2218508788347124, "QA", "sonnet"], "calM-sonnet-15530-1": [true, 1.298531139336908, "QA", "sonnet"], "calM-haiku-15541-0": [true, 0.8345856009913031, "QA", "haiku"], "calM-haiku-15541-1": [true, 2.065009146489169, "QA", "haiku"], "calM-sonnet-15541-0": [true, 1.7581322922996407, "QA", "sonnet"], "calM-sonnet-15541-1": [true, 1.252270402899685, "QA", "sonnet"], "calM-haiku-15557-0": [false, 3.0873520872815177, "QA", "haiku"], "calM-haiku-15557-1": [false, 1.6618378204078408, "QA", "haiku"], "calM-sonnet-15557-0": [true, 1.6262622989217577, "QA", "sonnet"], "calM-sonnet-15557-1": [true, 1.7267317399699245, "QA", "sonnet"], "calM-haiku-15574-0": [true, 1.1142426267598164, "QA", "haiku"], "calM-haiku-15574-1": [true, 0.15665486943160717, "QA", "haiku"], "calM-sonnet-15574-0": [true, 1.5441504459197781, "QA", "sonnet"], "calM-sonnet-15574-1": [false, 1.0392844044421163, "QA", "sonnet"], "calM-haiku-15585-0": [true, 2.17866995570851, "QA", "haiku"], "calM-haiku-15585-1": [true, 1.0093898568603543, "QA", "haiku"], "calM-sonnet-15585-0": [true, 1.5556266062298802, "QA", "sonnet"], "calM-sonnet-15585-1": [true, 1.0327078097498088, "QA", "sonnet"], "calM-haiku-15605-0": [true, 2.0531044737339994, "Summary", "haiku"], "calM-haiku-15605-1": [true, 1.5644634844954655, "Summary", "haiku"], "calM-sonnet-15605-0": [true, 0.7861283712720418, "Summary", "sonnet"], "calM-sonnet-15605-1": [true, 0.6158604864514274, "Summary", "sonnet"], "calM-haiku-15610-0": [false, 1.7612695865365484, "Summary", "haiku"], "calM-haiku-15610-1": [true, 2.0820407872240327, "Summary", "haiku"], "calM-sonnet-15610-0": [true, 1.900407121008764, "Summary", "sonnet"], "calM-sonnet-15610-1": [true, 1.900407121008764, "Summary", "sonnet"], "calM-haiku-15618-0": [true, 0.1026990833722241, "Summary", "haiku"], "calM-haiku-15618-1": [true, 0.5689744887633283, "Summary", "haiku"], "calM-sonnet-15618-0": [true, 1.993666196138791, "Summary", "sonnet"], "calM-sonnet-15618-1": [true, 1.9761755647012558, "Summary", "sonnet"], "calM-haiku-15621-0": [false, 2.0874185063966078, "Summary", "haiku"], "calM-haiku-15621-1": [false, 1.3242590209608713, "Summary", "haiku"], "calM-sonnet-15621-0": [true, 3.0708535221612046, "Summary", "sonnet"], "calM-sonnet-15621-1": [true, 2.7301645369865195, "Summary", "sonnet"], "calM-haiku-15625-0": [true, 0.49847431926174746, "Summary", "haiku"], "calM-haiku-15625-1": [true, 0.90543729015309, "Summary", "haiku"], "calM-sonnet-15625-0": [true, 1.184649882409682, "Summary", "sonnet"], "calM-sonnet-15625-1": [true, 1.098352132015297, "Summary", "sonnet"], "calM-haiku-15626-0": [true, 2.3782309508588906, "Summary", "haiku"], "calM-haiku-15626-1": [true, 0.7687664332015361, "Summary", "haiku"], "calM-sonnet-15626-0": [true, -0.17418341731978607, "Summary", "sonnet"], "calM-sonnet-15626-1": [true, 1.0615641019416528, "Summary", "sonnet"], "calM-haiku-15643-0": [true, 0.26266513028402005, "Summary", "haiku"], "calM-haiku-15643-1": [true, 0.1174184275172589, "Summary", "haiku"], "calM-sonnet-15643-0": [true, 0.14250665859836464, "Summary", "sonnet"], "calM-sonnet-15643-1": [true, 0.32868759922175417, "Summary", "sonnet"], "calM-haiku-15663-0": [true, 2.085493698022228, "Summary", "haiku"], "calM-haiku-15663-1": [true, 1.4387828776758433, "Summary", "haiku"], "calM-sonnet-15663-0": [true, 1.463589929052939, "Summary", "sonnet"], "calM-sonnet-15663-1": [true, 1.3451905721947794, "Summary", "sonnet"], "calM-haiku-15680-0": [false, 3.240199495511444, "Summary", "haiku"], "calM-haiku-15680-1": [false, 2.2798803992574843, "Summary", "haiku"], "calM-sonnet-15680-0": [true, 2.7961787755754934, "Summary", "sonnet"], "calM-sonnet-15680-1": [false, 3.2667591123171364, "Summary", "sonnet"], "calM-haiku-15689-0": [false, 0.7718663704030342, "Summary", "haiku"], "calM-haiku-15689-1": [true, 1.9512881015702916, "Summary", "haiku"], "calM-sonnet-15689-0": [false, 2.153707773518153, "Summary", "sonnet"], "calM-sonnet-15689-1": [true, 2.6967903580587618, "Summary", "sonnet"], "calM-haiku-15692-0": [true, -0.0005519924679659449, "Summary", "haiku"], "calM-haiku-15692-1": [true, 0.07144648615804494, "Summary", "haiku"], "calM-sonnet-15692-0": [true, -0.2503404457765489, "Summary", "sonnet"], "calM-sonnet-15692-1": [true, 0.7126183113874871, "Summary", "sonnet"], "calM-haiku-15704-0": [true, 0.7066260822823804, "Summary", "haiku"], "calM-haiku-15704-1": [true, 0.4662848328829216, "Summary", "haiku"], "calM-sonnet-15704-0": [true, 1.1034173832543783, "Summary", "sonnet"], "calM-sonnet-15704-1": [true, 1.2497990629497722, "Summary", "sonnet"], "calM-haiku-15748-0": [true, 0.1300584514506667, "Summary", "haiku"], "calM-haiku-15748-1": [false, 0.4396234997870993, "Summary", "haiku"], "calM-sonnet-15748-0": [true, 0.9323506019032649, "Summary", "sonnet"], "calM-sonnet-15748-1": [true, 1.2360039707393846, "Summary", "sonnet"], "calM-haiku-15756-0": [true, 1.8399021653910514, "Summary", "haiku"], "calM-haiku-15756-1": [true, 2.234810112083701, "Summary", "haiku"], "calM-sonnet-15756-0": [true, 0.03598613035686518, "Summary", "sonnet"], "calM-sonnet-15756-1": [true, 2.1094639950803677, "Summary", "sonnet"], "calM-haiku-15760-0": [true, 0.26505039507683215, "Summary", "haiku"], "calM-haiku-15760-1": [true, 0.9538129175786935, "Summary", "haiku"], "calM-sonnet-15760-0": [true, 0.4320349419004623, "Summary", "sonnet"], "calM-sonnet-15760-1": [true, 0.266647739099665, "Summary", "sonnet"], "calM-haiku-15765-0": [true, 0.5072797554772517, "Summary", "haiku"], "calM-haiku-15765-1": [true, 0.12207325571165734, "Summary", "haiku"], "calM-sonnet-15765-0": [true, 0.7000315680506295, "Summary", "sonnet"], "calM-sonnet-15765-1": [true, 0.6781682787537107, "Summary", "sonnet"], "calM-haiku-15766-0": [true, -0.06977585542497147, "Summary", "haiku"], "calM-haiku-15766-1": [true, 0.43860322893774245, "Summary", "haiku"], "calM-sonnet-15766-0": [true, 0.4929794497437938, "Summary", "sonnet"], "calM-sonnet-15766-1": [true, 0.501569161000837, "Summary", "sonnet"], "calM-haiku-15780-0": [true, 0.27706898983217515, "Summary", "haiku"], "calM-haiku-15780-1": [true, 3.4729277372586758, "Summary", "haiku"], "calM-sonnet-15780-0": [true, 0.9056508811991802, "Summary", "sonnet"], "calM-sonnet-15780-1": [true, 0.46587182100118546, "Summary", "sonnet"], "calM-haiku-15784-0": [true, 0.7659719216365091, "Summary", "haiku"], "calM-haiku-15784-1": [true, 0.3536193730610563, "Summary", "haiku"], "calM-sonnet-15784-0": [false, 0.2733399618012761, "Summary", "sonnet"], "calM-sonnet-15784-1": [false, 1.0629370962772404, "Summary", "sonnet"], "calM-haiku-15793-0": [false, 0.599200603488961, "Summary", "haiku"], "calM-haiku-15793-1": [true, 0.2293642553567828, "Summary", "haiku"], "calM-sonnet-15793-0": [true, 1.5663327562136133, "Summary", "sonnet"], "calM-sonnet-15793-1": [true, 0.9259348627287103, "Summary", "sonnet"], "calM-haiku-15822-0": [false, 1.8088081011975323, "Summary", "haiku"], "calM-haiku-15822-1": [true, 1.3308638167099407, "Summary", "haiku"], "calM-sonnet-15822-0": [false, 1.5203782720694423, "Summary", "sonnet"], "calM-sonnet-15822-1": [false, 0.9910043855461909, "Summary", "sonnet"], "calM-haiku-15824-0": [false, 2.445629551961823, "Summary", "haiku"], "calM-haiku-15824-1": [false, 2.459778570278894, "Summary", "haiku"], "calM-sonnet-15824-0": [true, 3.31068088136944, "Summary", "sonnet"], "calM-sonnet-15824-1": [true, 1.9558306473424714, "Summary", "sonnet"], "calM-haiku-15829-0": [true, 0.590762719278972, "Summary", "haiku"], "calM-haiku-15829-1": [true, 0.3506401628066435, "Summary", "haiku"], "calM-sonnet-15829-0": [true, 0.38338978408861646, "Summary", "sonnet"], "calM-sonnet-15829-1": [true, 0.40047365665551116, "Summary", "sonnet"]}, "devM": {"devM-haiku-11354-0": [true, 0.5576852885629315, "Summary", "haiku"], "devM-haiku-11354-1": [true, 0.2738991326458888, "Summary", "haiku"], "devM-sonnet-11354-0": [true, 1.7089671911942608, "Summary", "sonnet"], "devM-sonnet-11354-1": [true, 0.23893037579301019, "Summary", "sonnet"], "devM-haiku-11386-0": [true, 0.1559799066032668, "Summary", "haiku"], "devM-haiku-11386-1": [true, 0.7563260856517865, "Summary", "haiku"], "devM-sonnet-11386-0": [true, 0.6867258967854935, "Summary", "sonnet"], "devM-sonnet-11386-1": [true, 0.48967207160965054, "Summary", "sonnet"], "devM-haiku-11427-0": [true, 0.7150145086067404, "Summary", "haiku"], "devM-haiku-11427-1": [true, 0.7806339094203472, "Summary", "haiku"], "devM-sonnet-11427-0": [true, 1.0886265804820792, "Summary", "sonnet"], "devM-sonnet-11427-1": [true, 1.3955433791140122, "Summary", "sonnet"], "devM-haiku-11428-0": [false, 1.4804554735569986, "Summary", "haiku"], "devM-haiku-11428-1": [true, 0.6422115087001314, "Summary", "haiku"], "devM-sonnet-11428-0": [true, 0.5871433208132069, "Summary", "sonnet"], "devM-sonnet-11428-1": [true, 1.3203379949592584, "Summary", "sonnet"], "devM-haiku-11429-0": [true, 0.4092817289664919, "Summary", "haiku"], "devM-haiku-11429-1": [true, 2.3707271793728593, "Summary", "haiku"], "devM-sonnet-11429-0": [true, 0.6615545311868856, "Summary", "sonnet"], "devM-sonnet-11429-1": [true, 0.4420052589427472, "Summary", "sonnet"], "devM-haiku-11430-0": [false, 0.12628857697668783, "Summary", "haiku"], "devM-haiku-11430-1": [false, 1.7974460233866543, "Summary", "haiku"], "devM-sonnet-11430-0": [true, 0.22089883820973372, "Summary", "sonnet"], "devM-sonnet-11430-1": [true, 0.2770315623581022, "Summary", "sonnet"], "devM-haiku-11772-0": [true, 2.768185423076199, "Summary", "haiku"], "devM-haiku-11772-1": [true, 0.2741282863131696, "Summary", "haiku"], "devM-sonnet-11772-0": [true, 0.5146094229406283, "Summary", "sonnet"], "devM-sonnet-11772-1": [true, 0.7552799913498732, "Summary", "sonnet"], "devM-haiku-11773-0": [true, 0.3353900788303863, "Summary", "haiku"], "devM-haiku-11773-1": [true, 0.04960890707305496, "Summary", "haiku"], "devM-sonnet-11773-0": [true, -0.24124419924883816, "Summary", "sonnet"], "devM-sonnet-11773-1": [true, -0.24124419924883816, "Summary", "sonnet"], "devM-haiku-11789-0": [true, 1.7225612729682438, "Summary", "haiku"], "devM-haiku-11789-1": [true, 0.47045413395511065, "Summary", "haiku"], "devM-sonnet-11789-0": [false, 0.9441515150484355, "Summary", "sonnet"], "devM-sonnet-11789-1": [false, 1.2294467313313455, "Summary", "sonnet"], "devM-haiku-11797-0": [false, 3.5413031758930043, "Summary", "haiku"], "devM-haiku-11797-1": [true, 2.1900341467458055, "Summary", "haiku"], "devM-sonnet-11797-0": [true, 1.1970939751396004, "Summary", "sonnet"], "devM-sonnet-11797-1": [true, 1.052397104316114, "Summary", "sonnet"], "devM-haiku-11809-0": [true, 1.194471890347226, "Summary", "haiku"], "devM-haiku-11809-1": [true, 0.7003227633809695, "Summary", "haiku"], "devM-sonnet-11809-0": [true, 0.9291072408076733, "Summary", "sonnet"], "devM-sonnet-11809-1": [true, 0.8202070666369434, "Summary", "sonnet"], "devM-haiku-11819-0": [true, 1.0879300491886605, "Summary", "haiku"], "devM-haiku-11819-1": [true, 0.3632838774865643, "Summary", "haiku"], "devM-sonnet-11819-0": [true, 1.0618331965759653, "Summary", "sonnet"], "devM-sonnet-11819-1": [true, 0.48661069874495966, "Summary", "sonnet"], "devM-haiku-11836-0": [true, 1.4415381493846429, "Summary", "haiku"], "devM-haiku-11836-1": [false, 2.1117762895749954, "Summary", "haiku"], "devM-sonnet-11836-0": [true, 1.4949930370041207, "Summary", "sonnet"], "devM-sonnet-11836-1": [true, 1.464249461224497, "Summary", "sonnet"], "devM-haiku-11856-0": [true, 0.2958811159130007, "Summary", "haiku"], "devM-haiku-11856-1": [true, 1.05901316632699, "Summary", "haiku"], "devM-sonnet-11856-0": [true, 1.0936481778084621, "Summary", "sonnet"], "devM-sonnet-11856-1": [true, 1.0594804326746727, "Summary", "sonnet"], "devM-haiku-11861-0": [false, 1.697566161534155, "Summary", "haiku"], "devM-haiku-11861-1": [false, 1.7752571058264623, "Summary", "haiku"], "devM-sonnet-11861-0": [true, 1.0963948981703997, "Summary", "sonnet"], "devM-sonnet-11861-1": [true, 1.6273388507953217, "Summary", "sonnet"], "devM-haiku-11879-0": [true, 1.0379349184617106, "Summary", "haiku"], "devM-haiku-11879-1": [false, 1.3636503219109746, "Summary", "haiku"], "devM-sonnet-11879-0": [true, 0.2983523492654037, "Summary", "sonnet"], "devM-sonnet-11879-1": [true, 0.5237021185349238, "Summary", "sonnet"], "devM-haiku-11889-0": [true, -0.16818433155288456, "Summary", "haiku"], "devM-haiku-11889-1": [true, -0.017510895405446314, "Summary", "haiku"], "devM-sonnet-11889-0": [true, 1.0966494715483226, "Summary", "sonnet"], "devM-sonnet-11889-1": [true, 0.15816485248410253, "Summary", "sonnet"], "devM-haiku-11923-0": [true, 0.4836604798205573, "Summary", "haiku"], "devM-haiku-11923-1": [true, 0.3736786292470978, "Summary", "haiku"], "devM-sonnet-11923-0": [true, 0.15291792420135986, "Summary", "sonnet"], "devM-sonnet-11923-1": [true, 0.14803681775367564, "Summary", "sonnet"], "devM-haiku-11929-0": [true, 2.072653886958822, "Summary", "haiku"], "devM-haiku-11929-1": [true, 2.799817831150876, "Summary", "haiku"], "devM-sonnet-11929-0": [true, 0.8465573641456314, "Summary", "sonnet"], "devM-sonnet-11929-1": [true, 1.523119143162198, "Summary", "sonnet"], "devM-haiku-11930-0": [true, 0.7488328758413841, "Summary", "haiku"], "devM-haiku-11930-1": [true, 1.2688615652469286, "Summary", "haiku"], "devM-sonnet-11930-0": [true, 0.4970280115637487, "Summary", "sonnet"], "devM-sonnet-11930-1": [true, 0.10834935155660014, "Summary", "sonnet"], "devM-haiku-11942-0": [true, 2.470320745372987, "Summary", "haiku"], "devM-haiku-11942-1": [true, 1.6172054119095336, "Summary", "haiku"], "devM-sonnet-11942-0": [true, 2.065867646533863, "Summary", "sonnet"], "devM-sonnet-11942-1": [true, 1.3861994424426434, "Summary", "sonnet"], "devM-haiku-11972-0": [true, -0.014586388655145388, "Summary", "haiku"], "devM-haiku-11972-1": [true, -0.29773140906390977, "Summary", "haiku"], "devM-sonnet-11972-0": [true, -0.3593928507362469, "Summary", "sonnet"], "devM-sonnet-11972-1": [true, -0.29773140906390977, "Summary", "sonnet"], "devM-haiku-11984-0": [false, 2.6510225395986646, "Summary", "haiku"], "devM-haiku-11984-1": [true, 1.7684580856396337, "Summary", "haiku"], "devM-sonnet-11984-0": [true, 1.846086682968159, "Summary", "sonnet"], "devM-sonnet-11984-1": [true, 1.8720030963354057, "Summary", "sonnet"], "devM-haiku-11987-0": [true, 1.7459036199840614, "Summary", "haiku"], "devM-haiku-11987-1": [true, 2.407930943110787, "Summary", "haiku"], "devM-sonnet-11987-0": [true, 2.057135606873554, "Summary", "sonnet"], "devM-sonnet-11987-1": [true, 2.5094265495737433, "Summary", "sonnet"], "devM-haiku-12014-0": [true, 1.3046968299363515, "Summary", "haiku"], "devM-haiku-12014-1": [true, 1.9158191038297476, "Summary", "haiku"], "devM-sonnet-12014-0": [true, 1.4309169362236471, "Summary", "sonnet"], "devM-sonnet-12014-1": [true, 1.4309169362236471, "Summary", "sonnet"], "devM-haiku-12022-0": [true, 0.3674785943450508, "Summary", "haiku"], "devM-haiku-12022-1": [true, -0.04272985479264611, "Summary", "haiku"], "devM-sonnet-12022-0": [true, 0.4200071694994327, "Summary", "sonnet"], "devM-sonnet-12022-1": [true, 0.3006803482895211, "Summary", "sonnet"], "devM-haiku-12035-0": [true, 0.6906120942685773, "Summary", "haiku"], "devM-haiku-12035-1": [true, 1.6926521258427394, "Summary", "haiku"], "devM-sonnet-12035-0": [true, 0.03431005945436955, "Summary", "sonnet"], "devM-sonnet-12035-1": [true, 0.2651206612021346, "Summary", "sonnet"], "devM-haiku-12037-0": [false, 2.3670359328052375, "Summary", "haiku"], "devM-haiku-12037-1": [false, 1.7312548058931454, "Summary", "haiku"], "devM-sonnet-12037-0": [true, 3.266235436905183, "Summary", "sonnet"], "devM-sonnet-12037-1": [true, 2.1170811745272893, "Summary", "sonnet"], "devM-haiku-12052-0": [false, 1.195564758648754, "Summary", "haiku"], "devM-haiku-12052-1": [true, 1.2785957508708208, "Summary", "haiku"], "devM-sonnet-12052-0": [true, 1.354035487052915, "Summary", "sonnet"], "devM-sonnet-12052-1": [true, 2.333767107769182, "Summary", "sonnet"], "devM-haiku-12053-0": [true, 0.7689348691400132, "Summary", "haiku"], "devM-haiku-12053-1": [true, 0.7139686407860821, "Summary", "haiku"], "devM-sonnet-12053-0": [true, 1.3059687288430875, "Summary", "sonnet"], "devM-sonnet-12053-1": [true, 0.08042730307203588, "Summary", "sonnet"], "devM-haiku-12054-0": [false, 1.4295726126689399, "Summary", "haiku"], "devM-haiku-12054-1": [false, 1.8471254824452035, "Summary", "haiku"], "devM-sonnet-12054-0": [false, 0.586779958079023, "Summary", "sonnet"], "devM-sonnet-12054-1": [false, 0.5003479410163347, "Summary", "sonnet"], "devM-haiku-12061-0": [false, 2.15178670670099, "Summary", "haiku"], "devM-haiku-12061-1": [true, 1.317176053010148, "Summary", "haiku"], "devM-sonnet-12061-0": [true, 2.1258639989507415, "Summary", "sonnet"], "devM-sonnet-12061-1": [false, 1.8576143464459314, "Summary", "sonnet"], "devM-haiku-12070-0": [true, 0.9911928266969723, "Summary", "haiku"], "devM-haiku-12070-1": [true, 0.3049695358338651, "Summary", "haiku"], "devM-sonnet-12070-0": [true, 0.17322392715847817, "Summary", "sonnet"], "devM-sonnet-12070-1": [true, 0.17322392715847817, "Summary", "sonnet"], "devM-haiku-12089-0": [true, 1.831752504618388, "Summary", "haiku"], "devM-haiku-12089-1": [true, 1.6054376623380393, "Summary", "haiku"], "devM-sonnet-12089-0": [true, 2.2222069213355575, "Summary", "sonnet"], "devM-sonnet-12089-1": [true, 1.079125885990701, "Summary", "sonnet"], "devM-haiku-12165-0": [true, 0.8191184617036977, "QA", "haiku"], "devM-haiku-12165-1": [true, 1.4801455108125667, "QA", "haiku"], "devM-sonnet-12165-0": [true, 2.4224173483251814, "QA", "sonnet"], "devM-sonnet-12165-1": [true, 2.6501123680385597, "QA", "sonnet"], "devM-haiku-12170-0": [true, 2.2738594193464574, "QA", "haiku"], "devM-haiku-12170-1": [true, 2.3579726795412896, "QA", "haiku"], "devM-sonnet-12170-0": [false, 2.7567388183370722, "QA", "sonnet"], "devM-sonnet-12170-1": [true, 2.806455935275294, "QA", "sonnet"], "devM-haiku-12179-0": [false, 1.1749989056663002, "QA", "haiku"], "devM-haiku-12179-1": [true, 1.5853049180897285, "QA", "haiku"], "devM-sonnet-12179-0": [true, 1.0844150819279699, "QA", "sonnet"], "devM-sonnet-12179-1": [true, 1.6214289401185988, "QA", "sonnet"], "devM-haiku-12205-0": [true, 2.0591572177158666, "QA", "haiku"], "devM-haiku-12205-1": [true, 2.2898103149705094, "QA", "haiku"], "devM-sonnet-12205-0": [true, 2.310677128350195, "QA", "sonnet"], "devM-sonnet-12205-1": [true, 3.012181543725956, "QA", "sonnet"], "devM-haiku-12245-0": [false, 1.1278008667361403, "QA", "haiku"], "devM-haiku-12245-1": [true, 1.7883477916725892, "QA", "haiku"], "devM-sonnet-12245-0": [false, 1.0121440237164356, "QA", "sonnet"], "devM-sonnet-12245-1": [true, 0.883513066989089, "QA", "sonnet"], "devM-haiku-12247-0": [true, 1.2036163712296744, "QA", "haiku"], "devM-haiku-12247-1": [true, 1.5959835033259286, "QA", "haiku"], "devM-sonnet-12247-0": [true, 0.965716033499721, "QA", "sonnet"], "devM-sonnet-12247-1": [true, 0.798264850551577, "QA", "sonnet"], "devM-haiku-12248-0": [true, 1.07967581347348, "QA", "haiku"], "devM-haiku-12248-1": [true, 2.0297376336185127, "QA", "haiku"], "devM-sonnet-12248-0": [true, 1.036941437085617, "QA", "sonnet"], "devM-sonnet-12248-1": [true, 1.1891754611108332, "QA", "sonnet"], "devM-haiku-12254-0": [false, 1.2237584054497272, "QA", "haiku"], "devM-haiku-12254-1": [true, 1.333266176347788, "QA", "haiku"], "devM-sonnet-12254-0": [true, 1.8077217000832062, "QA", "sonnet"], "devM-sonnet-12254-1": [true, 1.1355440244106614, "QA", "sonnet"], "devM-haiku-12257-0": [false, 2.4593029416180423, "QA", "haiku"], "devM-haiku-12257-1": [true, 2.4791136179648996, "QA", "haiku"], "devM-sonnet-12257-0": [true, 2.4503008487024323, "QA", "sonnet"], "devM-sonnet-12257-1": [true, 1.57574565719204, "QA", "sonnet"], "devM-haiku-12260-0": [true, 0.8374121274942374, "QA", "haiku"], "devM-haiku-12260-1": [false, 3.152714529713278, "QA", "haiku"], "devM-sonnet-12260-0": [false, 1.1644836002949956, "QA", "sonnet"], "devM-sonnet-12260-1": [false, 1.2560941601077125, "QA", "sonnet"], "devM-haiku-12279-0": [true, 1.3399175486819743, "QA", "haiku"], "devM-haiku-12279-1": [true, 0.6467173891858157, "QA", "haiku"], "devM-sonnet-12279-0": [true, 1.093181292554568, "QA", "sonnet"], "devM-sonnet-12279-1": [true, 1.2612743958466932, "QA", "sonnet"], "devM-haiku-12289-0": [true, 1.2989572234439732, "QA", "haiku"], "devM-haiku-12289-1": [true, 1.826870492245684, "QA", "haiku"], "devM-sonnet-12289-0": [true, 0.8743496089729024, "QA", "sonnet"], "devM-sonnet-12289-1": [true, 2.092060605219968, "QA", "sonnet"], "devM-haiku-12292-0": [true, 2.4797632432942955, "QA", "haiku"], "devM-haiku-12292-1": [true, 2.372410121204471, "QA", "haiku"], "devM-sonnet-12292-0": [true, 2.601037037517739, "QA", "sonnet"], "devM-sonnet-12292-1": [true, 2.6356115415221684, "QA", "sonnet"], "devM-haiku-12303-0": [false, 3.1550888207077428, "QA", "haiku"], "devM-haiku-12303-1": [false, 2.4088156250537005, "QA", "haiku"], "devM-sonnet-12303-0": [false, 1.888236065058853, "QA", "sonnet"], "devM-sonnet-12303-1": [false, 3.2754643437804014, "QA", "sonnet"], "devM-haiku-12306-0": [false, 1.5246189599001745, "QA", "haiku"], "devM-haiku-12306-1": [false, 0.9159316164426936, "QA", "haiku"], "devM-sonnet-12306-0": [true, 1.2557820265354631, "QA", "sonnet"], "devM-sonnet-12306-1": [true, 1.4893162311946346, "QA", "sonnet"], "devM-haiku-12313-0": [true, 2.938420308746532, "QA", "haiku"], "devM-haiku-12313-1": [true, 3.363131363791673, "QA", "haiku"], "devM-sonnet-12313-0": [true, 2.333222871284763, "QA", "sonnet"], "devM-sonnet-12313-1": [true, 2.2758698705244735, "QA", "sonnet"], "devM-haiku-12324-0": [true, 3.292085180376793, "QA", "haiku"], "devM-haiku-12324-1": [false, 3.587383799520426, "QA", "haiku"], "devM-sonnet-12324-0": [true, 3.574758349539376, "QA", "sonnet"], "devM-sonnet-12324-1": [true, 3.3998520351640256, "QA", "sonnet"], "devM-haiku-12340-0": [true, 0.603520561754368, "QA", "haiku"], "devM-haiku-12340-1": [true, 0.8513862013137234, "QA", "haiku"], "devM-sonnet-12340-0": [true, 0.7298393485517768, "QA", "sonnet"], "devM-sonnet-12340-1": [true, 0.7576975384016393, "QA", "sonnet"], "devM-haiku-12350-0": [true, 1.0956573638591949, "QA", "haiku"], "devM-haiku-12350-1": [true, 2.0555881884982963, "QA", "haiku"], "devM-sonnet-12350-0": [true, 0.5067810468134305, "QA", "sonnet"], "devM-sonnet-12350-1": [true, 0.4421554218015489, "QA", "sonnet"], "devM-haiku-12368-0": [true, 1.400043917951166, "QA", "haiku"], "devM-haiku-12368-1": [true, 1.2945920524695533, "QA", "haiku"], "devM-sonnet-12368-0": [true, 2.4971186614816308, "QA", "sonnet"], "devM-sonnet-12368-1": [true, 0.8606171800434532, "QA", "sonnet"], "devM-haiku-12369-0": [true, 1.7059557725117873, "QA", "haiku"], "devM-haiku-12369-1": [true, 1.202782547431826, "QA", "haiku"], "devM-sonnet-12369-0": [true, 1.3446553341332157, "QA", "sonnet"], "devM-sonnet-12369-1": [true, 0.7948713265971836, "QA", "sonnet"], "devM-haiku-12371-0": [true, 0.018972614507774918, "QA", "haiku"], "devM-haiku-12371-1": [true, 0.6677190952665712, "QA", "haiku"], "devM-sonnet-12371-0": [true, 1.0113148847714781, "QA", "sonnet"], "devM-sonnet-12371-1": [true, 1.104826524669121, "QA", "sonnet"], "devM-haiku-12375-0": [true, -0.05944556581680976, "QA", "haiku"], "devM-haiku-12375-1": [true, -0.1943828059388633, "QA", "haiku"], "devM-sonnet-12375-0": [false, 0.37348128026596333, "QA", "sonnet"], "devM-sonnet-12375-1": [true, 0.2909740087521009, "QA", "sonnet"], "devM-haiku-12381-0": [true, 1.6350528273495382, "QA", "haiku"], "devM-haiku-12381-1": [true, 0.5746617708606868, "QA", "haiku"], "devM-sonnet-12381-0": [true, 0.9162669639444931, "QA", "sonnet"], "devM-sonnet-12381-1": [true, 1.0915800371904845, "QA", "sonnet"], "devM-haiku-12382-0": [false, 2.3543757495791056, "QA", "haiku"], "devM-haiku-12382-1": [false, 2.423495898222363, "QA", "haiku"], "devM-sonnet-12382-0": [true, 1.8540237866598155, "QA", "sonnet"], "devM-sonnet-12382-1": [true, 2.3876373102871074, "QA", "sonnet"], "devM-haiku-12384-0": [true, 1.8331559560766548, "QA", "haiku"], "devM-haiku-12384-1": [true, 1.4090097315795003, "QA", "haiku"], "devM-sonnet-12384-0": [true, 1.1569516572679313, "QA", "sonnet"], "devM-sonnet-12384-1": [true, 1.2518396792359798, "QA", "sonnet"], "devM-haiku-12385-0": [true, 3.3892664513622694, "QA", "haiku"], "devM-haiku-12385-1": [true, 2.2968341698171093, "QA", "haiku"], "devM-sonnet-12385-0": [true, 2.9958040802278845, "QA", "sonnet"], "devM-sonnet-12385-1": [false, 2.2975426015198486, "QA", "sonnet"], "devM-haiku-12393-0": [true, 1.2686281319818717, "QA", "haiku"], "devM-haiku-12393-1": [true, 2.082756011568535, "QA", "haiku"], "devM-sonnet-12393-0": [true, 1.3322037626746521, "QA", "sonnet"], "devM-sonnet-12393-1": [true, 1.9268411340375278, "QA", "sonnet"], "devM-haiku-12415-0": [true, 1.9725540037884772, "QA", "haiku"], "devM-haiku-12415-1": [true, 2.029907004548767, "QA", "haiku"], "devM-sonnet-12415-0": [true, 1.4936256607755145, "QA", "sonnet"], "devM-sonnet-12415-1": [true, 1.5862739939488821, "QA", "sonnet"], "devM-haiku-12417-0": [true, 1.9585677449449017, "QA", "haiku"], "devM-haiku-12417-1": [true, 1.9931422489493316, "QA", "haiku"], "devM-sonnet-12417-0": [true, 1.2154271789118578, "QA", "sonnet"], "devM-sonnet-12417-1": [true, 1.2925079848982044, "QA", "sonnet"], "devM-haiku-12429-0": [false, 2.2152892608555907, "QA", "haiku"], "devM-haiku-12429-1": [false, 2.5910758727542422, "QA", "haiku"], "devM-sonnet-12429-0": [true, 2.310147220613099, "QA", "sonnet"], "devM-sonnet-12429-1": [true, 2.5131756798280804, "QA", "sonnet"], "devM-haiku-12430-0": [false, 2.3352031193541256, "QA", "haiku"], "devM-haiku-12430-1": [false, 1.8653307533003225, "QA", "haiku"], "devM-sonnet-12430-0": [true, 1.0723226455036112, "QA", "sonnet"], "devM-sonnet-12430-1": [true, 2.412168010101555, "QA", "sonnet"], "devM-haiku-12433-0": [true, 2.2050666318557024, "QA", "haiku"], "devM-haiku-12433-1": [false, 2.504989918009887, "QA", "haiku"], "devM-sonnet-12433-0": [true, 3.0023782618305974, "QA", "sonnet"], "devM-sonnet-12433-1": [true, 3.3877336147880595, "QA", "sonnet"], "devM-haiku-12443-0": [true, 1.4933596231891728, "QA", "haiku"], "devM-haiku-12443-1": [false, 2.886876826082605, "QA", "haiku"], "devM-sonnet-12443-0": [true, 2.0634082929048274, "QA", "sonnet"], "devM-sonnet-12443-1": [true, 1.7536023224562016, "QA", "sonnet"], "devM-haiku-12458-0": [true, 0.6456318240899607, "QA", "haiku"], "devM-haiku-12458-1": [true, 1.1926828531222875, "QA", "haiku"], "devM-sonnet-12458-0": [true, 0.9512798956169566, "QA", "sonnet"], "devM-sonnet-12458-1": [true, 0.9651096972187285, "QA", "sonnet"], "devM-haiku-13500-0": [true, -0.05390650079331813, "Summary", "haiku"], "devM-haiku-13500-1": [true, 1.3076254725469858, "Summary", "haiku"], "devM-sonnet-13500-0": [true, 1.03492473757745, "Summary", "sonnet"], "devM-sonnet-13500-1": [true, 1.4676430265507967, "Summary", "sonnet"], "devM-haiku-13509-0": [true, -0.06545931612857156, "Summary", "haiku"], "devM-haiku-13509-1": [true, 0.23829027817415993, "Summary", "haiku"], "devM-sonnet-13509-0": [true, 0.2634026645952247, "Summary", "sonnet"], "devM-sonnet-13509-1": [true, 0.14797428764271744, "Summary", "sonnet"], "devM-haiku-13516-0": [true, 0.2684794597601359, "Summary", "haiku"], "devM-haiku-13516-1": [true, 0.4425575918493459, "Summary", "haiku"], "devM-sonnet-13516-0": [true, 0.5876245884914788, "Summary", "sonnet"], "devM-sonnet-13516-1": [true, 0.1268636728124991, "Summary", "sonnet"], "devM-haiku-13555-0": [true, 0.7910313179623104, "Summary", "haiku"], "devM-haiku-13555-1": [true, 1.3858993392819503, "Summary", "haiku"], "devM-sonnet-13555-0": [true, -0.2583644200284715, "Summary", "sonnet"], "devM-sonnet-13555-1": [true, 0.16645549846988086, "Summary", "sonnet"], "devM-haiku-13556-0": [true, 0.892290475933117, "Summary", "haiku"], "devM-haiku-13556-1": [true, 0.2711928057645213, "Summary", "haiku"], "devM-sonnet-13556-0": [true, -0.1580652932831403, "Summary", "sonnet"], "devM-sonnet-13556-1": [true, 0.29893967934590066, "Summary", "sonnet"], "devM-haiku-13562-0": [true, 2.3027698047485967, "Summary", "haiku"], "devM-haiku-13562-1": [true, 1.5178979516638547, "Summary", "haiku"], "devM-sonnet-13562-0": [true, 1.3368647232398008, "Summary", "sonnet"], "devM-sonnet-13562-1": [true, 1.7440662312394128, "Summary", "sonnet"], "devM-haiku-13581-0": [true, 1.7260006812267257, "Summary", "haiku"], "devM-haiku-13581-1": [true, 1.8553401942215015, "Summary", "haiku"], "devM-sonnet-13581-0": [true, 0.45798374880511833, "Summary", "sonnet"], "devM-sonnet-13581-1": [true, 0.3446773346599785, "Summary", "sonnet"], "devM-haiku-13600-0": [true, 1.7093085878740264, "Data2txt", "haiku"], "devM-haiku-13600-1": [true, 2.202886546575216, "Data2txt", "haiku"], "devM-sonnet-13600-0": [true, 1.5726710781754396, "Data2txt", "sonnet"], "devM-sonnet-13600-1": [true, 1.887765564851512, "Data2txt", "sonnet"], "devM-haiku-13618-0": [true, 2.0229130676431506, "Data2txt", "haiku"], "devM-haiku-13618-1": [true, 0.9157381354687313, "Data2txt", "haiku"], "devM-sonnet-13618-0": [true, 2.0631120203037447, "Data2txt", "sonnet"], "devM-sonnet-13618-1": [true, 1.6031225691536828, "Data2txt", "sonnet"], "devM-haiku-13621-0": [true, 2.5352949821720925, "Data2txt", "haiku"], "devM-haiku-13621-1": [true, 2.767407849805742, "Data2txt", "haiku"], "devM-sonnet-13621-0": [true, 2.3339054494871023, "Data2txt", "sonnet"], "devM-sonnet-13621-1": [true, 2.3339054494871023, "Data2txt", "sonnet"], "devM-haiku-13622-0": [false, 2.1544458021637736, "Data2txt", "haiku"], "devM-haiku-13622-1": [false, 1.7318733370090897, "Data2txt", "haiku"], "devM-sonnet-13622-0": [false, 2.3966280537481275, "Data2txt", "sonnet"], "devM-sonnet-13622-1": [true, 2.8799153201619596, "Data2txt", "sonnet"], "devM-haiku-13643-0": [false, 1.6975316382747772, "Data2txt", "haiku"], "devM-haiku-13643-1": [true, 1.0946791093067543, "Data2txt", "haiku"], "devM-sonnet-13643-0": [true, 1.2044693203551629, "Data2txt", "sonnet"], "devM-sonnet-13643-1": [true, 1.3986418909136618, "Data2txt", "sonnet"], "devM-haiku-13649-0": [true, 2.444147999922718, "Data2txt", "haiku"], "devM-haiku-13649-1": [true, 1.0836512751260436, "Data2txt", "haiku"], "devM-sonnet-13649-0": [true, 1.4762180318910736, "Data2txt", "sonnet"], "devM-sonnet-13649-1": [true, 1.4424466489078056, "Data2txt", "sonnet"], "devM-haiku-13670-0": [false, 1.806435293206316, "Data2txt", "haiku"], "devM-haiku-13670-1": [false, 2.3351561477363134, "Data2txt", "haiku"], "devM-sonnet-13670-0": [true, 2.5802935049689175, "Data2txt", "sonnet"], "devM-sonnet-13670-1": [true, 1.4078497029722647, "Data2txt", "sonnet"], "devM-haiku-13696-0": [false, 1.146324029806931, "Data2txt", "haiku"], "devM-haiku-13696-1": [true, 1.4897975545121342, "Data2txt", "haiku"], "devM-sonnet-13696-0": [true, 1.176021928346048, "Data2txt", "sonnet"], "devM-sonnet-13696-1": [true, 1.7812883960970793, "Data2txt", "sonnet"], "devM-haiku-13704-0": [true, 3.358464161539491, "Data2txt", "haiku"], "devM-haiku-13704-1": [true, 1.9666615776558678, "Data2txt", "haiku"], "devM-sonnet-13704-0": [true, 1.108783231988569, "Data2txt", "sonnet"], "devM-sonnet-13704-1": [true, 1.0249635939904498, "Data2txt", "sonnet"], "devM-haiku-13723-0": [true, 2.051299770990219, "Data2txt", "haiku"], "devM-haiku-13723-1": [true, 2.011740693391565, "Data2txt", "haiku"], "devM-sonnet-13723-0": [false, 2.0582498954462585, "Data2txt", "sonnet"], "devM-sonnet-13723-1": [false, 2.1206521584343987, "Data2txt", "sonnet"], "devM-haiku-13729-0": [false, 2.9974287760027734, "Data2txt", "haiku"], "devM-haiku-13729-1": [false, 2.256619092256532, "Data2txt", "haiku"], "devM-sonnet-13729-0": [true, 3.7718332488458817, "Data2txt", "sonnet"], "devM-sonnet-13729-1": [true, 3.8678283423170043, "Data2txt", "sonnet"], "devM-haiku-13744-0": [true, 2.6502873410707877, "Data2txt", "haiku"], "devM-haiku-13744-1": [true, 3.1574747610199747, "Data2txt", "haiku"], "devM-sonnet-13744-0": [true, 2.937119912844202, "Data2txt", "sonnet"], "devM-sonnet-13744-1": [true, 3.014562989732192, "Data2txt", "sonnet"], "devM-haiku-13753-0": [true, 1.3188041037340121, "Data2txt", "haiku"], "devM-haiku-13753-1": [true, 1.3232490316288332, "Data2txt", "haiku"], "devM-sonnet-13753-0": [true, 1.5658113752130816, "Data2txt", "sonnet"], "devM-sonnet-13753-1": [true, 1.3028925063241874, "Data2txt", "sonnet"], "devM-haiku-13771-0": [true, 1.9232146024515167, "Data2txt", "haiku"], "devM-haiku-13771-1": [false, 1.2350468641485, "Data2txt", "haiku"], "devM-sonnet-13771-0": [true, 0.6616416449900149, "Data2txt", "sonnet"], "devM-sonnet-13771-1": [true, 1.061897053115466, "Data2txt", "sonnet"], "devM-haiku-13778-0": [true, 2.5921663484276425, "Data2txt", "haiku"], "devM-haiku-13778-1": [true, 2.624141518263762, "Data2txt", "haiku"], "devM-sonnet-13778-0": [false, 1.0973035207450519, "Data2txt", "sonnet"], "devM-sonnet-13778-1": [false, 1.1682988233667406, "Data2txt", "sonnet"], "devM-haiku-13789-0": [true, 2.0507400029135394, "Data2txt", "haiku"], "devM-haiku-13789-1": [false, 2.0739851008406385, "Data2txt", "haiku"], "devM-sonnet-13789-0": [true, 1.2173985150228228, "Data2txt", "sonnet"], "devM-sonnet-13789-1": [true, 1.15296141560165, "Data2txt", "sonnet"], "devM-haiku-13826-0": [true, -0.011916047616508524, "Summary", "haiku"], "devM-haiku-13826-1": [true, 0.14122070468398443, "Summary", "haiku"], "devM-sonnet-13826-0": [true, 0.024645736213865538, "Summary", "sonnet"], "devM-sonnet-13826-1": [true, -0.07178805670279695, "Summary", "sonnet"], "devM-haiku-13831-0": [true, 1.2342109458480146, "Summary", "haiku"], "devM-haiku-13831-1": [true, 0.20914268873541111, "Summary", "haiku"], "devM-sonnet-13831-0": [true, 0.5898328246270453, "Summary", "sonnet"], "devM-sonnet-13831-1": [true, 0.4990282257640966, "Summary", "sonnet"], "devM-haiku-13838-0": [true, 1.6970051866881923, "Summary", "haiku"], "devM-haiku-13838-1": [true, 2.0101551759471588, "Summary", "haiku"], "devM-sonnet-13838-0": [true, 1.9694792888831236, "Summary", "sonnet"], "devM-sonnet-13838-1": [true, 2.0239849775489307, "Summary", "sonnet"], "devM-haiku-13844-0": [true, 0.8698447974055761, "Summary", "haiku"], "devM-haiku-13844-1": [true, 1.345849610031921, "Summary", "haiku"], "devM-sonnet-13844-0": [true, 0.2671622407902765, "Summary", "sonnet"], "devM-sonnet-13844-1": [true, 0.2671622407902765, "Summary", "sonnet"], "devM-haiku-13855-0": [true, 1.160652468461546, "Summary", "haiku"], "devM-haiku-13855-1": [true, 1.615854143062702, "Summary", "haiku"], "devM-sonnet-13855-0": [true, 0.6273157538158713, "Summary", "sonnet"], "devM-sonnet-13855-1": [true, 0.9558061786382276, "Summary", "sonnet"], "devM-haiku-13886-0": [true, 1.0101499645349634, "Summary", "haiku"], "devM-haiku-13886-1": [true, 2.909944085906458, "Summary", "haiku"], "devM-sonnet-13886-0": [true, 0.5856412549600598, "Summary", "sonnet"], "devM-sonnet-13886-1": [true, 0.6019043681057457, "Summary", "sonnet"], "devM-haiku-13890-0": [true, 0.8719081350246045, "Summary", "haiku"], "devM-haiku-13890-1": [false, 0.5002434852009623, "Summary", "haiku"], "devM-sonnet-13890-0": [true, 0.22877701194444305, "Summary", "sonnet"], "devM-sonnet-13890-1": [true, 0.38645419679069803, "Summary", "sonnet"], "devM-haiku-13898-0": [true, 0.1518089413428252, "Summary", "haiku"], "devM-haiku-13898-1": [true, 1.3480408995163546, "Summary", "haiku"], "devM-sonnet-13898-0": [true, -0.16036738218654412, "Summary", "sonnet"], "devM-sonnet-13898-1": [true, -0.045072790139129765, "Summary", "sonnet"], "devM-haiku-13904-0": [false, 1.7959916419146011, "Summary", "haiku"], "devM-haiku-13904-1": [true, 0.8581096890708136, "Summary", "haiku"], "devM-sonnet-13904-0": [true, 1.6372148783367533, "Summary", "sonnet"], "devM-sonnet-13904-1": [true, 1.7898881003323273, "Summary", "sonnet"], "devM-haiku-13906-0": [true, 1.960980674639182, "Summary", "haiku"], "devM-haiku-13906-1": [true, 2.0982573858424614, "Summary", "haiku"], "devM-sonnet-13906-0": [true, 1.3679093728817697, "Summary", "sonnet"], "devM-sonnet-13906-1": [true, 1.0862095137218937, "Summary", "sonnet"], "devM-haiku-13921-0": [true, 2.1218610988779645, "Summary", "haiku"], "devM-haiku-13921-1": [true, 2.0560106084313188, "Summary", "haiku"], "devM-sonnet-13921-0": [true, 1.7956964899467593, "Summary", "sonnet"], "devM-sonnet-13921-1": [true, 1.1029665175085297, "Summary", "sonnet"], "devM-haiku-13958-0": [true, 2.2776577541250695, "Summary", "haiku"], "devM-haiku-13958-1": [true, 0.9681214950328179, "Summary", "haiku"], "devM-sonnet-13958-0": [true, 1.299255341418312, "Summary", "sonnet"], "devM-sonnet-13958-1": [true, 1.3109972939766914, "Summary", "sonnet"], "devM-haiku-13959-0": [true, 0.14382473427264814, "Summary", "haiku"], "devM-haiku-13959-1": [true, -0.11181984225205663, "Summary", "haiku"], "devM-sonnet-13959-0": [true, 0.022696963123328584, "Summary", "sonnet"], "devM-sonnet-13959-1": [true, 0.4985969245731912, "Summary", "sonnet"], "devM-haiku-13975-0": [true, -0.38254868668052433, "Summary", "haiku"], "devM-haiku-13975-1": [true, 1.2411472772630536, "Summary", "haiku"], "devM-sonnet-13975-0": [true, -0.038507876774605654, "Summary", "sonnet"], "devM-sonnet-13975-1": [true, -0.39478780101247346, "Summary", "sonnet"], "devM-haiku-13986-0": [false, 1.398373637735366, "Summary", "haiku"], "devM-haiku-13986-1": [true, 1.0274273096950166, "Summary", "haiku"], "devM-sonnet-13986-0": [true, 1.236190359208407, "Summary", "sonnet"], "devM-sonnet-13986-1": [true, 1.0540659995208854, "Summary", "sonnet"], "devM-haiku-13997-0": [false, 1.8959134294715145, "Data2txt", "haiku"], "devM-haiku-13997-1": [false, 2.1760019297672475, "Data2txt", "haiku"], "devM-sonnet-13997-0": [true, 0.7681328441417938, "Data2txt", "sonnet"], "devM-sonnet-13997-1": [true, 1.465586349609123, "Data2txt", "sonnet"], "devM-haiku-14002-0": [true, 2.1565662405959865, "Data2txt", "haiku"], "devM-haiku-14002-1": [true, 2.3813946928850065, "Data2txt", "haiku"], "devM-sonnet-14002-0": [true, 2.678260719386708, "Data2txt", "sonnet"], "devM-sonnet-14002-1": [true, 2.629858708191928, "Data2txt", "sonnet"], "devM-haiku-14042-0": [true, 2.649981800231517, "Data2txt", "haiku"], "devM-haiku-14042-1": [true, 2.5833187354161713, "Data2txt", "haiku"], "devM-sonnet-14042-0": [true, 2.966341368562264, "Data2txt", "sonnet"], "devM-sonnet-14042-1": [true, 3.1509600529292214, "Data2txt", "sonnet"], "devM-haiku-14061-0": [true, 3.006493534918095, "Data2txt", "haiku"], "devM-haiku-14061-1": [false, 1.6112750678927799, "Data2txt", "haiku"], "devM-sonnet-14061-0": [true, 0.6221712794005386, "Data2txt", "sonnet"], "devM-sonnet-14061-1": [true, 0.9467012355130112, "Data2txt", "sonnet"], "devM-haiku-14067-0": [true, 1.3419747854081538, "Data2txt", "haiku"], "devM-haiku-14067-1": [true, 1.307383186558689, "Data2txt", "haiku"], "devM-sonnet-14067-0": [true, 1.2432123179479537, "Data2txt", "sonnet"], "devM-sonnet-14067-1": [true, 0.5230560810543898, "Data2txt", "sonnet"], "devM-haiku-14071-0": [false, 1.683567438040294, "Data2txt", "haiku"], "devM-haiku-14071-1": [true, 1.8388437423276058, "Data2txt", "haiku"], "devM-sonnet-14071-0": [true, 2.4030900036624177, "Data2txt", "sonnet"], "devM-sonnet-14071-1": [true, 2.3402859103483626, "Data2txt", "sonnet"], "devM-haiku-14075-0": [true, 2.820411472618402, "Data2txt", "haiku"], "devM-haiku-14075-1": [false, 2.20993456068612, "Data2txt", "haiku"], "devM-sonnet-14075-0": [true, 2.078201007032587, "Data2txt", "sonnet"], "devM-sonnet-14075-1": [true, 1.8072265053043246, "Data2txt", "sonnet"], "devM-haiku-14085-0": [false, 3.3471687989205003, "Data2txt", "haiku"], "devM-haiku-14085-1": [true, 2.814489807988574, "Data2txt", "haiku"], "devM-sonnet-14085-0": [true, 1.5423924637646147, "Data2txt", "sonnet"], "devM-sonnet-14085-1": [true, 1.8627570340704938, "Data2txt", "sonnet"], "devM-haiku-14097-0": [true, 1.0166372446481222, "Data2txt", "haiku"], "devM-haiku-14097-1": [false, 3.3391599421759484, "Data2txt", "haiku"], "devM-sonnet-14097-0": [true, 1.4282541805111657, "Data2txt", "sonnet"], "devM-sonnet-14097-1": [true, 1.49392077137145, "Data2txt", "sonnet"], "devM-haiku-14106-0": [true, 1.8277324191080746, "Data2txt", "haiku"], "devM-haiku-14106-1": [true, 1.4841935556499801, "Data2txt", "haiku"], "devM-sonnet-14106-0": [true, 2.076165505157054, "Data2txt", "sonnet"], "devM-sonnet-14106-1": [true, 1.6962786753959307, "Data2txt", "sonnet"], "devM-haiku-14107-0": [true, 0.8311778162339635, "Data2txt", "haiku"], "devM-haiku-14107-1": [false, 1.8766172139754773, "Data2txt", "haiku"], "devM-sonnet-14107-0": [false, 2.003919064609879, "Data2txt", "sonnet"], "devM-sonnet-14107-1": [false, 1.3768253199771299, "Data2txt", "sonnet"], "devM-haiku-14136-0": [true, 2.6236924100174406, "Data2txt", "haiku"], "devM-haiku-14136-1": [true, 2.391098560750346, "Data2txt", "haiku"], "devM-sonnet-14136-0": [true, 2.6907836002975474, "Data2txt", "sonnet"], "devM-sonnet-14136-1": [true, 2.6757335220838545, "Data2txt", "sonnet"], "devM-haiku-14142-0": [true, 2.892173012342192, "Data2txt", "haiku"], "devM-haiku-14142-1": [true, 2.4179767866962933, "Data2txt", "haiku"], "devM-sonnet-14142-0": [true, 2.7360533629651185, "Data2txt", "sonnet"], "devM-sonnet-14142-1": [true, 2.472215241349452, "Data2txt", "sonnet"], "devM-haiku-14154-0": [true, 1.7553814705941442, "Data2txt", "haiku"], "devM-haiku-14154-1": [true, 0.8275854524464973, "Data2txt", "haiku"], "devM-sonnet-14154-0": [false, 2.730295129403861, "Data2txt", "sonnet"], "devM-sonnet-14154-1": [false, 2.0850368697665704, "Data2txt", "sonnet"], "devM-haiku-14159-0": [true, 1.9447550381881646, "Data2txt", "haiku"], "devM-haiku-14159-1": [true, 2.0360724046469234, "Data2txt", "haiku"], "devM-sonnet-14159-0": [false, 1.665176603410561, "Data2txt", "sonnet"], "devM-sonnet-14159-1": [false, 3.1297280233273486, "Data2txt", "sonnet"], "devM-haiku-14164-0": [true, 2.3173293859234483, "Data2txt", "haiku"], "devM-haiku-14164-1": [true, 3.4017871323785345, "Data2txt", "haiku"], "devM-sonnet-14164-0": [true, 2.1759982769172677, "Data2txt", "sonnet"], "devM-sonnet-14164-1": [true, 3.0215787008981883, "Data2txt", "sonnet"], "devM-haiku-14173-0": [true, 2.738541555273803, "Data2txt", "haiku"], "devM-haiku-14173-1": [true, 1.4182940297833997, "Data2txt", "haiku"], "devM-sonnet-14173-0": [true, 2.0833506325495557, "Data2txt", "sonnet"], "devM-sonnet-14173-1": [false, 2.592718500149237, "Data2txt", "sonnet"], "devM-haiku-14179-0": [false, 3.7122531953153377, "Data2txt", "haiku"], "devM-haiku-14179-1": [true, 2.3758184495168226, "Data2txt", "haiku"], "devM-sonnet-14179-0": [true, 2.3807839017077512, "Data2txt", "sonnet"], "devM-sonnet-14179-1": [true, 3.3015176970929314, "Data2txt", "sonnet"], "devM-haiku-14198-0": [true, 1.5804956315917162, "Data2txt", "haiku"], "devM-haiku-14198-1": [true, 1.9720625407289292, "Data2txt", "haiku"], "devM-sonnet-14198-0": [true, 1.4601221641440751, "Data2txt", "sonnet"], "devM-sonnet-14198-1": [true, 1.3506563765518287, "Data2txt", "sonnet"], "devM-haiku-14214-0": [false, 1.7281390446359273, "Data2txt", "haiku"], "devM-haiku-14214-1": [true, 2.678251451148519, "Data2txt", "haiku"], "devM-sonnet-14214-0": [true, 2.3197392137224786, "Data2txt", "sonnet"], "devM-sonnet-14214-1": [true, 2.399057193497347, "Data2txt", "sonnet"], "devM-haiku-14232-0": [true, 2.0702513953565664, "Data2txt", "haiku"], "devM-haiku-14232-1": [true, 1.114909178092846, "Data2txt", "haiku"], "devM-sonnet-14232-0": [true, 1.9580408427487508, "Data2txt", "sonnet"], "devM-sonnet-14232-1": [true, 1.5386266384828118, "Data2txt", "sonnet"], "devM-haiku-14250-0": [true, 1.4400086657972357, "Data2txt", "haiku"], "devM-haiku-14250-1": [true, 0.6583814116062983, "Data2txt", "haiku"], "devM-sonnet-14250-0": [true, 1.5454254934252087, "Data2txt", "sonnet"], "devM-sonnet-14250-1": [true, 1.5454254934252087, "Data2txt", "sonnet"], "devM-haiku-14251-0": [true, 2.1189930341830365, "Data2txt", "haiku"], "devM-haiku-14251-1": [true, 1.2774488215934992, "Data2txt", "haiku"], "devM-sonnet-14251-0": [false, 0.8812248351467171, "Data2txt", "sonnet"], "devM-sonnet-14251-1": [true, 1.0701386223094038, "Data2txt", "sonnet"], "devM-haiku-14294-0": [false, 2.277044196850102, "QA", "haiku"], "devM-haiku-14294-1": [true, 2.194675525545431, "QA", "haiku"], "devM-sonnet-14294-0": [false, 1.7942292893031024, "QA", "sonnet"], "devM-sonnet-14294-1": [true, 1.6208068666745292, "QA", "sonnet"], "devM-haiku-14299-0": [true, 1.6151542587824805, "QA", "haiku"], "devM-haiku-14299-1": [true, 2.1470878009226264, "QA", "haiku"], "devM-sonnet-14299-0": [true, 1.9619184904148017, "QA", "sonnet"], "devM-sonnet-14299-1": [true, 3.5246465760107917, "QA", "sonnet"], "devM-haiku-14303-0": [true, 1.8644883594318313, "QA", "haiku"], "devM-haiku-14303-1": [true, 1.318683134076178, "QA", "haiku"], "devM-sonnet-14303-0": [true, 1.3321891754360586, "QA", "sonnet"], "devM-sonnet-14303-1": [true, 1.4191582885047733, "QA", "sonnet"], "devM-haiku-14305-0": [true, 1.6733469591822703, "QA", "haiku"], "devM-haiku-14305-1": [true, 1.6607374341924195, "QA", "haiku"], "devM-sonnet-14305-0": [true, 1.716127533023634, "QA", "sonnet"], "devM-sonnet-14305-1": [true, 2.713773665465582, "QA", "sonnet"], "devM-haiku-14312-0": [true, 0.9375841787257743, "QA", "haiku"], "devM-haiku-14312-1": [true, 1.0014521973085369, "QA", "haiku"], "devM-sonnet-14312-0": [true, 1.4004753417445892, "QA", "sonnet"], "devM-sonnet-14312-1": [true, 1.371188703058484, "QA", "sonnet"], "devM-haiku-14338-0": [true, 3.5442912014230954, "QA", "haiku"], "devM-haiku-14338-1": [true, 2.1317723953708003, "QA", "haiku"], "devM-sonnet-14338-0": [true, 1.2694600370245417, "QA", "sonnet"], "devM-sonnet-14338-1": [true, 1.9509399348968883, "QA", "sonnet"], "devM-haiku-14352-0": [true, 1.9322948240336366, "QA", "haiku"], "devM-haiku-14352-1": [true, 2.4989863359748163, "QA", "haiku"], "devM-sonnet-14352-0": [true, 2.297938273351514, "QA", "sonnet"], "devM-sonnet-14352-1": [true, 3.0953963941624782, "QA", "sonnet"], "devM-haiku-14370-0": [false, 3.713166851826654, "QA", "haiku"], "devM-haiku-14370-1": [true, 1.268474957820322, "QA", "haiku"], "devM-sonnet-14370-0": [false, 1.4862788294551197, "QA", "sonnet"], "devM-sonnet-14370-1": [true, 1.4338636022524618, "QA", "sonnet"], "devM-haiku-14378-0": [true, 1.7823436741972538, "QA", "haiku"], "devM-haiku-14378-1": [true, 2.564005379778848, "QA", "haiku"], "devM-sonnet-14378-0": [true, 2.206390579447293, "QA", "sonnet"], "devM-sonnet-14378-1": [true, 1.6889885033367935, "QA", "sonnet"], "devM-haiku-14380-0": [false, 2.607698983832703, "QA", "haiku"], "devM-haiku-14380-1": [false, 3.448510959444401, "QA", "haiku"], "devM-sonnet-14380-0": [true, 1.7152992601449912, "QA", "sonnet"], "devM-sonnet-14380-1": [true, 1.7741346140095824, "QA", "sonnet"], "devM-haiku-14384-0": [true, 1.1376261909695262, "QA", "haiku"], "devM-haiku-14384-1": [true, 1.2384507932038162, "QA", "haiku"], "devM-sonnet-14384-0": [true, 1.1379827361855919, "QA", "sonnet"], "devM-sonnet-14384-1": [true, 1.0862850375463422, "QA", "sonnet"], "devM-haiku-14388-0": [true, 1.1975016499207227, "QA", "haiku"], "devM-haiku-14388-1": [true, 0.7356297431496426, "QA", "haiku"], "devM-sonnet-14388-0": [true, 1.4710970001147754, "QA", "sonnet"], "devM-sonnet-14388-1": [true, 1.2334975464568732, "QA", "sonnet"], "devM-haiku-14389-0": [true, 1.9190058293798138, "QA", "haiku"], "devM-haiku-14389-1": [true, 1.9601129969488014, "QA", "haiku"], "devM-sonnet-14389-0": [true, 2.751075094738377, "QA", "sonnet"], "devM-sonnet-14389-1": [true, 1.7983701101413057, "QA", "sonnet"], "devM-haiku-14403-0": [true, 3.0041416031109285, "QA", "haiku"], "devM-haiku-14403-1": [true, 2.385909279330087, "QA", "haiku"], "devM-sonnet-14403-0": [true, 3.064748674836341, "QA", "sonnet"], "devM-sonnet-14403-1": [true, 3.0736973699904286, "QA", "sonnet"], "devM-haiku-14407-0": [false, 2.9586414263297387, "QA", "haiku"], "devM-haiku-14407-1": [false, 2.611819521262464, "QA", "haiku"], "devM-sonnet-14407-0": [true, 2.013481249094558, "QA", "sonnet"], "devM-sonnet-14407-1": [true, 1.6039677103961059, "QA", "sonnet"], "devM-haiku-14408-0": [true, 2.101057821039678, "QA", "haiku"], "devM-haiku-14408-1": [false, 2.670102688452186, "QA", "haiku"], "devM-sonnet-14408-0": [true, 2.1362450769304484, "QA", "sonnet"], "devM-sonnet-14408-1": [true, 0.8809697881203311, "QA", "sonnet"], "devM-haiku-14412-0": [true, 0.7043585021049759, "QA", "haiku"], "devM-haiku-14412-1": [true, 0.5173624701445552, "QA", "haiku"], "devM-sonnet-14412-0": [true, 1.4662659342514086, "QA", "sonnet"], "devM-sonnet-14412-1": [true, 1.20420521088589, "QA", "sonnet"], "devM-haiku-14457-0": [true, 2.2760619014862895, "QA", "haiku"], "devM-haiku-14457-1": [true, 2.135083501699018, "QA", "haiku"], "devM-sonnet-14457-0": [true, 2.5932401160118284, "QA", "sonnet"], "devM-sonnet-14457-1": [true, 1.6327386588139825, "QA", "sonnet"], "devM-haiku-14471-0": [true, 1.6043454119545315, "QA", "haiku"], "devM-haiku-14471-1": [true, 1.390569284983719, "QA", "haiku"], "devM-sonnet-14471-0": [true, 0.9254162798359066, "QA", "sonnet"], "devM-sonnet-14471-1": [true, 1.4410652848593313, "QA", "sonnet"], "devM-haiku-14483-0": [true, 1.5711440662601133, "QA", "haiku"], "devM-haiku-14483-1": [false, 1.5264650607567933, "QA", "haiku"], "devM-sonnet-14483-0": [true, 2.6725709971699985, "QA", "sonnet"], "devM-sonnet-14483-1": [true, 1.6443255053024273, "QA", "sonnet"], "devM-haiku-14493-0": [true, 1.5485358644805225, "Data2txt", "haiku"], "devM-haiku-14493-1": [false, 1.8748406911535342, "Data2txt", "haiku"], "devM-sonnet-14493-0": [true, 1.1808908684934232, "Data2txt", "sonnet"], "devM-sonnet-14493-1": [true, 2.4114249869290045, "Data2txt", "sonnet"], "devM-haiku-14514-0": [true, 1.4174028230518845, "Data2txt", "haiku"], "devM-haiku-14514-1": [true, 0.8313245544949839, "Data2txt", "haiku"], "devM-sonnet-14514-0": [true, 2.4010695720358317, "Data2txt", "sonnet"], "devM-sonnet-14514-1": [true, 1.9819995868434277, "Data2txt", "sonnet"], "devM-haiku-14517-0": [true, 2.5949477114421597, "Data2txt", "haiku"], "devM-haiku-14517-1": [true, 0.9972757451489216, "Data2txt", "haiku"], "devM-sonnet-14517-0": [true, 2.414730698912723, "Data2txt", "sonnet"], "devM-sonnet-14517-1": [true, 2.3364253199550706, "Data2txt", "sonnet"], "devM-haiku-14522-0": [true, 1.7935351636995605, "Data2txt", "haiku"], "devM-haiku-14522-1": [true, 0.8494942902499669, "Data2txt", "haiku"], "devM-sonnet-14522-0": [true, 1.3868496832040869, "Data2txt", "sonnet"], "devM-sonnet-14522-1": [true, 1.7292227249487921, "Data2txt", "sonnet"], "devM-haiku-14525-0": [true, 1.9071627718976638, "Data2txt", "haiku"], "devM-haiku-14525-1": [true, 2.3433507041442327, "Data2txt", "haiku"], "devM-sonnet-14525-0": [true, 2.5912311894334525, "Data2txt", "sonnet"], "devM-sonnet-14525-1": [true, 1.7227213756603095, "Data2txt", "sonnet"], "devM-haiku-14534-0": [true, 1.1422647687181648, "Data2txt", "haiku"], "devM-haiku-14534-1": [false, 1.2481978823062114, "Data2txt", "haiku"], "devM-sonnet-14534-0": [true, 1.287131999140738, "Data2txt", "sonnet"], "devM-sonnet-14534-1": [true, 1.7056838158162295, "Data2txt", "sonnet"], "devM-haiku-14554-0": [true, 2.1803934392145936, "Data2txt", "haiku"], "devM-haiku-14554-1": [false, 3.1159264262577118, "Data2txt", "haiku"], "devM-sonnet-14554-0": [true, 1.8852853350384262, "Data2txt", "sonnet"], "devM-sonnet-14554-1": [true, 0.8879486606635175, "Data2txt", "sonnet"], "devM-haiku-14561-0": [true, 3.674672310489443, "Data2txt", "haiku"], "devM-haiku-14561-1": [true, 1.686993578089007, "Data2txt", "haiku"], "devM-sonnet-14561-0": [true, 2.055720233796733, "Data2txt", "sonnet"], "devM-sonnet-14561-1": [true, 2.055720233796733, "Data2txt", "sonnet"], "devM-haiku-14570-0": [true, 0.6791741361497602, "Data2txt", "haiku"], "devM-haiku-14570-1": [true, 0.7000837319382841, "Data2txt", "haiku"], "devM-sonnet-14570-0": [true, 1.6873236366382451, "Data2txt", "sonnet"], "devM-sonnet-14570-1": [true, 1.4560964681509838, "Data2txt", "sonnet"], "devM-haiku-14586-0": [true, 1.5057679725626232, "Data2txt", "haiku"], "devM-haiku-14586-1": [true, 2.288626431867578, "Data2txt", "haiku"], "devM-sonnet-14586-0": [true, 2.087503604888632, "Data2txt", "sonnet"], "devM-sonnet-14586-1": [true, 1.9997428010349858, "Data2txt", "sonnet"], "devM-haiku-14594-0": [true, 2.6844413650322982, "Data2txt", "haiku"], "devM-haiku-14594-1": [false, 2.5997550422030327, "Data2txt", "haiku"], "devM-sonnet-14594-0": [true, 2.4787912936923195, "Data2txt", "sonnet"], "devM-sonnet-14594-1": [true, 2.8110288027303714, "Data2txt", "sonnet"], "devM-haiku-14595-0": [true, 1.1067909595978673, "Data2txt", "haiku"], "devM-haiku-14595-1": [false, 1.6749591450106225, "Data2txt", "haiku"], "devM-sonnet-14595-0": [false, 0.34521959184456025, "Data2txt", "sonnet"], "devM-sonnet-14595-1": [true, 1.4723342108066233, "Data2txt", "sonnet"], "devM-haiku-14606-0": [false, 2.1254435189078222, "Data2txt", "haiku"], "devM-haiku-14606-1": [true, 2.1972153834830093, "Data2txt", "haiku"], "devM-sonnet-14606-0": [true, 2.554490854308365, "Data2txt", "sonnet"], "devM-sonnet-14606-1": [true, 2.6988103086371664, "Data2txt", "sonnet"], "devM-haiku-14621-0": [true, 2.0295694283516923, "Data2txt", "haiku"], "devM-haiku-14621-1": [true, 1.4390843486369849, "Data2txt", "haiku"], "devM-sonnet-14621-0": [true, 1.6663161758051546, "Data2txt", "sonnet"], "devM-sonnet-14621-1": [true, 1.538046055747827, "Data2txt", "sonnet"], "devM-haiku-14623-0": [true, 1.9989290494127323, "Data2txt", "haiku"], "devM-haiku-14623-1": [true, 1.9306503305794318, "Data2txt", "haiku"], "devM-sonnet-14623-0": [false, 1.496769253319769, "Data2txt", "sonnet"], "devM-sonnet-14623-1": [true, 1.325405921437493, "Data2txt", "sonnet"], "devM-haiku-14626-0": [true, 1.377138620838257, "Data2txt", "haiku"], "devM-haiku-14626-1": [true, 1.7570234039579529, "Data2txt", "haiku"], "devM-sonnet-14626-0": [true, 2.108312388349873, "Data2txt", "sonnet"], "devM-sonnet-14626-1": [true, 2.0484240278823465, "Data2txt", "sonnet"], "devM-haiku-14636-0": [false, 2.327723284903903, "Data2txt", "haiku"], "devM-haiku-14636-1": [true, 1.2373377980685523, "Data2txt", "haiku"], "devM-sonnet-14636-0": [true, 2.1764350763314098, "Data2txt", "sonnet"], "devM-sonnet-14636-1": [true, 2.4492032926047544, "Data2txt", "sonnet"], "devM-haiku-14643-0": [false, 1.8509367242691486, "Data2txt", "haiku"], "devM-haiku-14643-1": [true, 2.251266154568926, "Data2txt", "haiku"], "devM-sonnet-14643-0": [false, 2.7972421419831006, "Data2txt", "sonnet"], "devM-sonnet-14643-1": [true, 3.2304631021442876, "Data2txt", "sonnet"], "devM-haiku-14645-0": [true, 1.4178260918406416, "Data2txt", "haiku"], "devM-haiku-14645-1": [true, 1.7494494653199253, "Data2txt", "haiku"], "devM-sonnet-14645-0": [true, 1.089183007877105, "Data2txt", "sonnet"], "devM-sonnet-14645-1": [true, 1.1447654217527905, "Data2txt", "sonnet"], "devM-haiku-14651-0": [true, 1.9636344264912626, "Data2txt", "haiku"], "devM-haiku-14651-1": [false, 3.218891931517661, "Data2txt", "haiku"], "devM-sonnet-14651-0": [true, 2.227665546070538, "Data2txt", "sonnet"], "devM-sonnet-14651-1": [true, 2.426630016733127, "Data2txt", "sonnet"], "devM-haiku-14666-0": [false, 1.3957664897559012, "Data2txt", "haiku"], "devM-haiku-14666-1": [false, 1.4460306963351828, "Data2txt", "haiku"], "devM-sonnet-14666-0": [true, 1.8080759592686544, "Data2txt", "sonnet"], "devM-sonnet-14666-1": [true, 1.8377693568253999, "Data2txt", "sonnet"], "devM-haiku-14687-0": [false, 1.0331259914569617, "Data2txt", "haiku"], "devM-haiku-14687-1": [false, 3.2158983284958937, "Data2txt", "haiku"], "devM-sonnet-14687-0": [true, 1.1496562454768284, "Data2txt", "sonnet"], "devM-sonnet-14687-1": [true, 1.1499783826363161, "Data2txt", "sonnet"], "devM-haiku-14691-0": [false, 2.002964584303609, "Data2txt", "haiku"], "devM-haiku-14691-1": [false, 1.8177279576719707, "Data2txt", "haiku"], "devM-sonnet-14691-0": [true, 0.819597278395842, "Data2txt", "sonnet"], "devM-sonnet-14691-1": [true, 0.7854514064957387, "Data2txt", "sonnet"], "devM-haiku-14692-0": [true, 1.3331037546340028, "Data2txt", "haiku"], "devM-haiku-14692-1": [false, 1.824612660373563, "Data2txt", "haiku"], "devM-sonnet-14692-0": [true, 2.5990917486218126, "Data2txt", "sonnet"], "devM-sonnet-14692-1": [false, 2.451857908560978, "Data2txt", "sonnet"], "devM-haiku-14701-0": [true, 2.4286233723095076, "Data2txt", "haiku"], "devM-haiku-14701-1": [false, 2.8416670992057864, "Data2txt", "haiku"], "devM-sonnet-14701-0": [true, 2.367774323820866, "Data2txt", "sonnet"], "devM-sonnet-14701-1": [true, 2.581569205900621, "Data2txt", "sonnet"], "devM-haiku-14703-0": [false, 1.556316641535397, "Data2txt", "haiku"], "devM-haiku-14703-1": [true, 1.788758867588567, "Data2txt", "haiku"], "devM-sonnet-14703-0": [true, 1.901660681385735, "Data2txt", "sonnet"], "devM-sonnet-14703-1": [true, 1.71649947590538, "Data2txt", "sonnet"], "devM-haiku-14713-0": [true, 1.8901594676593436, "Data2txt", "haiku"], "devM-haiku-14713-1": [false, 1.7771500695790827, "Data2txt", "haiku"], "devM-sonnet-14713-0": [true, 2.2888975378117435, "Data2txt", "sonnet"], "devM-sonnet-14713-1": [true, 2.2888975378117435, "Data2txt", "sonnet"], "devM-haiku-14740-0": [true, 1.6193347148380208, "Data2txt", "haiku"], "devM-haiku-14740-1": [true, 1.0115394761883858, "Data2txt", "haiku"], "devM-sonnet-14740-0": [true, 1.9012889627570142, "Data2txt", "sonnet"], "devM-sonnet-14740-1": [true, 1.7544037152690795, "Data2txt", "sonnet"], "devM-haiku-14743-0": [true, 1.8704181677422627, "Data2txt", "haiku"], "devM-haiku-14743-1": [true, 2.5745965410722795, "Data2txt", "haiku"], "devM-sonnet-14743-0": [true, 2.47016322759363, "Data2txt", "sonnet"], "devM-sonnet-14743-1": [true, 2.4223877966785503, "Data2txt", "sonnet"], "devM-haiku-14757-0": [true, 1.443504668229337, "Data2txt", "haiku"], "devM-haiku-14757-1": [true, 2.309051445868646, "Data2txt", "haiku"], "devM-sonnet-14757-0": [true, 1.8775898018031265, "Data2txt", "sonnet"], "devM-sonnet-14757-1": [true, 1.7711308304059747, "Data2txt", "sonnet"], "devM-haiku-14758-0": [true, 1.583594633801437, "Data2txt", "haiku"], "devM-haiku-14758-1": [true, 1.6138419417848766, "Data2txt", "haiku"], "devM-sonnet-14758-0": [true, 1.5527648149659437, "Data2txt", "sonnet"], "devM-sonnet-14758-1": [true, 2.3023255957557582, "Data2txt", "sonnet"], "devM-haiku-14768-0": [true, 1.702259764317122, "Data2txt", "haiku"], "devM-haiku-14768-1": [true, 2.104705790331931, "Data2txt", "haiku"], "devM-sonnet-14768-0": [true, 2.2592002495997003, "Data2txt", "sonnet"], "devM-sonnet-14768-1": [true, 2.0122350397137976, "Data2txt", "sonnet"], "devM-haiku-14781-0": [true, 2.219132091227404, "Data2txt", "haiku"], "devM-haiku-14781-1": [true, 3.1450282180605296, "Data2txt", "haiku"], "devM-sonnet-14781-0": [true, 3.101911777772652, "Data2txt", "sonnet"], "devM-sonnet-14781-1": [true, 3.074252174569108, "Data2txt", "sonnet"], "devM-haiku-14784-0": [false, 1.304310123536439, "Data2txt", "haiku"], "devM-haiku-14784-1": [true, 1.2329015288747036, "Data2txt", "haiku"], "devM-sonnet-14784-0": [true, 2.2380589546409206, "Data2txt", "sonnet"], "devM-sonnet-14784-1": [true, 2.2679068773980564, "Data2txt", "sonnet"], "devM-haiku-14792-0": [true, 1.2428948560134092, "Data2txt", "haiku"], "devM-haiku-14792-1": [true, 1.0055640514270272, "Data2txt", "haiku"], "devM-sonnet-14792-0": [true, 0.7028319845240658, "Data2txt", "sonnet"], "devM-sonnet-14792-1": [true, 0.8661645000260032, "Data2txt", "sonnet"], "devM-haiku-14794-0": [true, 1.8812953939503512, "Data2txt", "haiku"], "devM-haiku-14794-1": [false, 2.621366635498196, "Data2txt", "haiku"], "devM-sonnet-14794-0": [true, 2.119360627028343, "Data2txt", "sonnet"], "devM-sonnet-14794-1": [true, 1.1984876918033045, "Data2txt", "sonnet"], "devM-haiku-14822-0": [false, 1.3750502348621847, "Data2txt", "haiku"], "devM-haiku-14822-1": [true, 1.4997572788498603, "Data2txt", "haiku"], "devM-sonnet-14822-0": [true, 2.517480388621709, "Data2txt", "sonnet"], "devM-sonnet-14822-1": [true, 2.037733131560794, "Data2txt", "sonnet"], "devM-haiku-14836-0": [true, 1.8268871419768986, "Data2txt", "haiku"], "devM-haiku-14836-1": [true, 1.930437909387178, "Data2txt", "haiku"], "devM-sonnet-14836-0": [true, 1.8743561979964083, "Data2txt", "sonnet"], "devM-sonnet-14836-1": [true, 1.3904095899579125, "Data2txt", "sonnet"], "devM-haiku-14853-0": [false, 0.9637775089354009, "Data2txt", "haiku"], "devM-haiku-14853-1": [true, 1.1047755988233918, "Data2txt", "haiku"], "devM-sonnet-14853-0": [true, 0.627124991681198, "Data2txt", "sonnet"], "devM-sonnet-14853-1": [true, 1.1858407477201847, "Data2txt", "sonnet"], "devM-haiku-14855-0": [false, 3.5720746949947193, "Data2txt", "haiku"], "devM-haiku-14855-1": [false, 2.155529468209317, "Data2txt", "haiku"], "devM-sonnet-14855-0": [true, 2.049392070824017, "Data2txt", "sonnet"], "devM-sonnet-14855-1": [true, 1.639807835685878, "Data2txt", "sonnet"], "devM-haiku-14863-0": [true, 1.5176128424269277, "Data2txt", "haiku"], "devM-haiku-14863-1": [false, 1.367315439725318, "Data2txt", "haiku"], "devM-sonnet-14863-0": [true, 1.124687069477348, "Data2txt", "sonnet"], "devM-sonnet-14863-1": [false, 1.1954631129687694, "Data2txt", "sonnet"], "devM-haiku-14883-0": [true, 1.6297306879012163, "Data2txt", "haiku"], "devM-haiku-14883-1": [true, 1.9665898445323247, "Data2txt", "haiku"], "devM-sonnet-14883-0": [false, 0.9590677012689824, "Data2txt", "sonnet"], "devM-sonnet-14883-1": [false, 1.6898868589275056, "Data2txt", "sonnet"], "devM-haiku-14885-0": [true, 1.3319921435286812, "Data2txt", "haiku"], "devM-haiku-14885-1": [true, 2.019790623060452, "Data2txt", "haiku"], "devM-sonnet-14885-0": [true, 1.4595483410597514, "Data2txt", "sonnet"], "devM-sonnet-14885-1": [true, 1.6671538035901436, "Data2txt", "sonnet"], "devM-haiku-14889-0": [true, 1.3072594055068916, "Data2txt", "haiku"], "devM-haiku-14889-1": [true, 1.065447668459177, "Data2txt", "haiku"], "devM-sonnet-14889-0": [true, 2.0809444502007874, "Data2txt", "sonnet"], "devM-sonnet-14889-1": [true, 1.568279683930894, "Data2txt", "sonnet"], "devM-haiku-14919-0": [false, 1.5567695604119767, "Data2txt", "haiku"], "devM-haiku-14919-1": [true, 1.7905946413655587, "Data2txt", "haiku"], "devM-sonnet-14919-0": [true, 2.1449497705921896, "Data2txt", "sonnet"], "devM-sonnet-14919-1": [true, 1.9563976900901814, "Data2txt", "sonnet"], "devM-haiku-14939-0": [true, 1.4111407976695465, "Data2txt", "haiku"], "devM-haiku-14939-1": [false, 0.9386569089271838, "Data2txt", "haiku"], "devM-sonnet-14939-0": [true, 1.8397760259078009, "Data2txt", "sonnet"], "devM-sonnet-14939-1": [true, 1.4417317209511071, "Data2txt", "sonnet"], "devM-haiku-14940-0": [true, 2.660307455720584, "Data2txt", "haiku"], "devM-haiku-14940-1": [false, 2.8560493430304397, "Data2txt", "haiku"], "devM-sonnet-14940-0": [true, 0.9086357349274727, "Data2txt", "sonnet"], "devM-sonnet-14940-1": [true, 0.9708343836182342, "Data2txt", "sonnet"], "devM-haiku-14944-0": [true, 1.9014001306985668, "Data2txt", "haiku"], "devM-haiku-14944-1": [true, 1.2166435234965751, "Data2txt", "haiku"], "devM-sonnet-14944-0": [true, 2.038911146670763, "Data2txt", "sonnet"], "devM-sonnet-14944-1": [false, 2.0962641474310524, "Data2txt", "sonnet"], "devM-haiku-14971-0": [true, 1.3870418279303305, "Data2txt", "haiku"], "devM-haiku-14971-1": [true, 1.189048608926658, "Data2txt", "haiku"], "devM-sonnet-14971-0": [false, 1.5482278240373142, "Data2txt", "sonnet"], "devM-sonnet-14971-1": [false, 1.5620576256390861, "Data2txt", "sonnet"], "devM-haiku-14979-0": [false, 1.6573797148957525, "Data2txt", "haiku"], "devM-haiku-14979-1": [false, 1.728999058180685, "Data2txt", "haiku"], "devM-sonnet-14979-0": [true, 1.2250427141573788, "Data2txt", "sonnet"], "devM-sonnet-14979-1": [true, 1.263098742862828, "Data2txt", "sonnet"], "devM-haiku-14982-0": [true, 1.927606522283894, "Data2txt", "haiku"], "devM-haiku-14982-1": [true, 1.6461620412669966, "Data2txt", "haiku"], "devM-sonnet-14982-0": [true, 0.9259042731926512, "Data2txt", "sonnet"], "devM-sonnet-14982-1": [true, 1.5550020670418188, "Data2txt", "sonnet"], "devM-haiku-14995-0": [true, 2.1621727603508174, "Data2txt", "haiku"], "devM-haiku-14995-1": [false, 1.4789819869111436, "Data2txt", "haiku"], "devM-sonnet-14995-0": [false, 1.732380706336056, "Data2txt", "sonnet"], "devM-sonnet-14995-1": [true, 1.732380706336056, "Data2txt", "sonnet"], "devM-haiku-14999-0": [false, 1.4782573357915743, "Data2txt", "haiku"], "devM-haiku-14999-1": [false, 1.8333202477739943, "Data2txt", "haiku"], "devM-sonnet-14999-0": [true, 2.1273239126182193, "Data2txt", "sonnet"], "devM-sonnet-14999-1": [true, 2.2957908455372533, "Data2txt", "sonnet"], "devM-haiku-15005-0": [true, 2.2988138015793456, "Data2txt", "haiku"], "devM-haiku-15005-1": [false, 3.3431784048699154, "Data2txt", "haiku"], "devM-sonnet-15005-0": [true, 2.1077409046718287, "Data2txt", "sonnet"], "devM-sonnet-15005-1": [true, 1.772203130368348, "Data2txt", "sonnet"], "devM-haiku-15017-0": [false, 2.0892936384066068, "Data2txt", "haiku"], "devM-haiku-15017-1": [false, 2.326372812973437, "Data2txt", "haiku"], "devM-sonnet-15017-0": [true, 1.4006114909676528, "Data2txt", "sonnet"], "devM-sonnet-15017-1": [true, 1.1689401322113335, "Data2txt", "sonnet"], "devM-haiku-15033-0": [true, 3.274060076989617, "Data2txt", "haiku"], "devM-haiku-15033-1": [true, 2.9154578218118092, "Data2txt", "haiku"], "devM-sonnet-15033-0": [false, 4.023623918302775, "Data2txt", "sonnet"], "devM-sonnet-15033-1": [false, 2.743648083270349, "Data2txt", "sonnet"], "devM-haiku-15065-0": [false, 2.5026718287711898, "Data2txt", "haiku"], "devM-haiku-15065-1": [false, 1.7926276449965406, "Data2txt", "haiku"], "devM-sonnet-15065-0": [true, 1.30928017685536, "Data2txt", "sonnet"], "devM-sonnet-15065-1": [true, 1.465224063683558, "Data2txt", "sonnet"], "devM-haiku-15077-0": [false, 3.3288432378513124, "Data2txt", "haiku"], "devM-haiku-15077-1": [false, 3.3188947856619095, "Data2txt", "haiku"], "devM-sonnet-15077-0": [false, 1.8875182218294544, "Data2txt", "sonnet"], "devM-sonnet-15077-1": [false, 2.714704280933179, "Data2txt", "sonnet"], "devM-haiku-15081-0": [false, 1.9802726144617881, "Data2txt", "haiku"], "devM-haiku-15081-1": [true, 2.0718024966296475, "Data2txt", "haiku"], "devM-sonnet-15081-0": [true, 2.348171520234239, "Data2txt", "sonnet"], "devM-sonnet-15081-1": [true, 2.1630720881148107, "Data2txt", "sonnet"], "devM-haiku-15087-0": [false, 2.702674608500752, "Data2txt", "haiku"], "devM-haiku-15087-1": [false, 2.8102449311525683, "Data2txt", "haiku"], "devM-sonnet-15087-0": [true, 1.969734912833924, "Data2txt", "sonnet"], "devM-sonnet-15087-1": [true, 1.3247095797098822, "Data2txt", "sonnet"], "devM-haiku-15093-0": [true, 1.8904419284956449, "QA", "haiku"], "devM-haiku-15093-1": [true, 1.7818781151058447, "QA", "haiku"], "devM-sonnet-15093-0": [true, 3.1355641231395, "QA", "sonnet"], "devM-sonnet-15093-1": [true, 1.3936887369874502, "QA", "sonnet"], "devM-haiku-15096-0": [true, 1.8033613897091356, "QA", "haiku"], "devM-haiku-15096-1": [true, 2.1090543529470427, "QA", "haiku"], "devM-sonnet-15096-0": [true, 1.570265510801433, "QA", "sonnet"], "devM-sonnet-15096-1": [true, 1.9502790866397648, "QA", "sonnet"], "devM-haiku-15099-0": [true, 0.9290933177644372, "QA", "haiku"], "devM-haiku-15099-1": [true, 1.0888925353246086, "QA", "haiku"], "devM-sonnet-15099-0": [true, 1.2974219895606907, "QA", "sonnet"], "devM-sonnet-15099-1": [true, 1.0846913842700223, "QA", "sonnet"], "devM-haiku-15101-0": [false, 2.333264055487553, "QA", "haiku"], "devM-haiku-15101-1": [true, 1.845907953207214, "QA", "haiku"], "devM-sonnet-15101-0": [true, 2.040810993657648, "QA", "sonnet"], "devM-sonnet-15101-1": [true, 1.5406928638209842, "QA", "sonnet"], "devM-haiku-15104-0": [true, 3.664847567483117, "QA", "haiku"], "devM-haiku-15104-1": [true, 2.012145157850784, "QA", "haiku"], "devM-sonnet-15104-0": [true, 1.0794057968675683, "QA", "sonnet"], "devM-sonnet-15104-1": [true, 1.3647646843145729, "QA", "sonnet"], "devM-haiku-15110-0": [false, 3.585937607025227, "QA", "haiku"], "devM-haiku-15110-1": [false, 3.276623622505208, "QA", "haiku"], "devM-sonnet-15110-0": [true, 2.5781960106509207, "QA", "sonnet"], "devM-sonnet-15110-1": [true, 1.4820921810196626, "QA", "sonnet"], "devM-haiku-15113-0": [true, 0.9312057237945017, "QA", "haiku"], "devM-haiku-15113-1": [true, 2.4597213045101993, "QA", "haiku"], "devM-sonnet-15113-0": [true, 1.066928544978862, "QA", "sonnet"], "devM-sonnet-15113-1": [true, 0.9386284203123936, "QA", "sonnet"], "devM-haiku-15114-0": [true, 1.1298875938076915, "QA", "haiku"], "devM-haiku-15114-1": [true, 2.2295456719277738, "QA", "haiku"], "devM-sonnet-15114-0": [true, 2.075036408302047, "QA", "sonnet"], "devM-sonnet-15114-1": [true, 1.1407092598072444, "QA", "sonnet"], "devM-haiku-15151-0": [false, 2.6835358069053346, "QA", "haiku"], "devM-haiku-15151-1": [true, 3.174632565848387, "QA", "haiku"], "devM-sonnet-15151-0": [true, 2.287104259866648, "QA", "sonnet"], "devM-sonnet-15151-1": [true, 1.9908626059013834, "QA", "sonnet"], "devM-haiku-15174-0": [true, 1.466582090443364, "QA", "haiku"], "devM-haiku-15174-1": [true, 0.6640639335149858, "QA", "haiku"], "devM-sonnet-15174-0": [true, 1.7419342075094641, "QA", "sonnet"], "devM-sonnet-15174-1": [true, 2.0119814735954593, "QA", "sonnet"], "devM-haiku-15177-0": [true, 2.5827847125046945, "QA", "haiku"], "devM-haiku-15177-1": [false, 1.941181569322739, "QA", "haiku"], "devM-sonnet-15177-0": [true, 2.5974801691629317, "QA", "sonnet"], "devM-sonnet-15177-1": [true, 2.5619754081365675, "QA", "sonnet"], "devM-haiku-15179-0": [true, 1.4907491347941113, "QA", "haiku"], "devM-haiku-15179-1": [true, 1.680417913483046, "QA", "haiku"], "devM-sonnet-15179-0": [true, 1.9045608555354179, "QA", "sonnet"], "devM-sonnet-15179-1": [true, 1.992925429481129, "QA", "sonnet"], "devM-haiku-15183-0": [false, 4.382353392225167, "QA", "haiku"], "devM-haiku-15183-1": [true, 2.869864793236763, "QA", "haiku"], "devM-sonnet-15183-0": [true, 2.1757093233023808, "QA", "sonnet"], "devM-sonnet-15183-1": [true, 2.7701341715564376, "QA", "sonnet"], "devM-haiku-15185-0": [true, 1.6437892332757804, "QA", "haiku"], "devM-haiku-15185-1": [false, 1.900897097140719, "QA", "haiku"], "devM-sonnet-15185-0": [true, 1.0527777265847855, "QA", "sonnet"], "devM-sonnet-15185-1": [true, 1.2747986359232413, "QA", "sonnet"], "devM-haiku-15200-0": [false, 1.4472899733600986, "QA", "haiku"], "devM-haiku-15200-1": [true, 2.512303322722355, "QA", "haiku"], "devM-sonnet-15200-0": [true, 1.4320774101691889, "QA", "sonnet"], "devM-sonnet-15200-1": [true, 1.9336977390289805, "QA", "sonnet"], "devM-haiku-15204-0": [true, 0.6450665115127073, "QA", "haiku"], "devM-haiku-15204-1": [true, 0.7851041625089639, "QA", "haiku"], "devM-sonnet-15204-0": [true, 0.9372464927795546, "QA", "sonnet"], "devM-sonnet-15204-1": [true, 1.2019194054193483, "QA", "sonnet"], "devM-haiku-15222-0": [true, 0.4213709522171454, "QA", "haiku"], "devM-haiku-15222-1": [true, 0.4254987313268344, "QA", "haiku"], "devM-sonnet-15222-0": [true, 0.7232473038969058, "QA", "sonnet"], "devM-sonnet-15222-1": [true, 0.8601216638673841, "QA", "sonnet"], "devM-haiku-15226-0": [true, 0.7100321841276868, "QA", "haiku"], "devM-haiku-15226-1": [true, 0.9568076904542969, "QA", "haiku"], "devM-sonnet-15226-0": [true, 0.7938300889712748, "QA", "sonnet"], "devM-sonnet-15226-1": [true, 1.2810625273647887, "QA", "sonnet"], "devM-haiku-15249-0": [true, -0.1712429855782865, "QA", "haiku"], "devM-haiku-15249-1": [true, 0.18235297096551228, "QA", "haiku"], "devM-sonnet-15249-0": [true, 2.337210501199808, "QA", "sonnet"], "devM-sonnet-15249-1": [true, 2.3477656727261644, "QA", "sonnet"], "devM-haiku-15260-0": [false, 1.7580474482976352, "QA", "haiku"], "devM-haiku-15260-1": [true, 1.6745903532447224, "QA", "haiku"], "devM-sonnet-15260-0": [true, 0.872760046260665, "QA", "sonnet"], "devM-sonnet-15260-1": [true, 0.7759999081894029, "QA", "sonnet"], "devM-haiku-15362-0": [true, 1.875605577216795, "QA", "haiku"], "devM-haiku-15362-1": [true, 1.4419842017885656, "QA", "haiku"], "devM-sonnet-15362-0": [true, 1.4493872576923907, "QA", "sonnet"], "devM-sonnet-15362-1": [true, 1.8632901003231535, "QA", "sonnet"], "devM-haiku-15365-0": [false, 2.6036737284969727, "QA", "haiku"], "devM-haiku-15365-1": [false, 2.5808674535990175, "QA", "haiku"], "devM-sonnet-15365-0": [false, 3.1264895602796674, "QA", "sonnet"], "devM-sonnet-15365-1": [false, 2.2583963093216752, "QA", "sonnet"], "devM-haiku-15367-0": [true, 1.6262644121481753, "QA", "haiku"], "devM-haiku-15367-1": [false, 1.1087150790896843, "QA", "haiku"], "devM-sonnet-15367-0": [true, 1.163835526894311, "QA", "sonnet"], "devM-sonnet-15367-1": [true, 2.0234816800622664, "QA", "sonnet"], "devM-haiku-15368-0": [true, 1.9824973767449652, "QA", "haiku"], "devM-haiku-15368-1": [true, 2.1520885646424377, "QA", "haiku"], "devM-sonnet-15368-0": [true, 1.996964483295405, "QA", "sonnet"], "devM-sonnet-15368-1": [true, 2.33974916271108, "QA", "sonnet"], "devM-haiku-15387-0": [true, 0.4409845843514762, "QA", "haiku"], "devM-haiku-15387-1": [false, 0.8560098550451447, "QA", "haiku"], "devM-sonnet-15387-0": [false, 0.2802949015614137, "QA", "sonnet"], "devM-sonnet-15387-1": [true, 0.29005711445678206, "QA", "sonnet"], "devM-haiku-15406-0": [true, 0.7284783977809519, "QA", "haiku"], "devM-haiku-15406-1": [true, 0.9393255158467774, "QA", "haiku"], "devM-sonnet-15406-0": [true, 1.2502199533777232, "QA", "sonnet"], "devM-sonnet-15406-1": [true, 1.1344175882614287, "QA", "sonnet"], "devM-haiku-15416-0": [true, 1.452752762490055, "QA", "haiku"], "devM-haiku-15416-1": [true, 1.019753998398876, "QA", "haiku"], "devM-sonnet-15416-0": [true, 0.8762540898128249, "QA", "sonnet"], "devM-sonnet-15416-1": [true, 0.8762540898128249, "QA", "sonnet"], "devM-haiku-15429-0": [true, 1.9181065045590793, "QA", "haiku"], "devM-haiku-15429-1": [true, 1.5906763752780102, "QA", "haiku"], "devM-sonnet-15429-0": [true, 2.6480972667364417, "QA", "sonnet"], "devM-sonnet-15429-1": [true, 3.1581475653256463, "QA", "sonnet"], "devM-haiku-15439-0": [true, 3.0598936596943744, "QA", "haiku"], "devM-haiku-15439-1": [true, 3.1277064179225813, "QA", "haiku"], "devM-sonnet-15439-0": [true, 1.1546880864239975, "QA", "sonnet"], "devM-sonnet-15439-1": [true, 1.5013793178433195, "QA", "sonnet"], "devM-haiku-15449-0": [true, 0.9000580220719698, "QA", "haiku"], "devM-haiku-15449-1": [true, 0.47874995369065054, "QA", "haiku"], "devM-sonnet-15449-0": [true, 0.8840386084602521, "QA", "sonnet"], "devM-sonnet-15449-1": [true, 2.46118956569141, "QA", "sonnet"], "devM-haiku-15456-0": [true, 1.1132286555771929, "QA", "haiku"], "devM-haiku-15456-1": [true, 0.7800845917408413, "QA", "haiku"], "devM-sonnet-15456-0": [true, 1.0678389144855789, "QA", "sonnet"], "devM-sonnet-15456-1": [true, 1.9664864933451849, "QA", "sonnet"], "devM-haiku-15462-0": [true, 1.1724630166713723, "QA", "haiku"], "devM-haiku-15462-1": [true, 1.1262194769957492, "QA", "haiku"], "devM-sonnet-15462-0": [true, 0.8867552168589383, "QA", "sonnet"], "devM-sonnet-15462-1": [true, 0.7805946365730178, "QA", "sonnet"], "devM-haiku-15475-0": [true, 1.7793011654917308, "QA", "haiku"], "devM-haiku-15475-1": [true, 0.931014092384198, "QA", "haiku"], "devM-sonnet-15475-0": [true, 0.6866071102606259, "QA", "sonnet"], "devM-sonnet-15475-1": [true, 0.6935220110615119, "QA", "sonnet"], "devM-haiku-15480-0": [true, 0.7125896538928419, "QA", "haiku"], "devM-haiku-15480-1": [true, 1.0338014259715929, "QA", "haiku"], "devM-sonnet-15480-0": [true, 1.2720558003300844, "QA", "sonnet"], "devM-sonnet-15480-1": [false, 3.035082447846647, "QA", "sonnet"], "devM-haiku-15496-0": [true, 1.1074485331555413, "QA", "haiku"], "devM-haiku-15496-1": [true, 1.2530680409460182, "QA", "haiku"], "devM-sonnet-15496-0": [true, 0.46474857733970465, "QA", "sonnet"], "devM-sonnet-15496-1": [true, 2.3553109538695467, "QA", "sonnet"], "devM-haiku-15508-0": [false, 1.0383917693654938, "QA", "haiku"], "devM-haiku-15508-1": [false, 0.9830363920063268, "QA", "haiku"], "devM-sonnet-15508-0": [true, 0.7747040651755551, "QA", "sonnet"], "devM-sonnet-15508-1": [false, 0.6009344344945675, "QA", "sonnet"], "devM-haiku-15510-0": [true, 1.0632194458350868, "QA", "haiku"], "devM-haiku-15510-1": [true, 1.3051301702567235, "QA", "haiku"], "devM-sonnet-15510-0": [true, 1.2336046179478262, "QA", "sonnet"], "devM-sonnet-15510-1": [true, 1.429801764107991, "QA", "sonnet"], "devM-haiku-15534-0": [true, 0.8652777669145313, "QA", "haiku"], "devM-haiku-15534-1": [true, 1.5742194547227146, "QA", "haiku"], "devM-sonnet-15534-0": [true, 1.8856138199919137, "QA", "sonnet"], "devM-sonnet-15534-1": [true, 1.8856138199919137, "QA", "sonnet"], "devM-haiku-15548-0": [true, 1.3458152391568472, "QA", "haiku"], "devM-haiku-15548-1": [true, 0.8531991629409068, "QA", "haiku"], "devM-sonnet-15548-0": [true, 1.659939748887663, "QA", "sonnet"], "devM-sonnet-15548-1": [true, 1.3376635518259234, "QA", "sonnet"], "devM-haiku-15559-0": [true, 2.331132915034499, "QA", "haiku"], "devM-haiku-15559-1": [true, 1.8994936417581074, "QA", "haiku"], "devM-sonnet-15559-0": [true, 2.0595359153428427, "QA", "sonnet"], "devM-sonnet-15559-1": [true, 2.5452810223346187, "QA", "sonnet"], "devM-haiku-15565-0": [true, 1.2082903887717449, "QA", "haiku"], "devM-haiku-15565-1": [true, 3.271499142959203, "QA", "haiku"], "devM-sonnet-15565-0": [true, 1.765140988266964, "QA", "sonnet"], "devM-sonnet-15565-1": [true, 1.6807385226090912, "QA", "sonnet"], "devM-haiku-15569-0": [false, 2.186718851412314, "QA", "haiku"], "devM-haiku-15569-1": [false, 1.5093599286838302, "QA", "haiku"], "devM-sonnet-15569-0": [true, 2.1727401021018276, "QA", "sonnet"], "devM-sonnet-15569-1": [true, 2.5101316489756327, "QA", "sonnet"], "devM-haiku-15595-0": [false, 2.3562309033660247, "Summary", "haiku"], "devM-haiku-15595-1": [false, 2.300758161825219, "Summary", "haiku"], "devM-sonnet-15595-0": [true, 2.333773868600472, "Summary", "sonnet"], "devM-sonnet-15595-1": [true, 2.6978774120443827, "Summary", "sonnet"], "devM-haiku-15598-0": [false, 0.7878017647071547, "Summary", "haiku"], "devM-haiku-15598-1": [false, 2.1636675618759105, "Summary", "haiku"], "devM-sonnet-15598-0": [true, 1.3301911029967712, "Summary", "sonnet"], "devM-sonnet-15598-1": [true, 1.0017292657625114, "Summary", "sonnet"], "devM-haiku-15609-0": [true, -0.4282397368748427, "Summary", "haiku"], "devM-haiku-15609-1": [true, -0.4282397368748427, "Summary", "haiku"], "devM-sonnet-15609-0": [true, -0.20589027660069553, "Summary", "sonnet"], "devM-sonnet-15609-1": [true, -0.15757009418082815, "Summary", "sonnet"], "devM-haiku-15611-0": [true, 2.356563688038909, "Summary", "haiku"], "devM-haiku-15611-1": [true, 3.1086080043701885, "Summary", "haiku"], "devM-sonnet-15611-0": [true, 1.6576550285689724, "Summary", "sonnet"], "devM-sonnet-15611-1": [true, 2.5946607691371257, "Summary", "sonnet"], "devM-haiku-15619-0": [true, 0.8332206070478478, "Summary", "haiku"], "devM-haiku-15619-1": [true, 0.868436669026251, "Summary", "haiku"], "devM-sonnet-15619-0": [true, 0.8451545389861892, "Summary", "sonnet"], "devM-sonnet-15619-1": [true, 0.2999870754660584, "Summary", "sonnet"], "devM-haiku-15640-0": [true, -0.06650595006115204, "Summary", "haiku"], "devM-haiku-15640-1": [true, 0.11592329019462784, "Summary", "haiku"], "devM-sonnet-15640-0": [false, -0.011895533802466607, "Summary", "sonnet"], "devM-sonnet-15640-1": [true, 0.08140654629330885, "Summary", "sonnet"], "devM-haiku-15666-0": [true, 0.5121075094596643, "Summary", "haiku"], "devM-haiku-15666-1": [true, 0.08399600192961426, "Summary", "haiku"], "devM-sonnet-15666-0": [true, 0.7490083392836833, "Summary", "sonnet"], "devM-sonnet-15666-1": [true, 0.4626373165531784, "Summary", "sonnet"], "devM-haiku-15677-0": [true, 1.9478876943816332, "Summary", "haiku"], "devM-haiku-15677-1": [false, 1.2389336674689408, "Summary", "haiku"], "devM-sonnet-15677-0": [true, 1.6728210799513972, "Summary", "sonnet"], "devM-sonnet-15677-1": [true, 1.7501834486977148, "Summary", "sonnet"], "devM-haiku-15697-0": [false, 3.4607535833378593, "Summary", "haiku"], "devM-haiku-15697-1": [false, 2.900782743078861, "Summary", "haiku"], "devM-sonnet-15697-0": [false, 3.1130510868917534, "Summary", "sonnet"], "devM-sonnet-15697-1": [false, 2.588788654947601, "Summary", "sonnet"], "devM-haiku-15709-0": [false, 1.8467479699204712, "Summary", "haiku"], "devM-haiku-15709-1": [true, 0.9393840873580614, "Summary", "haiku"], "devM-sonnet-15709-0": [true, 1.2968451551736186, "Summary", "sonnet"], "devM-sonnet-15709-1": [true, 0.8231524682950749, "Summary", "sonnet"], "devM-haiku-15726-0": [false, 1.6819989287727657, "Summary", "haiku"], "devM-haiku-15726-1": [true, 0.7032299281190275, "Summary", "haiku"], "devM-sonnet-15726-0": [true, 1.3319390229822101, "Summary", "sonnet"], "devM-sonnet-15726-1": [true, 2.4546808614823714, "Summary", "sonnet"], "devM-haiku-15736-0": [true, 1.2356634243454112, "Summary", "haiku"], "devM-haiku-15736-1": [false, 2.1576581739057374, "Summary", "haiku"], "devM-sonnet-15736-0": [true, 1.261026169996713, "Summary", "sonnet"], "devM-sonnet-15736-1": [true, 1.3090237167322742, "Summary", "sonnet"], "devM-haiku-15742-0": [true, 0.5123052661070578, "Summary", "haiku"], "devM-haiku-15742-1": [true, 0.5302831341513058, "Summary", "haiku"], "devM-sonnet-15742-0": [true, 1.1080668417145754, "Summary", "sonnet"], "devM-sonnet-15742-1": [false, 1.1080668417145754, "Summary", "sonnet"], "devM-haiku-15752-0": [true, 0.6264750408972408, "Summary", "haiku"], "devM-haiku-15752-1": [true, 0.540162871163031, "Summary", "haiku"], "devM-sonnet-15752-0": [true, 0.8015654360439575, "Summary", "sonnet"], "devM-sonnet-15752-1": [true, 1.1226750654425381, "Summary", "sonnet"], "devM-haiku-15782-0": [false, 1.5962174270010567, "Summary", "haiku"], "devM-haiku-15782-1": [true, 0.056636543258669625, "Summary", "haiku"], "devM-sonnet-15782-0": [true, -0.16284302317678645, "Summary", "sonnet"], "devM-sonnet-15782-1": [true, 0.0803909548592973, "Summary", "sonnet"], "devM-haiku-15808-0": [false, 0.550881500330428, "Summary", "haiku"], "devM-haiku-15808-1": [false, 0.43395987280045234, "Summary", "haiku"], "devM-sonnet-15808-0": [false, 0.7193956705545631, "Summary", "sonnet"], "devM-sonnet-15808-1": [true, 0.6286504053904758, "Summary", "sonnet"]}}
THR_B = 2.7385038673533924      # 판정기 v1.0 임계값 B

import pandas as pd
SEED = 2041                              # 부트스트랩 (사전 고정)

def make_table(K2T, CASES, GENS, thr):
    # K2 표(정답·1차 점수) + 드라이브의 응답 원문·근거. 근거 = Opus가 정답을 매길 때 본 것과 같은 ref_label
    rows = []
    for sp, tab in K2T.items():
        for cid, (clean, mx, task, gen) in tab.items():
            c, g = CASES[sp][cid], GENS[sp][cid]
            assert g['text'].strip(), cid
            rows.append({'id': cid, 'part': sp, 'task': task, 'gen': gen, 'hal': not clean, 'max1': mx, 'flag1': bool(mx >= thr),
                         'ref': c.get('ref_label') or c['ref_judge'], 'text': g['text']})
    return pd.DataFrame(rows)

CASES = {sp: load_json(os.path.join(K2_DIR, f'cases_{sp}.json')) for sp in K2T}
GENS = {sp: load_json(os.path.join(K2_DIR, f'gen_{sp}.json')) for sp in K2T}
P = make_table(K2T, CASES, GENS, THR_B)
print(P.groupby('part').agg(n=('id', 'size'), 환각=('hal', 'sum')))
for sp, d in P.groupby('part'):
    print(f"{sp} 1차(v1.0): 탐지 {int(d[d.hal].flag1.sum())}/{int(d.hal.sum())} = {d[d.hal].flag1.mean():.3f} | 오경보 {int(d[~d.hal].flag1.sum())}/{int((~d.hal).sum())} = {d[~d.hal].flag1.mean():.3f}")
D_ = P[P.part == 'devM']
J2_0_repro = (int(D_[D_.hal].flag1.sum()), int(D_.hal.sum()), int(D_[~D_.hal].flag1.sum()), int((~D_.hal).sum())) == (36, 191, 58, 881)
print('=== J2-0 1차 재현(36/191, 58/881):', '충족' if J2_0_repro else '미달 — 멈춤', '===')
assert J2_0_repro

## 3. Haiku 5회 투표 — 비용 추정, 형식 확인, 배치

In [ ]:
import anthropic
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=3)
PRICE = {'claude-haiku-5-5': (0.05, 0.25), 'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}   # 배치 $/백만 토큰 (2026-11)

def text_of(m): return ''.join(b.text for b in m.content if getattr(b, 'type', '') == 'text')
def rec(m): return {'text': text_of(m), 'stop': m.stop_reason, 'model': m.model, 'in': m.usage.input_tokens, 'out': m.usage.output_tokens}
def cost(res): return sum(r['in'] / 1e6 * PRICE.get(r['model'], (2, 10))[0] + r['out'] / 1e6 * PRICE.get(r['model'], (2, 10))[1] for r in res.values())

def run_batch(reqs, path, chunk=400):
    # reqs: cid -> 요청. 결과: cid -> rec. 다시 실행하면 제출한 묶음·받은 결과는 건너뛴다
    res = load_json(path) or {}
    bpath = path.replace('.json', '_batches.json')
    B = load_json(bpath) or {'chunks': []}
    sent = {c for ch in B['chunks'] for c in ch['cids']}
    todo = [c for c in reqs if c not in sent and c not in res]
    for k in range(0, len(todo), chunk):
        part = todo[k:k + chunk]
        b = A.messages.batches.create(requests=[{'custom_id': c, 'params': reqs[c]} for c in part])
        B['chunks'].append({'id': b.id, 'cids': part, 'done': False}); save_json(B, bpath)
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    for ch in B['chunks']:
        if ch.get('done'): continue
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded': res[r.custom_id] = rec(r.result.message)
        ch['done'] = True; save_json(B, bpath); save_json(res, path)
    miss = [c for c in reqs if c not in res]
    for c in miss[:100]:                                   # 배치 실패분은 바로 요청 1회
        try: res[c] = rec(A.messages.create(**reqs[c]))
        except Exception as e: print(f'  {c}: {type(e).__name__} {str(e)[:150]}'); break
    save_json(res, path)
    print(f'  결과 {len(res)}/{len(reqs)} | 사용량 약 ${cost({c: res[c] for c in reqs if c in res}):.2f}')
    return res


import pysbd
SEG2 = pysbd.Segmenter(language='en', clean=False)
def sents(text):
    ss = [s.strip() for s in SEG2.segment(text) if s.strip()]
    return ss or [text.strip()]

SYS_SENT = '''You check whether a RESPONSE is fully supported by a REFERENCE document.
You receive the REFERENCE and the RESPONSE split into numbered sentences. Judge EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution, certainty).
- Evident baseless info: states information that is not in the reference and cannot be directly inferred from it, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization or generalization of what the reference says; statements that the reference lacks the information.

Labels: "S" = fully supported. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give "type" (evident_conflict, subtle_conflict, evident_baseless, subtle_baseless), "phrase" (shortest hallucinated span copied exactly) and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

def user_sent(r):
    ss = sents(r.text)
    lines = '\n'.join(f'[{i + 1}] {s}' for i, s in enumerate(ss))
    return f"REFERENCE:\n<<<\n{r.ref}\n>>>\n\nRESPONSE ({len(ss)} numbered sentences):\n{lines}"

VOTE_MODEL, N_VOTE = 'claude-haiku-5-5', 5             # 문장 판정 투표 (사전 고정)

def requests_all(P):
    return {f'haiku-{r.id}-{v}': {'model': VOTE_MODEL, 'max_tokens': 2048, 'system': SYS_SENT,
                                  'messages': [{'role': 'user', 'content': user_sent(r)}]}
            for r in P.itertuples() for v in range(N_VOTE)}

def est_cost(reqs, out_tokens=250):
    # 요청 글자 수 / 3.5 ≈ 입력 토큰 (추정). 배치 단가
    tot = 0.0
    for p in reqs.values():
        n_in = (len(p['system']) + sum(len(m['content']) for m in p['messages'])) / 3.5
        pin, pout = PRICE[p['model']]
        tot += n_in / 1e6 * pin + out_tokens / 1e6 * pout
    return tot
def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

def haiku_scores(P, RES):
    # 투표 점수 = H(또는 H·A) 문장이 있다고 본 판정의 비율. 파싱 실패 판정 = 표시 안 함(탐지에 불리한 쪽)
    sH, sHA, ok = [], [], []
    for r in P.itertuples():
        n = len(sents(r.text)); h = ha = o = 0
        for v in range(N_VOTE):
            x = RES.get(f'haiku-{r.id}-{v}')
            labs, good = parse(x['text'], n) if x else ({}, False)
            ls = [e['label'] for e in labs.values()] if good else []
            h += 'H' in ls; ha += ('H' in ls or 'A' in ls); o += good
        sH.append(h / N_VOTE); sHA.append(ha / N_VOTE); ok.append(o / N_VOTE)
    P['s_haiku_H'], P['s_haiku_HA'], P['ok_haiku'] = sH, sHA, ok
    return P

BUDGET_USD, STOP_USD = 10.0, 8.0
REQ = requests_all(P)
EST = est_cost(REQ)
print(f'요청 {len(REQ)}건 | 예상 비용 ${EST:.2f} (멈춤 기준 ${STOP_USD}, 상한 ${BUDGET_USD})')
assert EST <= STOP_USD, '예상 비용이 멈춤 기준을 넘음 — 멈춤'
_r = P.iloc[0]
x = rec(A.messages.create(**REQ[f'haiku-{_r.id}-0']))
print(f"  형식 확인: 입력 {x['in']} 출력 {x['out']} | 파싱 {parse(x['text'], len(sents(_r.text)))[1]} | " + x['text'][:160].replace('\n', ' '))
RES = run_batch(REQ, os.path.join(J2_DIR, 'v_haiku.json'))
P = haiku_scores(P, RES)
OK_H = {sp: float(d.ok_haiku.mean()) for sp, d in P.groupby('part')}
print('  Haiku 파싱 성공', {k: round(v, 3) for k, v in OK_H.items()})

## 4. MiniCheck (7B → 안 되면 flan-t5-large → 안 되면 제외)

In [ ]:
MC_MODEL = None
mp = os.path.join(J2_DIR, 'minicheck.json'); MC = load_json(mp) or {}
if MC.get('_model') and all(i in MC for i in P.id):
    MC_MODEL = MC['_model']
else:
    try:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'minicheck[llm] @ git+https://github.com/Liyan06/MiniCheck.git@main'], check=True)
    except Exception:
        traceback.print_exc()
    for name in ['Bespoke-MiniCheck-7B', 'flan-t5-large']:
        try:
            from minicheck.minicheck import MiniCheck
            kw = {'enable_prefix_caching': False} if name.startswith('Bespoke') else {}
            mc = MiniCheck(model_name=name, cache_dir='/content/ckpts', **kw)
            todo = [r for r in P.itertuples() if r.id not in MC]
            for k in range(0, len(todo), 100):
                part = todo[k:k + 100]
                _, prob, _, _ = mc.score(docs=[r.ref for r in part], claims=[r.text for r in part])
                for r, p in zip(part, prob): MC[r.id] = float(p)
                MC['_model'] = name; save_json(MC, mp); print(f'  {name} {min(k + 100, len(todo))}/{len(todo)}', flush=True)
            MC_MODEL = name; break
        except Exception:
            print(f'MiniCheck {name} 실행 안 됨:'); traceback.print_exc()
            MC = {}                               # 실패한 모델의 부분 결과는 버림
if MC_MODEL:
    P['s_minicheck'] = [1 - MC[i] for i in P.id]; print('MiniCheck 모델:', MC_MODEL)

## 5. calM에서 임계값·선택 → devM 판정

In [ ]:
MAX_FLAG, MIN_DET, MIN_LO = 0.20, 0.40, 0.30           # 사전 고정 (사용자 결정: 오경보 20%, 탐지 40%)
ORDER = ['haiku_H', 'haiku_HA', 'minicheck', 'ens']    # 동점이면 앞쪽(서비스에서 단순·싼 쪽)

def comb(d, col, t):
    # 서비스 경고 = 1차(판정기 v1.0) 경고 또는 2차 점수 ≥ t
    return d.flag1.values | (d[col].values >= t)

def thr_at(d, col):
    # calM 멀쩡한 응답의 결합 경고율 ≤ MAX_FLAG인 가장 작은 t (calM에 나온 점수 값 중에서)
    cl = d[~d.hal]
    for t in np.unique(np.concatenate([d[col].values.astype(float), [np.inf]])):
        if comb(cl, col, t).mean() <= MAX_FLAG: return float(t)
    return float('inf')

def rates(d, col, t):
    f = comb(d, col, t)
    return float(f[d.hal.values].mean()), float(f[~d.hal.values].mean())

def select(P, cands):
    C = P[P.part == 'calM']; thr, ok = {}, []
    for c in cands:
        t = thr_at(C, f's_{c}'); thr[c] = t
        det, fl = rates(C, f's_{c}', t)
        print(f'  calM {c:10} 임계값 {t:.3f} | 결합 탐지 {det:.3f} | 결합 오경보 {fl:.3f}')
        ok.append((det, -ORDER.index(c), c))
    return (max(ok)[2] if ok else None), thr

def boot(D, fn, n=2000, seed=SEED):
    # 응답 단위 부트스트랩 95% 구간
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        v = fn(D.iloc[rng.integers(0, len(D), len(D))])
        if v == v: vals.append(v)
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def judge_j2(P, sel, thr):
    D = P[P.part == 'devM']
    det1, fl1 = float(D[D.hal].flag1.mean()), float(D[~D.hal].flag1.mean())
    out = {'n_dev': len(D), 'n_dev_hal': int(D.hal.sum()), 'n_dev_clean': int((~D.hal).sum()), 'selected': sel,
           'det1': det1, 'flag1': fl1}
    if sel is None:
        out.update(J2_1=False, J2_2=False, J2=False); return out
    col, t = f's_{sel}', thr[sel]
    det, fl = rates(D, col, t)
    out.update(det=det, flag=fl, thr=t)
    out['det_ci'] = boot(D, lambda d: float(comb(d[d.hal], col, t).mean()))
    out['flag_ci'] = boot(D, lambda d: float(comb(d[~d.hal], col, t).mean()))
    out['J2_1'] = bool(det >= MIN_DET and out['det_ci'][0] >= MIN_LO)
    out['J2_2'] = bool(fl <= MAX_FLAG)
    out['J2'] = out['J2_1'] and out['J2_2']
    out['det2_only'], out['flag2_only'] = float((D[D.hal][col] >= t).mean()), float((D[~D.hal][col] >= t).mean())
    out['by'] = {f'{k[0]}|{k[1]}': dict(zip(['det', 'flag'], rates(g, col, t)), n_hal=int(g.hal.sum()), n_clean=int((~g.hal).sum()))
                 for k, g in D.groupby(['gen', 'task'])}
    return out

CANDS = []
if min(OK_H.values()) >= 0.95: CANDS += ['haiku_H', 'haiku_HA']
if MC_MODEL: CANDS.append('minicheck')
if 'haiku_H' in CANDS and MC_MODEL:
    P['s_ens'] = P[['s_haiku_H', 's_minicheck']].mean(axis=1); CANDS.append('ens')
print('J2-0 통과 후보:', CANDS)
SEL, THR = select(P, CANDS)
R = judge_j2(P, SEL, THR)
D = P[P.part == 'devM']
print(f"\ndevM: 환각 {R['n_dev_hal']}, 멀쩡 {R['n_dev_clean']} | 1차만: 탐지 {R['det1']:.3f}, 오경보 {R['flag1']:.3f}")
REP = {}
for c in CANDS:
    d, f = rates(D, f's_{c}', THR[c]); REP[c] = {'thr': THR[c], 'det': d, 'flag': f,
          'det2_only': float((D[D.hal][f's_{c}'] >= THR[c]).mean()), 'flag2_only': float((D[~D.hal][f's_{c}'] >= THR[c]).mean())}
    print(f"  (보고) devM {c:10} 결합 탐지 {d:.3f} | 결합 오경보 {f:.3f} | 2차 단독 {REP[c]['det2_only']:.3f} / {REP[c]['flag2_only']:.3f}")
if SEL is None:
    print('\n=== J2 미달: 유효한 후보 없음 ===')
else:
    print(f"\n=== 선택: {SEL} (calM 임계값 {THR[SEL]:.3f}) ===")
    print(f"=== J2-1(주): 결합 탐지율 {R['det']:.3f} [{R['det_ci'][0]:.3f}, {R['det_ci'][1]:.3f}] (≥ 0.40, 하한 ≥ 0.30) → {'충족' if R['J2_1'] else '미달'} ===")
    print(f"=== J2-2: 결합 오경보 {R['flag']:.3f} [{R['flag_ci'][0]:.3f}, {R['flag_ci'][1]:.3f}] (≤ 0.20) → {'충족' if R['J2_2'] else '미달'} ===")
    print(f"=== J2 {'채택' if R['J2'] else '미채택'} === (1차만 {R['det1']:.3f} → 결합 {R['det']:.3f})")
    for k, v in R['by'].items(): print(f"  (보고) {k:18} 탐지 {v['det']:.3f} (환각 {v['n_hal']}) | 오경보 {v['flag']:.3f} (멀쩡 {v['n_clean']})")

## 6. J2-3 지연 — 선택 후보 응답 1개 검증 시간 (devM 30개, 실시간)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
LAT = None
if SEL:
    use_h, use_mc = SEL in ('haiku_H', 'haiku_HA', 'ens'), SEL in ('minicheck', 'ens')
    rows = list(D.itertuples())[:30]; LAT = []
    for r in rows:
        calls = [REQ[f'haiku-{r.id}-{v}'] for v in range(N_VOTE)] if use_h else []
        t1 = time.perf_counter()
        if calls:
            with ThreadPoolExecutor(len(calls)) as ex: list(ex.map(lambda p: A.messages.create(**p), calls))
        ms = (time.perf_counter() - t1) * 1000
        if use_mc and 'mc' in globals():
            t2 = time.perf_counter(); mc.score(docs=[r.ref], claims=[r.text]); ms = max(ms, (time.perf_counter() - t2) * 1000)
        LAT.append(ms)
    p50, p95 = np.percentile(LAT, 50), np.percentile(LAT, 95)
    R['latency_ms'] = {'n': len(LAT), 'p50': float(p50), 'p95': float(p95)}; R['J2_3'] = bool(p95 <= 10000)
    print(f"=== J2-3: {SEL} p50 {p50:.0f}ms, p95 {p95:.0f}ms (≤ 10000) → {'hold 가능' if R['J2_3'] else '응답 뒤 경고(비동기)'} ===")

## 7. 저장

In [ ]:
usage = {'v_haiku.json': round(cost(load_json(os.path.join(J2_DIR, 'v_haiku.json')) or {}), 2)}
print('API 사용량(배치 단가, 지연 측정·형식 확인 제외) $', usage)
LOCAL = '/content/j2_local'; os.makedirs(LOCAL, exist_ok=True)
cols = ['id', 'part', 'task', 'gen', 'hal', 'max1', 'flag1'] + [c for c in P.columns if c.startswith(('s_', 'ok_'))]
P[cols].to_csv(os.path.join(LOCAL, 'j2_rows.csv'), index=False)        # 원문(근거·응답) 없음
json.dump(dict(R, cands=CANDS, thr=THR, report=REP, ok_haiku=OK_H, minicheck_model=MC_MODEL, est_usd=EST, cost_usd=usage, gpu=GPU),
          open(os.path.join(LOCAL, 'j2_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
for f in os.listdir(LOCAL):
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(J2_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/j2_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)